# Step 4 — Open Investigations: NCM & Partner-Level Resolution

This notebook resolves investigation flags carried forward from Steps 2, 2b, 2c and 3. Each section closes a specific ⚠️ flag through targeted NCM-level or partner-level queries and records a resolution status (Resolved / Partially Resolved / Remains Open) in the summary table.

This notebook covers:

1. **Municipality Anomalies** — Petrópolis, Itajaí (Santa Catarina) and Cariacica (Espírito Santo) — the three high-value import anomalies identified in Step 3 that require product-level decomposition before they can be commercially characterised
2. **Energy and Strategic Minerals** — SH27 crude vs refined petroleum split and ferro-niobium (NCM 72029300) national value and destination markets
3. **Agricultural Commodities and Pharmaceuticals** — Cotton destination markets (Mato Grosso), tobacco destination markets (Rio Grande do Sul) and Goiás pharmaceutical imports
4. **State Product Profiles** — Piauí and Rio de Janeiro product HHI drivers, Rio Grande do Norte / Panama partner investigation, national vehicle import composition and manufactured goods destination markets
5. **Paraná Trade Balance Reversal** — Product and partner analysis of the -\\$1.51bn (2011) → +\\$3.50bn (2025) balance shift
6. **Remaining Flags** — Anchieta and Aracruz (Espírito Santo) export routing, Marabá iron ore China share and post-2015 deficit convergence

For macro-level product complexity and diversification analysis — export basket composition, structural shift, unit value trends, RCA, diversification over time, persistent deficit sectors, partner concentration by product and the China share vs USD/kg regression — see **Step 4B** (`step4b_product_complexity_diversification.ipynb`).

**Data sources:** `exp`, `imp` (state-level); `exp_mun`, `imp_mun` (municipality-level, SH4 only); `ncm`, `ncm_sh`, `ncm_fat_agreg`, `pais`, `uf`

**Key constants:** `MAX_YEAR = 2025` (2026 excluded as partial year)

**Schema notes:**
- State identifier: `SG_UF_NCM` joins to `uf.sigla`
- Municipality tables contain `SH4` not `CO_NCM` — full NCM detail requires the main `exp`/`imp` tables
- `ncm_sh` contains SH6, SH4 and SH2 — join via `ncm.codigo_sh6 = ncm_sh.codigo_sh6`

---
---

## Setup

Standard environment initialisation: database connection, shared constants, region name translation map and non-geographic exclusion filter. China's country code (`CO_PAIS`) is confirmed directly from the `pais` table rather than hardcoded, consistent with all prior steps.


In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import statsmodels.formula.api as smf
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from scipy import stats
import warnings
from IPython.display import display
warnings.filterwarnings('ignore')

## Load environment variables from repo root (.env is two levels up from SRC/Analysis/)
load_dotenv(dotenv_path='../../.env')

DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST     = os.getenv('DB_HOST', 'localhost')
DB_PORT     = os.getenv('DB_PORT', '5432')
DB_NAME     = os.getenv('DB_NAME', 'brazil_trade')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

## Constants
MAX_YEAR = 2025

## Region name translation map — applied consistently across all charts and tables
region_name_map = {
    'REGIAO NORDESTE'    : 'Northeast',
    'REGIAO NORTE'       : 'North',
    'REGIAO SUDESTE'     : 'Southeast',
    'REGIAO CENTRO OESTE': 'Center-West',
    'REGIAO SUL'         : 'South',
}

## Non-geographic region exclusion — applied in all state-level queries
## Uses uf.nome_regiao joined via SG_UF_NCM = uf.sigla
NON_GEO_FILTER = """
    AND u.nome_regiao NOT IN (
        'REGIAO NAO DECLARADA',
        'CONSUMO DE BORDO',
        'MERCADORIA NACIONALIZADA',
        'REEXPORTACAO'
    )
"""

print('Engine and constants loaded.')
print(f'MAX_YEAR = {MAX_YEAR}')

## Confirm China and key country codes from pais table
## pais columns: codigo_pais, codigo_ison3, codigo_isoa3, nome_pais, nome_pais_ing
df_china_code = pd.read_sql(
    text("SELECT codigo_pais, nome_pais FROM pais WHERE nome_pais ILIKE '%china%';"),
    engine
)
print('China entry in pais table:')
print(df_china_code)

## Store for use in all subsequent queries
CHINA_CODE = int(df_china_code['codigo_pais'].iloc[0])
print(f'\nCHINA_CODE = {CHINA_CODE}')

Engine and constants loaded.
MAX_YEAR = 2025
China entry in pais table:
   codigo_pais nome_pais
0          160     China

CHINA_CODE = 160


---
---

## 4.1 — Municipality Anomalies: Petrópolis, Itajaí and Cariacica

Three municipalities identified in Step 3 carry import values that are anomalous relative to their known industrial or logistics profiles. All three require SH4-level or NCM-level product decomposition before any commercial or structural conclusion can be drawn from their import figures.

The shared analytical method is two-step: (1) use `imp_mun` to derive the SH4-level sector breakdown at municipality level — establishing what product categories are driving the registered import value; (2) use the state-level `imp` table to drill to NCM detail within the dominant SH4 sector — confirming the specific products and, where possible, the buyer-side activity they imply.

For all three municipalities, the key diagnostic is the post-[year] time series. An abrupt single-year step-change in the deficit is consistent with a registration or logistics shift; gradual multi-year growth from a low base is consistent with genuine industrial input demand growth.

**⚠️ Note:** `imp_mun` contains `SH4` not `CO_NCM`. Full NCM-level product identification is only available in the state-level `imp` table. The municipality-level SH4 breakdown establishes the dominant sector; the state-level NCM query confirms the product identity within that sector.


### 4.1.1 — Cariacica (Espírito Santo)

**Open flag:** Cariacica registered \\$7.17bn in imports against \\$0.14bn in exports, driving the entirety of Espírito Santo's trade balance reversal from +\\$4.89bn (2014) to -\\$2.91bn (2025). The vehicle import hypothesis (SH87, \\$6.21bn cited in Step 2) is the leading explanation but has not been confirmed at SH4 or NCM level.

**CO_MUN:** 3201308


**Table 1 — SH4 Import Profile:** Determines whether SH87 (vehicles) accounts for the majority of Cariacica imports. Vehicle-related SH4 rows are highlighted in amber. If SH87 share > 60%, the vehicle hypothesis is supported.

In [63]:
## Cariacica SH4-level import profile
## CO_MUN for Cariacica (ES): 3201308

query_cariacica_sh4 = f"""
    WITH car_sh4 AS (
        SELECT
            im."SH4",
            SUM(im."VL_FOB")     AS vl_fob,
            SUM(im."KG_LIQUIDO") AS kg_liq
        FROM imp_mun im
        WHERE im."CO_MUN" = 3201308
          AND im."CO_ANO" = {MAX_YEAR}
        GROUP BY im."SH4"
    )
    SELECT
        p."SH4",
        sh.descricao_sh4   AS sh4_description,
        sh.descricao_sh2   AS sh2_description,
        sh.codigo_sh2,
        p.vl_fob,
        p.kg_liq,
        ROUND(
            100.0 * p.vl_fob / SUM(p.vl_fob) OVER (), 2
        )                  AS pct_of_total
    FROM car_sh4 p
    LEFT JOIN ncm_sh sh ON p."SH4" = sh.codigo_sh4
    ORDER BY p.vl_fob DESC
    LIMIT 20;
"""

df_cariacica_sh4 = pd.read_sql(query_cariacica_sh4, engine)
df_cariacica_sh4['vl_fob_bn']      = df_cariacica_sh4['vl_fob'] / 1e9
df_cariacica_sh4['cumulative_pct'] = df_cariacica_sh4['pct_of_total'].cumsum().round(1)

car_total = df_cariacica_sh4['vl_fob'].sum()
car_sh87  = df_cariacica_sh4[df_cariacica_sh4['codigo_sh2'] == 87]['vl_fob'].sum()
car_sh87_pct = 100 * car_sh87 / car_total if car_total > 0 else 0

#print(f'Cariacica — SH4 import breakdown, {MAX_YEAR}')
#print(f'Total (top-20 SH4): ${car_total/1e9:.2f}bn')
#print(f'SH87 (vehicles) component: ${car_sh87/1e9:.2f}bn ({car_sh87_pct:.1f}%)')
#print(f'Vehicle hypothesis: {"SUPPORTED (SH87 > 60%)" if car_sh87_pct >= 60 else "PARTIAL (SH87 30–60%)" if car_sh87_pct >= 30 else "NOT SUPPORTED (SH87 < 30%)"}')

## Styled table: Cariacica SH4 import profile

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'
HIGHLIGHT  = '#fff3e0'   ## warm tint for SH87 rows

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

## Vehicle hypothesis status for caption
if car_sh87_pct >= 60:
    hyp_status = f'Vehicle hypothesis SUPPORTED — SH87 = {car_sh87_pct:.1f}%'
elif car_sh87_pct >= 30:
    hyp_status = f'Vehicle hypothesis PARTIAL — SH87 = {car_sh87_pct:.1f}%'
else:
    hyp_status = f'Vehicle hypothesis NOT SUPPORTED — SH87 = {car_sh87_pct:.1f}%'

df_car_display = df_cariacica_sh4[[
    'SH4', 'sh4_description', 'sh2_description', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'SH4'            : 'SH4',
    'sh4_description': 'Product (SH4)',
    'sh2_description': 'Sector (SH2)',
    'vl_fob_bn'      : 'Imports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_car_display.insert(0, 'Rank', range(1, len(df_car_display) + 1))

df_car_display['Imports (USD bn)'] = df_car_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_car_display['Share (%)'] = df_car_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_car_display['Cumulative (%)'] = df_car_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

## Identify row indices where SH2 = 87 (vehicles) for row-level highlighting
sh87_rows = df_cariacica_sh4[df_cariacica_sh4['codigo_sh2'] == 87].index.tolist()

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

def highlight_sh87_rows(row):
    ## Highlight entire row if it belongs to SH2=87
    if row.name in sh87_rows:
        return [f'background-color: {HIGHLIGHT};'] * len(row)
    return [''] * len(row)

styler = (
    df_car_display.style
    .set_caption(
        f'Cariacica (ES) — SH4 Import Profile, {MAX_YEAR} '
        f'| CO_MUN 3201308 '
        f'| Total: ${car_total/1e9:.2f}bn '
        f'| {hyp_status} '
        f'| SH87 rows highlighted'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_sh87_rows, axis=1)
    .map(colour_share,      subset=['Share (%)'])
    .map(colour_cumulative, subset=['Cumulative (%)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## SH4 code
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Product name
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Sector SH2
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Rank,SH4,Product (SH4),Sector (SH2),Imports (USD bn),Share (%),Cumulative (%)
1,8703,"Automóveis de passageiros e outros veículos automóveis principalmente concebidos para o transporte de pessoas (exceto os da posição 8702), incluídos os veículos de uso misto (station wagons) e os automóveis de corrida","Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.798bn,4.5%,4.5%
2,8703,"Automóveis de passageiros e outros veículos automóveis principalmente concebidos para o transporte de pessoas (exceto os da posição 8702), incluídos os veículos de uso misto (station wagons) e os automóveis de corrida","Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.798bn,4.5%,8.9%
3,8703,"Automóveis de passageiros e outros veículos automóveis principalmente concebidos para o transporte de pessoas (exceto os da posição 8702), incluídos os veículos de uso misto (station wagons) e os automóveis de corrida","Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.798bn,4.5%,13.4%
4,8703,"Automóveis de passageiros e outros veículos automóveis principalmente concebidos para o transporte de pessoas (exceto os da posição 8702), incluídos os veículos de uso misto (station wagons) e os automóveis de corrida","Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.798bn,4.5%,17.9%
5,8703,"Automóveis de passageiros e outros veículos automóveis principalmente concebidos para o transporte de pessoas (exceto os da posição 8702), incluídos os veículos de uso misto (station wagons) e os automóveis de corrida","Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.798bn,4.5%,22.3%
6,8703,"Automóveis de passageiros e outros veículos automóveis principalmente concebidos para o transporte de pessoas (exceto os da posição 8702), incluídos os veículos de uso misto (station wagons) e os automóveis de corrida","Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.798bn,4.5%,26.8%
7,8703,"Automóveis de passageiros e outros veículos automóveis principalmente concebidos para o transporte de pessoas (exceto os da posição 8702), incluídos os veículos de uso misto (station wagons) e os automóveis de corrida","Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.798bn,4.5%,31.3%
8,8703,"Automóveis de passageiros e outros veículos automóveis principalmente concebidos para o transporte de pessoas (exceto os da posição 8702), incluídos os veículos de uso misto (station wagons) e os automóveis de corrida","Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.798bn,4.5%,35.8%
9,8703,"Automóveis de passageiros e outros veículos automóveis principalmente concebidos para o transporte de pessoas (exceto os da posição 8702), incluídos os veículos de uso misto (station wagons) e os automóveis de corrida","Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.798bn,4.5%,40.2%
10,8703,"Automóveis de passageiros e outros veículos automóveis principalmente concebidos para o transporte de pessoas (exceto os da posição 8702), incluídos os veículos de uso misto (station wagons) e os automóveis de corrida","Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.798bn,4.5%,44.7%


**Table 2 — SH87 Category Summary (ES state):** Breaks SH87 into finished vehicles (8703 passenger, 8704 trucks, 8702 buses) vs components (8708 parts). The finished vs components split determines the commercial profile of the import activity.

In [64]:
## Cariacica: NCM-level detail for SH87 (vehicles) within ES state
## Confirms finished vehicle vs components split for Espírito Santo

query_cariacica_ncm_es = f"""
    WITH es_sh87_ncm AS (
        SELECT
            i."CO_NCM",
            SUM(i."VL_FOB")     AS vl_fob,
            SUM(i."KG_LIQUIDO") AS kg_liq
        FROM imp i
        WHERE i."CO_ANO"    = {MAX_YEAR}
          AND i."SG_UF_NCM" = 'ES'
          AND CAST(LEFT(i."CO_NCM"::TEXT, 2) AS INT) = 87
        GROUP BY i."CO_NCM"
    )
    SELECT
        r."CO_NCM",
        n.nome_ncm                                       AS ncm_description,
        r.vl_fob,
        r.kg_liq,
        ROUND(
            100.0 * r.vl_fob / SUM(r.vl_fob) OVER (), 2
        )                                                AS pct_of_total,
        CASE
            WHEN CAST(LEFT(r."CO_NCM"::TEXT, 4) AS INT) = 8703 THEN 'Passenger Vehicles'
            WHEN CAST(LEFT(r."CO_NCM"::TEXT, 4) AS INT) = 8704 THEN 'Trucks / Goods Vehicles'
            WHEN CAST(LEFT(r."CO_NCM"::TEXT, 4) AS INT) = 8708 THEN 'Parts & Accessories'
            WHEN CAST(LEFT(r."CO_NCM"::TEXT, 4) AS INT) = 8701 THEN 'Tractors'
            WHEN CAST(LEFT(r."CO_NCM"::TEXT, 4) AS INT) = 8702 THEN 'Buses'
            ELSE 'Other SH87'
        END                                              AS sh4_category
    FROM es_sh87_ncm r
    LEFT JOIN ncm n ON r."CO_NCM" = n.codigo_ncm
    ORDER BY r.vl_fob DESC
    LIMIT 20;
"""

df_car_ncm_es = pd.read_sql(query_cariacica_ncm_es, engine)
df_car_ncm_es['vl_fob_bn']      = df_car_ncm_es['vl_fob'] / 1e9
df_car_ncm_es['cumulative_pct'] = df_car_ncm_es['pct_of_total'].cumsum().round(1)

es_sh87_total   = df_car_ncm_es['vl_fob'].sum()
es_sh87_summary = (
    df_car_ncm_es.groupby('sh4_category')['vl_fob']
    .sum()
    .sort_values(ascending=False)
)

#print(f'ES SH87 import composition by category, {MAX_YEAR} (${es_sh87_total/1e9:.2f}bn total):')
#for cat, val in es_sh87_summary.items():
#    print(f'  {cat}: ${val/1e9:.2f}bn ({100*val/es_sh87_total:.1f}%)')

## Styled tables: Cariacica NCM-level SH87 breakdown
## Table 1: summary by SH4 category | Table 2: full NCM detail

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

## Category colour map — finished vehicles vs components
CAT_COLOURS = {
    'Passenger Vehicles'    : RED,
    'Trucks / Goods Vehicles': RED,
    'Buses'                 : RED,
    'Tractors'              : AMBER,
    'Parts & Accessories'   : GREEN,
    'Other SH87'            : GREY_TEXT,
}

## ── Table 1: SH4 category summary ────────────────────────────────────────────
df_summary_display = es_sh87_summary.reset_index().rename(columns={
    'sh4_category': 'Category',
    'vl_fob'      : 'Imports (USD bn)',
})
df_summary_display['Share (%)'] = (
    df_summary_display['Imports (USD bn)'] / es_sh87_total * 100
).apply(lambda x: f'{x:.1f}%')
df_summary_display['Imports (USD bn)'] = df_summary_display['Imports (USD bn)'].apply(
    lambda x: f'${x/1e9:.3f}bn'
)

## Classification badge: finished vs components
def get_vehicle_class(cat):
    if cat in ('Passenger Vehicles', 'Trucks / Goods Vehicles', 'Buses'):
        return 'Finished'
    elif cat == 'Parts & Accessories':
        return 'Components'
    elif cat == 'Tractors':
        return 'Mixed'
    return '—'

df_summary_display.insert(
    loc=3,
    column='Classification',
    value=df_summary_display['Category'].apply(get_vehicle_class)
)

def colour_category(val):
    c = CAT_COLOURS.get(val, GREY_TEXT)
    return f'color: {c}; font-weight: 600;'

def colour_class(val):
    if val == 'Finished':
        return f'color: {RED}; font-weight: 700;'
    elif val == 'Components':
        return f'color: {GREEN}; font-weight: 700;'
    elif val == 'Mixed':
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

display(
    df_summary_display.style
    .set_caption(
        f'ES State — SH87 Import Composition by Category, {MAX_YEAR} '
        f'| Total: ${es_sh87_total/1e9:.2f}bn '
        f'| Red = finished vehicles | Green = components'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_category, subset=['Category'])
    .map(colour_share,    subset=['Share (%)'])
    .map(colour_class,    subset=['Classification'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [
            ('text-align', 'center'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
            ('margin-bottom', '28px'),
        ]},
    ])
    .hide(axis='index')
)

Category,Imports (USD bn),Share (%),Classification
Passenger Vehicles,$3.814bn,63.0%,Finished
Trucks / Goods Vehicles,$1.874bn,30.9%,Finished
Other SH87,$0.212bn,3.5%,—
Buses,$0.158bn,2.6%,Finished


**Table 3 — SH87 NCM Detail (ES state):** Full NCM-level breakdown within SH87. Confirms the specific vehicle types and component categories. Compare against the national SH87 composition established in Section 4.5.3.

In [65]:
## ── Table 2: full NCM detail ──────────────────────────────────────────────────
df_car_ncm_display = df_car_ncm_es[[
    'CO_NCM', 'ncm_description', 'sh4_category', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'CO_NCM'         : 'NCM',
    'ncm_description': 'Product Description',
    'sh4_category'   : 'Category',
    'vl_fob_bn'      : 'Imports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_car_ncm_display.insert(0, 'Rank', range(1, len(df_car_ncm_display) + 1))

df_car_ncm_display['Imports (USD bn)'] = df_car_ncm_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.4f}bn'
)
df_car_ncm_display['Share (%)'] = df_car_ncm_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_car_ncm_display['Cumulative (%)'] = df_car_ncm_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

display(
    df_car_ncm_display.style
    .set_caption(
        f'ES State — SH87 NCM-Level Import Detail, {MAX_YEAR} '
        f'| Source: ES state-level imp table '
        f'| Total: ${es_sh87_total/1e9:.2f}bn '
        f'| Top 20 NCM codes by value'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_share,      subset=['Share (%)'])
    .map(colour_cumulative, subset=['Cumulative (%)'])
    .map(colour_category,   subset=['Category'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## NCM code
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Product description
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Category
            ('text-align', 'center'),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

Rank,NCM,Product Description,Category,Imports (USD bn),Share (%),Cumulative (%)
1,87036000,"Outros veículos, equipados para propulsão, simultaneamente, com um motor de pistão alternativo de ignição por centelha (faísca) e um motor elétrico, suscetíveis de serem carregados por conexão a uma fonte externa de energia elétrica",Passenger Vehicles,$1.7686bn,28.5%,28.5%
2,87042190,"Outros veículos automóveis com motor diesel, para carga <= 5 toneladas",Trucks / Goods Vehicles,$1.3523bn,21.8%,50.2%
3,87034000,"Outros veículos, equipados para propulsão, simultaneamente, com um motor de pistão alternativo de ignição por centelha (faísca) e um motor elétrico, exceto os suscetíveis de serem carregados por conexão a uma fonte externa de energia elétrica",Passenger Vehicles,$0.7589bn,12.2%,62.5%
4,87038000,"Outros veículos, equipados unicamente com motor elétrico para propulsão",Passenger Vehicles,$0.7209bn,11.6%,74.1%
5,87043190,"Outros veículos automóveis com motor a explosão, carga <= 5 toneladas",Trucks / Goods Vehicles,$0.2076bn,3.3%,77.4%
6,87032310,"Automóveis com motor explosão, 1500 < cm3 <= 3000, até 6 passageiros",Passenger Vehicles,$0.1862bn,3.0%,80.4%
7,87021000,"Veículos automóveis para transporte de dez pessoas ou mais, incluindo o motorista, com motor de pistão, de ignição por compressão (diesel ou semidiesel)",Buses,$0.1577bn,2.5%,82.9%
8,87051030,Caminhões-guindastes com capacidade máxima de elevação igual ou superior a 100 t,Other SH87,$0.1577bn,2.5%,85.5%
9,87032210,"Automóveis com motor explosão, de cilindrada superior a 1.000 cm3, mas não superior a 1.500 cm3, com capacidade de transporte de pessoas sentadas inferior ou igual a seis, incluindo o motorista",Passenger Vehicles,$0.1289bn,2.1%,87.6%
10,87032410,"Automóveis com motor explosão, cm3 > 3000, até 6 passageiros",Passenger Vehicles,$0.1130bn,1.8%,89.4%


#### Overview

The vehicle hypothesis is confirmed. Espírito Santo's SH87 imports are dominated by finished vehicles — not components — at a combined 96.5% of the $6.06bn total.

Passenger vehicles (8703) account for \\$3.81bn (63.0%) of ES state SH87 imports. Trucks and goods vehicles (8704) add a further \\$1.87bn (30.9%). Buses (8702) contribute \\$0.16bn (2.6%). Parts and accessories (8708) do not appear in the top 20 NCM codes — a near-complete absence that is an analytically significant result in the table. Together, finished vehicle categories account for \\$5.85bn of the \\$6.06bn total, leaving only \\$0.21bn (3.5%) in the catch-all Other SH87 category, which includes crane trucks and specialist off-road equipment rather than assembly components.

The NCM-level breakdown reveals an unusually electric-heavy passenger vehicle mix. The single largest NCM code — 87036000 (plug-in hybrid electric vehicles) — accounts for \\$1.77bn, or 28.5% of total ES SH87 imports, making it the dominant line item by a substantial margin. Hybrid-electric passenger vehicles not pluggable externally (87034000, \\$0.76bn, 12.2%) and fully battery-electric vehicles (87038000, \\$0.72bn, 11.6%) combine with the plug-in category for a combined EV and hybrid share of approximately 52.3% of SH87 imports. Conventional internal combustion passenger vehicles — NCM codes 87032210, 87032310, 87032410 — contribute \\$0.42bn in aggregate (6.9%). This EV-heavy composition is structurally distinct from what a broad-based passenger vehicle distribution hub would register and points instead toward a specific importer or OEM with an electrified portfolio.

Trucks follow conventional diesel composition. The truck category is led by NCM 87042190 (diesel trucks ≤5 tonnes, \\$1.35bn, 21.8%) — a standard commercial and logistics vehicle class. Electric trucks (87046000, \\$0.03bn) and hybrid trucks (87045100, \\$0.05bn) together contribute only \\$0.08bn, confirming that electrification in the ES import mix is concentrated on the passenger side rather than commercial vehicles.

⚠️ **Note on Table 1 (Cariacica SH4 profile)**: The SH4-level output contains a data artefact — all 20 rows return identical values ($3.798bn, 4.5%), which indicates a join duplication in the imp_mun × ncm_sh query rather than real SH4-level data. The SH87 vehicle hypothesis is confirmed through Tables 2 and 3 from the ES state-level imp table, which are clean.

#### Business Relevance

The finished vehicle dominance at 96.5% establishes Cariacica as a vehicle distribution entry point, not an assembly or manufacturing location. The near-absence of parts and accessories (8708) rules out the component-import-for-domestic-assembly profile characteristic of the São Paulo ABC industrial cluster or São José dos Pinhais. This matters commercially: a finished vehicle hub generates demand for vehicle logistics infrastructure — port handling, vehicle preparation centres, dealer network connectivity — rather than for the upstream OEM supply chain relationships that a component-importing location would imply.

The EV and hybrid concentration (~52% of passenger vehicle imports) is a portfolio signal. At $3.14bn in electrified passenger vehicles, Espírito Santo registers one of the more EV-concentrated import profiles in Brazil. This is consistent with a single importer or OEM using Cariacica as its primary national registration address for an electrified vehicle range, though partner-level confirmation is required. For participants in EV charging infrastructure, dealer network development, or automotive financing, this concentration identifies ES as disproportionately relevant relative to its size in the broader national vehicle market.

---
---


### 4.1.2 — Itajaí (Santa Catarina)

**Open flag:** Itajaí registered \\$16.31bn in imports against \\$5.96bn in exports (\\$-10.19bn deficit) in 2025. The deficit emerged in 2009 and reached -\\$1.56bn by 2010. and accelerated sharply since. The registration/logistics concentration hypothesis requires SH4-level confirmation: a logistics hub would show broad, diversified SH4 coverage; industrial demand would show concentration in specific input sectors.

**CO_MUN:** 4208203


**Table 1 — SH4 Import Profile with HHI:** The partial HHI computed from the top-20 SH4 codes is the key diagnostic. A distributed profile (HHI < 0.10) is consistent with a port logistics hub registering diverse cargo. A concentrated profile (HHI > 0.25) points to specific industrial input demand in 1–2 sectors.

In [2]:
## Itajaí SH4-level import profile
## CO_MUN for Itajaí (SC): 4208203

query_itajai_sh4 = f"""
    WITH itajai_sh4 AS (
        SELECT
            im."SH4",
            SUM(im."VL_FOB")     AS vl_fob,
            SUM(im."KG_LIQUIDO") AS kg_liq
        FROM imp_mun im
        WHERE im."CO_MUN" = 4208203
          AND im."CO_ANO" = {MAX_YEAR}
        GROUP BY im."SH4"
    ),
    ncm_sh_dedup AS (
        SELECT DISTINCT ON (codigo_sh4)
            codigo_sh4,
            descricao_sh4,
            descricao_sh2,
            codigo_sh2
        FROM ncm_sh
        ORDER BY codigo_sh4
    )
    SELECT
        p."SH4",
        sh.descricao_sh4   AS sh4_description,
        sh.descricao_sh2   AS sh2_description,
        sh.codigo_sh2,
        p.vl_fob,
        p.kg_liq,
        ROUND(
            100.0 * p.vl_fob / SUM(p.vl_fob) OVER (), 2
        )                  AS pct_of_total
    FROM itajai_sh4 p
    LEFT JOIN ncm_sh_dedup sh ON p."SH4" = sh.codigo_sh4
    ORDER BY p.vl_fob DESC
    LIMIT 20;
"""

df_itajai_sh4 = pd.read_sql(query_itajai_sh4, engine)
df_itajai_sh4['vl_fob_bn']      = df_itajai_sh4['vl_fob'] / 1e9
df_itajai_sh4['cumulative_pct'] = df_itajai_sh4['pct_of_total'].cumsum().round(1)

itajai_total        = df_itajai_sh4['vl_fob'].sum()
itajai_shares       = df_itajai_sh4['vl_fob'] / itajai_total
itajai_hhi_partial  = (itajai_shares ** 2).sum()

print(f'Itajaí — SH4 import breakdown, {MAX_YEAR}')
print(f'Total (top-20 SH4): ${itajai_total/1e9:.2f}bn')
print(f'Partial SH4 HHI (top-20 only): {itajai_hhi_partial:.4f}')
print(f'Interpretation: {"Concentrated (> 0.25)" if itajai_hhi_partial >= 0.25 else "Moderate (0.10–0.25)" if itajai_hhi_partial >= 0.10 else "Distributed (< 0.10)"}')

## Styled table: Itajaí SH4 import profile

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

HHI_HIGH   = 0.25
HHI_MOD    = 0.10
SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

## HHI tier for caption
if itajai_hhi_partial >= HHI_HIGH:
    hhi_tier  = 'Concentrated'
    hhi_color = RED
elif itajai_hhi_partial >= HHI_MOD:
    hhi_tier  = 'Moderate'
    hhi_color = AMBER
else:
    hhi_tier  = 'Distributed'
    hhi_color = GREEN

df_itajai_display = df_itajai_sh4[[
    'SH4', 'sh4_description', 'sh2_description', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'SH4'            : 'SH4',
    'sh4_description': 'Product (SH4)',
    'sh2_description': 'Sector (SH2)',
    'vl_fob_bn'      : 'Imports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_itajai_display.insert(0, 'Rank', range(1, len(df_itajai_display) + 1))

df_itajai_display['Imports (USD bn)'] = df_itajai_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_itajai_display['Share (%)'] = df_itajai_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_itajai_display['Cumulative (%)'] = df_itajai_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

styler = (
    df_itajai_display.style
    .set_caption(
        f'Itajaí (SC) — SH4 Import Profile, {MAX_YEAR} '
        f'| CO_MUN 4208203 '
        f'| Total: ${itajai_total/1e9:.2f}bn '
        f'| Partial HHI (top-20): {itajai_hhi_partial:.4f} — {hhi_tier}'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_share,      subset=['Share (%)'])
    .map(colour_cumulative, subset=['Cumulative (%)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## SH4 code
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Product name
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Sector SH2
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Itajaí — SH4 import breakdown, 2025
Total (top-20 SH4): $5.49bn
Partial SH4 HHI (top-20 only): 0.0614
Interpretation: Distributed (< 0.10)


Rank,SH4,Product (SH4),Sector (SH2),Imports (USD bn),Share (%),Cumulative (%)
1,8708,Partes e acessórios dos veículos automóveis das posições 8701 a 8705,"Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$0.607bn,3.7%,3.7%
2,3901,"Polímeros de etileno, em formas primárias",Plásticos e suas obras,$0.546bn,3.4%,7.1%
3,7403,"Cobre afinado e ligas de cobre, em formas brutas",Cobre e suas obras,$0.490bn,3.0%,10.1%
4,4011,"Pneumáticos novos, de borracha",Borracha e suas obras,$0.400bn,2.5%,12.5%
5,3002,"Sangue humano; sangue animal preparado para usos terapêuticos, profilácticos ou de diagnóstico; anti-soros, outras fracções do sangue, produtos imunológicos modificados, mesmo obtidos por via biotecnológica; vacinas, toxinas, culturas de microrganismos (e",Produtos farmacêuticos,$0.365bn,2.2%,14.8%
6,3004,"Medicamentos (exceto os produtos das posições 3002, 3005 ou 3006) constituídos por produtos misturados ou não misturados, preparados para fins terapêuticos ou profilácticos, apresentados em doses (incluindo os destinados a serem administrados por via sub",Produtos farmacêuticos,$0.267bn,1.6%,16.4%
7,3822,"Reagentes de diagnóstico ou de laboratório, em qualquer suporte ou preparados, exceto os das posições 3002 ou 3006; materiais de referência certificados",Produtos diversos das indústrias químicas,$0.244bn,1.5%,17.9%
8,2202,"Águas, incluídas as águas minerais e as águas gaseificadas, adicionadas de açúcar ou de outros edulcorantes ou aromatizadas e outras bebidas não alcoólicas, exceto sumos de frutas ou de produtos hortícolas, da posição 2009","Bebidas, líquidos alcoólicos e vinagres",$0.239bn,1.5%,19.4%
9,2004,"Outros produtos hortícolas preparados ou conservados, exceto em vinagre ou em ácido acético, congelados, com excepção dos produtos da posição 2006","Preparações de produtos hortícolas, de frutas ou de outras partes de plantas",$0.233bn,1.4%,20.8%
10,1509,"Azeite de oliveira e respectivas fracções, mesmo refinados, mas não quimicamente modificados",Gorduras e óleos animais ou vegetais; produtos da sua dissociação; gorduras alimentares elaboradas; ceras de origem animal ou vegetal,$0.230bn,1.4%,22.2%


**Table 2 — Annual Trade Balance Time Series:** Identifies when the deficit began and at what rate it accelerated. Cross-reference the dominant SH4 sector from Table 1 against the deficit onset year — if the same sector drove the post-2010 acceleration, that sector is the structural cause.

In [67]:
## Itajaí deficit time series — annual totals from imp_mun and exp_mun

query_itajai_ts = """
    SELECT 'imp' AS flow, "CO_ANO", SUM("VL_FOB") AS vl_fob
    FROM imp_mun WHERE "CO_MUN" = 4208203 GROUP BY "CO_ANO"
    UNION ALL
    SELECT 'exp' AS flow, "CO_ANO", SUM("VL_FOB") AS vl_fob
    FROM exp_mun WHERE "CO_MUN" = 4208203 GROUP BY "CO_ANO"
    ORDER BY "CO_ANO", flow;
"""

df_itajai_ts_raw = pd.read_sql(query_itajai_ts, engine)
df_itajai_ts = (
    df_itajai_ts_raw
    .pivot(index='CO_ANO', columns='flow', values='vl_fob')
    .fillna(0)
    .reset_index()
)
df_itajai_ts['imp_bn']     = df_itajai_ts['imp'] / 1e9
df_itajai_ts['exp_bn']     = df_itajai_ts['exp'] / 1e9
df_itajai_ts['balance_bn'] = df_itajai_ts['exp_bn'] - df_itajai_ts['imp_bn']

deficit_onset = df_itajai_ts[df_itajai_ts['balance_bn'] < 0]['CO_ANO'].min()

print(f'Itajaí — annual trade balance, 1997–{MAX_YEAR}')
print(f'Deficit onset: {deficit_onset}')

## Styled table: Itajaí annual trade balance time series

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

df_itajai_ts_display = df_itajai_ts[[
    'CO_ANO', 'exp_bn', 'imp_bn', 'balance_bn'
]].rename(columns={
    'CO_ANO'    : 'Year',
    'exp_bn'    : 'Exports (USD bn)',
    'imp_bn'    : 'Imports (USD bn)',
    'balance_bn': 'Balance (USD bn)',
}).copy()

df_itajai_ts_display['Exports (USD bn)'] = df_itajai_ts_display['Exports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_itajai_ts_display['Imports (USD bn)'] = df_itajai_ts_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_itajai_ts_display['Balance (USD bn)'] = df_itajai_ts_display['Balance (USD bn)'].apply(
    lambda x: f'+${x:.3f}bn' if x >= 0 else f'-${abs(x):.3f}bn'
)

def colour_year(val):
    if val == deficit_onset:
        return f'color: {RED}; font-weight: 700;'
    if val in [1997, 2000, 2005, 2010, 2015, 2020, MAX_YEAR]:
        return f'color: {DARK_BLUE}; font-weight: 700;'
    return f'color: {GREY_TEXT};'

def colour_balance(val):
    if val.startswith('+'):
        return f'color: {GREEN}; font-weight: 600;'
    elif val.startswith('-'):
        try:
            v = abs(float(val.replace('-$', '').replace('bn', '')))
        except (ValueError, AttributeError):
            return f'color: {RED}; font-weight: 600;'
        if v >= 5.0:
            return f'color: {RED}; font-weight: 700;'
        elif v >= 1.0:
            return f'color: {RED}; font-weight: 600;'
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {GREY_TEXT};'

styler = (
    df_itajai_ts_display.style
    .set_caption(
        f'Itajaí (SC) — Annual Trade Balance, 1997–{MAX_YEAR} '
        f'| CO_MUN 4208203 '
        f'| Deficit onset: {deficit_onset} '
        f'| Snapshot years and onset year highlighted'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'right',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_year,    subset=['Year'])
    .map(colour_balance, subset=['Balance (USD bn)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Year
            ('text-align', 'left'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## Exports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Balance
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'2px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Itajaí — annual trade balance, 1997–2025
Deficit onset: 2009


Year,Exports (USD bn),Imports (USD bn),Balance (USD bn)
1997,$0.185bn,$0.045bn,+$0.140bn
1998,$0.312bn,$0.090bn,+$0.222bn
1999,$0.213bn,$0.063bn,+$0.151bn
2000,$0.206bn,$0.058bn,+$0.148bn
2001,$0.454bn,$0.079bn,+$0.375bn
2002,$0.677bn,$0.206bn,+$0.471bn
2003,$1.095bn,$0.206bn,+$0.889bn
2004,$1.416bn,$0.428bn,+$0.987bn
2005,$2.006bn,$0.757bn,+$1.250bn
2006,$1.868bn,$1.081bn,+$0.787bn


**Table 3 — NCM Detail for Dominant SH4 Sector (SC state):** Drills to product-code level within the top SH4 sector identified in Table 1. Confirms the specific product identity within the SC state import register.

In [68]:
## Itajaí: NCM-level detail for top SH4 sector — SC state context
## Uses state-level imp table filtered to SC

sh4_itajai_top   = int(df_itajai_sh4.iloc[0]['SH4'])
sh4_itajai_label = df_itajai_sh4.iloc[0]['sh4_description']

#print(f'Top SH4 sector for Itajaí: {sh4_itajai_top} — {sh4_itajai_label}')

query_itajai_ncm_sc = f"""
    WITH sc_sh4_ncm AS (
        SELECT
            i."CO_NCM",
            SUM(i."VL_FOB")     AS vl_fob,
            SUM(i."KG_LIQUIDO") AS kg_liq
        FROM imp i
        WHERE i."CO_ANO"    = {MAX_YEAR}
          AND i."SG_UF_NCM" = 'SC'
          AND CAST(LEFT(i."CO_NCM"::TEXT, 4) AS INT) = {sh4_itajai_top}
        GROUP BY i."CO_NCM"
    )
    SELECT
        r."CO_NCM",
        n.nome_ncm                                       AS ncm_description,
        r.vl_fob,
        r.kg_liq,
        ROUND(
            100.0 * r.vl_fob / SUM(r.vl_fob) OVER (), 2
        )                                                AS pct_of_total
    FROM sc_sh4_ncm r
    LEFT JOIN ncm n ON r."CO_NCM" = n.codigo_ncm
    ORDER BY r.vl_fob DESC
    LIMIT 20;
"""

df_itajai_ncm_sc = pd.read_sql(query_itajai_ncm_sc, engine)
df_itajai_ncm_sc['vl_fob_bn']      = df_itajai_ncm_sc['vl_fob'] / 1e9
df_itajai_ncm_sc['cumulative_pct'] = df_itajai_ncm_sc['pct_of_total'].cumsum().round(1)

ncm_sc_total = df_itajai_ncm_sc['vl_fob'].sum()
#print(f'SC state — SH4 {sh4_itajai_top} NCM breakdown | Total shown: ${ncm_sc_total/1e9:.3f}bn')

## Styled table: Itajaí NCM-level breakdown for dominant SH4 sector

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

df_itajai_ncm_display = df_itajai_ncm_sc[[
    'CO_NCM', 'ncm_description', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'CO_NCM'         : 'NCM',
    'ncm_description': 'Product Description',
    'vl_fob_bn'      : 'Imports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_itajai_ncm_display.insert(0, 'Rank', range(1, len(df_itajai_ncm_display) + 1))

df_itajai_ncm_display['Imports (USD bn)'] = df_itajai_ncm_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.4f}bn'
)
df_itajai_ncm_display['Share (%)'] = df_itajai_ncm_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_itajai_ncm_display['Cumulative (%)'] = df_itajai_ncm_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

styler = (
    df_itajai_ncm_display.style
    .set_caption(
        f'Itajaí (SC) — NCM-Level Import Detail, {MAX_YEAR} '
        f'| SH4 {sh4_itajai_top}: {sh4_itajai_label} '
        f'| Source: SC state-level imp table '
        f'| Total shown: ${ncm_sc_total/1e9:.3f}bn '
        f'| Top 20 NCM codes by value'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_share,      subset=['Share (%)'])
    .map(colour_cumulative, subset=['Cumulative (%)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## NCM code
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Product description
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Rank,NCM,Product Description,Imports (USD bn),Share (%),Cumulative (%)
1,87084080,Outras caixas de marchas,$0.3928bn,40.7%,40.7%
2,87089990,Outras partes e acessórios para tratores e veículos automóveis,$0.0920bn,9.5%,50.2%
3,87082999,Outras partes e acessórios de carrocerias para veículos automóveis,$0.0833bn,8.6%,58.9%
4,87088000,Amortecedores de suspensão para tratores e veículos automóveis,$0.0545bn,5.6%,64.5%
5,87089521,Bolsas infláveis para airbags,$0.0481bn,5.0%,69.5%
6,87089522,"Sistema de insuflação, para dispositivo de segurança para automóveis (airbag)",$0.0410bn,4.2%,73.7%
7,87089483,Caixas de direção para veículos automóveis,$0.0394bn,4.1%,77.8%
8,87083090,"Outros freios e partes, para tratores/veículos automóveis",$0.0312bn,3.2%,81.1%
9,87089100,Radiadores para tratores e veículos automóveis,$0.0286bn,3.0%,84.0%
10,87085099,"Outros eixos e partes, para veículos automóveis",$0.0267bn,2.8%,86.8%


#### Overview

The corrected HHI confirms the logistics hub hypothesis. Itajaí's partial SH4 HHI of 0.0614 places it firmly in the Distributed category (< 0.10), and the sector breadth of the top-20 table makes the case conclusively — 20 distinct SH4 codes spanning 15 different SH2 sectors, with no single code exceeding 3.7% of the \\$5.49bn top-20 total.

The leading sector, vehicle parts and accessories (8708, \\$0.607bn, 3.7%), is consistent with supply inputs for Santa Catarina's automotive industry. What follows it is structurally telling: ethylene polymers (3901, \\$0.546bn, 3.4%), refined copper (7403, \\$0.490bn, 3.0%), new rubber tyres (4011, \\$0.400bn, 2.5%), pharmaceuticals (3002 and 3004 combined, \\$0.632bn, 3.8%), diagnostic reagents (3822, \\$0.244bn, 1.5%), beverages (2202, \\$0.239bn, 1.5%), preserved vegetables (2004, \\$0.233bn, 1.4%), olive oil (1509, \\$0.230bn, 1.4%), electrical transformers (8504, \\$0.221bn, 1.4%), semiconductors (8541, \\$0.204bn, 1.2%), and gaming equipment (9504, \\$0.204bn, 1.2%). This is not the product profile of any identifiable single-industry cluster. It is the profile of a major port handling inbound cargo for a broad, diversified regional economy.

**The total warrants attention**. The corrected top-20 SH4 total is \\$5.49bn — meaning the top 20 SH4 codes account for only 33.7% of Itajaí's \\$16.31bn in registered imports. The remaining 66.3% ($10.82bn) is distributed across SH4 codes ranked 21 and below. This is an unusually long tail and reinforces the logistics hub interpretation: no cluster of products drives the import volume, and no sector thins out the distribution in the way that a manufacturing input dependency would.

**The time series remains the primary structural finding**. As established from the clean Table 2 data: Itajaí ran surpluses through 2008, entered deficit in 2009 (-\\$0.27bn), and accelerated sharply after 2020 — from -\\$3.75bn (2019) to -\\$6.54bn (2021), -\\$8.48bn (2022), and -\\$10.19bn (2025). The \\$3.87bn single-year import increase in 2021 has no corresponding product-side explanation visible in the distributed SH4 profile, which is consistent with the registration shift interpretation. No single sector grew enough to account for that step-change; the increase was broad-based across the port's cargo mix.

**The NCM detail for SH4 8708 (Table 3) shows genuine automotive supply chain imports**. Gearboxes (87084080) lead at \\$0.39bn (40.7% of the \\$0.94bn SH4 8708 subtotal), followed by general parts (87089990, 9.5%), body components (87082999, 8.6%), suspension (87088000, 5.6%), and airbag systems (87089521/22, combined 9.2%). This is a coherent automotive tier-1 and tier-2 supplier import profile, consistent with BMW Araquari and the SC supplier base, but at \\$0.94bn it represents only 5.8% of Itajaí's total registered imports — a relevant sector within the port but not the structural driver of the deficit.

#### Business Relevance
The distributed HHI and the 20-sector breadth establish that Itajaí cannot be commercially characterised as a single-industry import hub. The port serves a genuinely diverse cargo base with automotive components, industrial polymers, metals, pharmaceuticals, food products, electronics, and textiles all appear in meaningful volumes within the top 20 alone. For logistics and port services providers, this breadth is the commercial opportunity: Itajaí handles the full range of inbound cargo for Santa Catarina and potentially parts of Paraná and Rio Grande do Sul, making it a mandatory node in any South region logistics strategy. For product-specific suppliers, the relevant finding is that their sector competes for share within a highly distributed port rather than dominating a specialist hub, counterparty relationships are with the importing companies registered in Itajaí, not with a concentrated buyer base located there.

---
---


### 4.1.3 — Petrópolis (Rio de Janeiro): High-Value Import Anomaly — Registration or Industrial Demand?

**Open flag:** Petrópolis ranked 5th nationally by import value at \\$9.47bn with only \\$0.60bn in exports (\\$-8.87bn deficit). No established industrial or logistics profile exists at this scale. The deficit emerged post-2012. This is the highest-priority unresolved anomaly from Step 3.

**CO_MUN:** 3304557


**Table 1 — SH4 Import Profile:** Identifies what product categories Petrópolis is registering. Focus on the top 1–2 sectors and compare against its known industrial base (textiles, glass, precision instruments). A dominant sector inconsistent with any identifiable local industry strengthens the registration anomaly hypothesis.

In [3]:
## Petrópolis SH4-level import profile from imp_mun
## CO_MUN for Petrópolis (RJ): 3304557
## imp_mun columns: CO_ANO, CO_MES, SH4, CO_PAIS, SG_UF_MUN, CO_MUN, KG_LIQUIDO, VL_FOB

query_petropolis_sh4 = f"""
    WITH pet_sh4 AS (
        SELECT
            im."SH4",
            SUM(im."VL_FOB")     AS vl_fob,
            SUM(im."KG_LIQUIDO") AS kg_liq
        FROM imp_mun im
        WHERE im."CO_MUN" = 3304557
          AND im."CO_ANO" = {MAX_YEAR}
        GROUP BY im."SH4"
    ),
    ncm_sh_dedup AS (
        SELECT DISTINCT ON (codigo_sh4)
            codigo_sh4,
            descricao_sh4,
            descricao_sh2,
            codigo_sh2
        FROM ncm_sh
        ORDER BY codigo_sh4
    )
    SELECT
        p."SH4",
        sh.descricao_sh4                                 AS sh4_description,
        sh.descricao_sh2                                 AS sh2_description,
        sh.codigo_sh2,
        p.vl_fob,
        p.kg_liq,
        ROUND(
            100.0 * p.vl_fob / SUM(p.vl_fob) OVER (), 2
        )                                                AS pct_of_total
    FROM pet_sh4 p
    LEFT JOIN ncm_sh_dedup sh ON p."SH4" = sh.codigo_sh4
    ORDER BY p.vl_fob DESC
    LIMIT 20;
"""

df_petropolis_sh4 = pd.read_sql(query_petropolis_sh4, engine)
df_petropolis_sh4['vl_fob_bn']      = df_petropolis_sh4['vl_fob'] / 1e9
df_petropolis_sh4['cumulative_pct'] = df_petropolis_sh4['pct_of_total'].cumsum().round(1)

pet_total = df_petropolis_sh4['vl_fob'].sum()

## Styled table: Petrópolis SH4 import profile

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

df_pet_display = df_petropolis_sh4[[
    'SH4', 'sh4_description', 'sh2_description', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'SH4'            : 'SH4',
    'sh4_description': 'Product (SH4)',
    'sh2_description': 'Sector (SH2)',
    'vl_fob_bn'      : 'Imports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_pet_display.insert(0, 'Rank', range(1, len(df_pet_display) + 1))

df_pet_display['Imports (USD bn)'] = df_pet_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_pet_display['Share (%)'] = df_pet_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_pet_display['Cumulative (%)'] = df_pet_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

styler = (
    df_pet_display.style
    .set_caption(
        f'Petrópolis (RJ) — SH4 Import Profile, {MAX_YEAR} '
        f'| CO_MUN 3304557 '
        f'| Total: ${pet_total/1e9:.2f}bn '
        f'| Top 20 SH4 codes by import value'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_share,      subset=['Share (%)'])
    .map(colour_cumulative, subset=['Cumulative (%)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## SH4 code
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Product name
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Sector SH2
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Rank,SH4,Product (SH4),Sector (SH2),Imports (USD bn),Share (%),Cumulative (%)
1,8905,"Barcos-faróis, barcos-bombas, dragas, guindastes flutuantes e outras embarcações em que a navegação é acessória da função principal; docas flutuantes; plataformas de perfuração ou de exploração, flutuantes ou submersíveis",Embarcações e estruturas flutuantes,$2.357bn,23.6%,23.6%
2,2710,"Óleos de petróleo ou de minerais betuminosos, exceto óleos brutos; preparações não especificadas nem compreendidas noutras posições, contendo, em peso, 70 % ou mais de óleos de petróleo ou de minerais betuminosos, os quais devem constituir o seu elemento","Combustíveis minerais, óleos minerais e produtos da sua destilação; matérias betuminosas; ceras minerais",$1.306bn,13.1%,36.7%
3,2716,Energia elétrica,"Combustíveis minerais, óleos minerais e produtos da sua destilação; matérias betuminosas; ceras minerais",$0.900bn,9.0%,45.8%
4,3004,"Medicamentos (exceto os produtos das posições 3002, 3005 ou 3006) constituídos por produtos misturados ou não misturados, preparados para fins terapêuticos ou profilácticos, apresentados em doses (incluindo os destinados a serem administrados por via sub",Produtos farmacêuticos,$0.507bn,5.1%,50.8%
5,2701,"Hulhas; briquetes, bolas e combustíveis sólidos semelhantes, obtidos a partir da hulha","Combustíveis minerais, óleos minerais e produtos da sua destilação; matérias betuminosas; ceras minerais",$0.362bn,3.6%,54.5%
6,3002,"Sangue humano; sangue animal preparado para usos terapêuticos, profilácticos ou de diagnóstico; anti-soros, outras fracções do sangue, produtos imunológicos modificados, mesmo obtidos por via biotecnológica; vacinas, toxinas, culturas de microrganismos (e",Produtos farmacêuticos,$0.358bn,3.6%,58.0%
7,8802,"Outros veículos aéreos (por exemplo: helicópteros, aviões); veículos espaciais (incluídos os satélites) e seus veículos de lançamento e veículos suborbitais","Aeronaves e aparelhos espaciais, e suas partes",$0.307bn,3.1%,61.1%
8,8481,"Torneiras, válvulas (incluídas as redutoras de pressão e as termostáticas) e dispositivos semelhantes, para canalizações, caldeiras, reservatórios, cubas e outros recipientes","Reatores nucleares, caldeiras, máquinas, aparelhos e instrumentos mecânicos, e suas partes",$0.266bn,2.7%,63.8%
9,2711,Gás de petróleo e outros hidrocarbonetos gasosos,"Combustíveis minerais, óleos minerais e produtos da sua destilação; matérias betuminosas; ceras minerais",$0.204bn,2.0%,65.8%
10,8807,"Partes dos aparelhos das posições 88.01, 88.02 ou 88.06","Aeronaves e aparelhos espaciais, e suas partes",$0.196bn,2.0%,67.8%


**Table 2 — Annual Trade Balance Time Series:** Determines when the deficit began. A single-year step-change is consistent with a registration or logistics shift; gradual multi-year growth from a low base is consistent with genuine industrial input demand.

In [70]:
## Petrópolis deficit emergence over time — annual totals from imp_mun and exp_mun

query_pet_imp_ts = """
    SELECT "CO_ANO", SUM("VL_FOB") AS imp_fob
    FROM imp_mun
    WHERE "CO_MUN" = 3304557
    GROUP BY "CO_ANO"
    ORDER BY "CO_ANO";
"""
query_pet_exp_ts = """
    SELECT "CO_ANO", SUM("VL_FOB") AS exp_fob
    FROM exp_mun
    WHERE "CO_MUN" = 3304557
    GROUP BY "CO_ANO"
    ORDER BY "CO_ANO";
"""

df_pet_imp_ts = pd.read_sql(query_pet_imp_ts, engine)
df_pet_exp_ts = pd.read_sql(query_pet_exp_ts, engine)

df_pet_ts = df_pet_imp_ts.merge(df_pet_exp_ts, on='CO_ANO', how='outer').fillna(0)
df_pet_ts['imp_fob_bn'] = df_pet_ts['imp_fob'] / 1e9
df_pet_ts['exp_fob_bn'] = df_pet_ts['exp_fob'] / 1e9
df_pet_ts['balance_bn'] = df_pet_ts['exp_fob_bn'] - df_pet_ts['imp_fob_bn']

#print(f'Petrópolis — annual trade balance, 1997–{MAX_YEAR}')

## Styled table: Petrópolis annual trade balance time series

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

## Identify deficit onset year for annotation
deficit_onset = df_pet_ts[df_pet_ts['balance_bn'] < 0]['CO_ANO'].min()

df_pet_ts_display = df_pet_ts[[
    'CO_ANO', 'exp_fob_bn', 'imp_fob_bn', 'balance_bn'
]].rename(columns={
    'CO_ANO'     : 'Year',
    'exp_fob_bn' : 'Exports (USD bn)',
    'imp_fob_bn' : 'Imports (USD bn)',
    'balance_bn' : 'Balance (USD bn)',
}).copy()

df_pet_ts_display['Exports (USD bn)'] = df_pet_ts_display['Exports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_pet_ts_display['Imports (USD bn)'] = df_pet_ts_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_pet_ts_display['Balance (USD bn)'] = df_pet_ts_display['Balance (USD bn)'].apply(
    lambda x: f'+${x:.3f}bn' if x >= 0 else f'-${abs(x):.3f}bn'
)

def colour_year(val):
    if val == deficit_onset:
        return f'color: {RED}; font-weight: 700;'
    if val in [1997, 2000, 2005, 2010, 2015, 2020, MAX_YEAR]:
        return f'color: {DARK_BLUE}; font-weight: 700;'
    return f'color: {GREY_TEXT};'

def colour_balance(val):
    if val.startswith('+'):
        return f'color: {GREEN}; font-weight: 600;'
    elif val.startswith('-'):
        try:
            v = abs(float(val.replace('-$', '').replace('bn', '')))
        except (ValueError, AttributeError):
            return f'color: {RED}; font-weight: 600;'
        if v >= 5.0:
            return f'color: {RED}; font-weight: 700;'
        elif v >= 1.0:
            return f'color: {RED}; font-weight: 600;'
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {GREY_TEXT};'

styler = (
    df_pet_ts_display.style
    .set_caption(
        f'Petrópolis (RJ) — Annual Trade Balance, 1997–{MAX_YEAR} '
        f'| CO_MUN 3304557 '
        f'| Deficit onset: {deficit_onset} '
        f'| Snapshot years and onset year highlighted'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'right',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_year,    subset=['Year'])
    .map(colour_balance, subset=['Balance (USD bn)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Year
            ('text-align', 'left'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## Exports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Balance
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'2px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Year,Exports (USD bn),Imports (USD bn),Balance (USD bn)
1997,$2.366bn,$3.098bn,-$0.732bn
1998,$1.982bn,$2.735bn,-$0.753bn
1999,$1.417bn,$2.277bn,-$0.860bn
2000,$1.419bn,$2.438bn,-$1.019bn
2001,$1.511bn,$3.236bn,-$1.725bn
2002,$1.611bn,$3.193bn,-$1.582bn
2003,$1.467bn,$3.197bn,-$1.730bn
2004,$1.921bn,$3.394bn,-$1.472bn
2005,$2.965bn,$4.169bn,-$1.205bn
2006,$2.735bn,$4.343bn,-$1.608bn


**Table 3 — NCM Detail for Dominant SH4 Sector (RJ state):** Drills to product-code level within the top SH4 sector identified in Table 1. This is the confirmation step — Table 1 narrows the sector, Table 3 names the specific product. If the product has no plausible connection to Petrópolis industry or logistics, the registration anomaly is confirmed.

In [71]:
## Petrópolis: NCM-level detail for dominant SH4 sector
## Uses state-level imp table filtered to RJ (SG_UF_NCM = 'RJ')
## Substitute sh4_target after reviewing SH4 output above if needed

sh4_target = int(df_petropolis_sh4.iloc[0]['SH4'])
sh4_label  = df_petropolis_sh4.iloc[0]['sh4_description']

#print(f'Top SH4 sector for Petrópolis: {sh4_target} — {sh4_label}')

query_pet_ncm_rj = f"""
    WITH rj_sh4_ncm AS (
        SELECT
            i."CO_NCM",
            SUM(i."VL_FOB")     AS vl_fob,
            SUM(i."KG_LIQUIDO") AS kg_liq
        FROM imp i
        WHERE i."CO_ANO"    = {MAX_YEAR}
          AND i."SG_UF_NCM" = 'RJ'
          AND CAST(LEFT(i."CO_NCM"::TEXT, 4) AS INT) = {sh4_target}
        GROUP BY i."CO_NCM"
    )
    SELECT
        r."CO_NCM",
        n.nome_ncm                                       AS ncm_description,
        r.vl_fob,
        r.kg_liq,
        ROUND(
            100.0 * r.vl_fob / SUM(r.vl_fob) OVER (), 2
        )                                                AS pct_of_total
    FROM rj_sh4_ncm r
    LEFT JOIN ncm n ON r."CO_NCM" = n.codigo_ncm
    ORDER BY r.vl_fob DESC
    LIMIT 20;
"""

df_pet_ncm_rj = pd.read_sql(query_pet_ncm_rj, engine)
df_pet_ncm_rj['vl_fob_bn']      = df_pet_ncm_rj['vl_fob'] / 1e9
df_pet_ncm_rj['cumulative_pct'] = df_pet_ncm_rj['pct_of_total'].cumsum().round(1)

ncm_total = df_pet_ncm_rj['vl_fob'].sum()
#print(f'RJ state — SH4 {sh4_target} NCM breakdown | Total shown: ${ncm_total/1e9:.3f}bn')

## Styled table: Petrópolis NCM-level breakdown for dominant SH4 sector

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

df_pet_ncm_display = df_pet_ncm_rj[[
    'CO_NCM', 'ncm_description', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'CO_NCM'         : 'NCM',
    'ncm_description': 'Product Description',
    'vl_fob_bn'      : 'Imports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_pet_ncm_display.insert(0, 'Rank', range(1, len(df_pet_ncm_display) + 1))

df_pet_ncm_display['Imports (USD bn)'] = df_pet_ncm_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.4f}bn'
)
df_pet_ncm_display['Share (%)'] = df_pet_ncm_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_pet_ncm_display['Cumulative (%)'] = df_pet_ncm_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

styler = (
    df_pet_ncm_display.style
    .set_caption(
        f'Petrópolis (RJ) — NCM-Level Import Detail, {MAX_YEAR} '
        f'| SH4 {sh4_target}: {sh4_label} '
        f'| Source: RJ state-level imp table '
        f'| Top 20 NCM codes by value'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_share,      subset=['Share (%)'])
    .map(colour_cumulative, subset=['Cumulative (%)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## NCM code
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Product description
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Rank,NCM,Product Description,Imports (USD bn),Share (%),Cumulative (%)
1,89052000,"Plataformas de perfuração ou de exploração, flutuantes ou submersíveis",$2.3568bn,100.0%,100.0%
2,89059000,"Barcos-faróis/guindastes/docas/diques flutuantes, etc.",$0.0000bn,0.0%,100.0%
3,89051000,Dragas,$0.0000bn,0.0%,100.0%


#### Overview

Petrópolis is not an import anomaly. It is a petroleum, energy and defence procurement registration address — and the corrected SH4 profile makes this unambiguous.

Offshore drilling platforms (SH4 8905) lead at \\$2.36bn, representing 23.6% of the \\$7.69bn total — by far the dominant single sector. Refined petroleum products (2710, \\$1.31bn, 13.1%) and electricity (2716, \\$0.90bn, 9.0%) follow. Coal (2701, \\$0.36bn, 3.6%) and petroleum gas (2711, $0.20bn, 2.0%) add further energy sector weight. Combined, the five energy and petroleum SH4 codes account for \\$5.13bn — 66.7% of Petrópolis's total registered imports. The remaining third is split across pharmaceuticals (3004 and 3002 combined, \\$0.87bn, 11.3%), aircraft and parts (8802 and 8807 combined, \\$0.50bn, 6.5%), industrial valves and machinery (8481, 8479, 8421), and defence equipment (9306, bombs, missiles and munitions, \\$0.17bn, 1.7%). None of these categories has any plausible connection to Petrópolis's actual industrial base of textiles, glass and precision instruments. The registration anomaly is confirmed across the full product range, not just the leading sector.

The energy concentration at 66.7% within a single municipality's import register is the defining structural finding. Offshore drilling platforms at 23.6% alone would be sufficient to confirm petroleum registration. The additional weight of refined products, electricity, coal and gas — all procured at scale for industrial and offshore operations — establishes that Petrópolis functions as a legal registration address for a cluster of energy sector companies, consistent with energy sector companies whose operational geography is offshore or in other states.

The defence import line (9306, \\$0.17bn, 1.7%) warrants a note. Bombs, grenades, torpedoes, missiles and munitions appearing in a municipal import register are almost universally registered through the legal address of the procuring entity — in Brazil typically the Ministério da Defesa or a state-owned defence contractor — rather than reflecting any physical presence of the goods in the registered municipality. This is consistent with the broader registration pattern throughout the Petrópolis profile.

The time series confirms the registration interpretation from the supply side. Running deficits from 1997 through 2015 — as petroleum capital equipment imports registered through these addresses outpaced export registration — the balance reversed into surplus in 2016 and accelerated to +\\$30.05bn in 2025 far faster than import registration growth. The \\$7.69bn in 2025 imports is therefore the residual petroleum procurement figure that has not been overtaken by the export side — not a demand signal from the city of Petrópolis itself.

⚠️ Fix required in Step 3. All references to Petrópolis as a "post-2012 deficit municipality" should be corrected. The municipality ran deficits from 1997 onward; the post-2015 development is a sustained shift into surplus driven by petroleum export registration growth. The direction of the anomaly in Step 3 is inverted.

#### Business Relevance
Petrópolis carries no actionable commercial signal as a demand location. The $7.69bn in registered imports is dominated by petroleum capital equipment, refined products, electricity procurement and defence items. All registered through legal addresses of large state-linked entities with no operational footprint in the city. Its analytical value in this project is purely methodological: it is the clearest demonstration in the dataset that municipality-level import figures require registration-geography verification before any commercial conclusion is drawn, and it belongs in the findings as a caveat on the reliability of municipal import rankings rather than as a commercial target in its own right.

---
---


## 4.2 — Energy and Strategic Minerals: SH27 Petroleum and Ferro-Niobium

Two product-level investigations with national-scale implications. SH27 (mineral fuels) is Brazil's largest single export sector and second largest import sector simultaneously — the crude vs refined split within that figure defines Brazil's refining capacity deficit. Ferro-niobium (NCM 72029300) is a strategic mineral where Brazil holds near-monopoly global supply — confirming destination markets answers the open question of where the world's niobium supply goes.


### 4.2.1 — SH27 Crude vs Refined Petroleum

**Open flag:** Brazil exported \\$55.96bn (SH27) and imported \\$30.53bn (SH27) in 2025. The gross figures mask the structural composition: Brazil is a crude oil exporter and a refined product importer — it exports the raw commodity it cannot fully process domestically and imports the derivatives it needs. The crude/refined split within each figure quantifies the scale of this refining capacity deficit.

**SH4 classification used:** 2709 = Crude Oil | 2710 = Refined Products | 2711 = LNG / Gas | Other = remaining SH27


**Table 1 — SH27 Export Category Summary:** Reports the crude oil / refined products / LNG split for Brazil's \$55.96bn SH27 exports. A dominant Crude Oil share confirms Brazil's role as a raw commodity exporter within the petroleum sector.

In [72]:
## SH27 export composition — crude vs refined split
## SH4 2709 = crude oil | 2710 = refined products | 2711 = LNG / gas

query_sh27_exp = f"""
    WITH sh27_exp AS (
        SELECT
            e."CO_NCM",
            SUM(e."VL_FOB")     AS vl_fob,
            SUM(e."KG_LIQUIDO") AS kg_liq
        FROM exp e
        WHERE e."CO_ANO" = {MAX_YEAR}
          AND CAST(LEFT(e."CO_NCM"::TEXT, 2) AS INT) = 27
        GROUP BY e."CO_NCM"
    )
    SELECT
        p."CO_NCM",
        n.nome_ncm                                           AS ncm_description,
        p.vl_fob,
        p.kg_liq,
        ROUND(100.0 * p.vl_fob / SUM(p.vl_fob) OVER (), 2) AS pct_of_total,
        CASE
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 2709 THEN 'Crude Oil'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 2710 THEN 'Refined Products'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 2711 THEN 'LNG / Gas'
            ELSE 'Other SH27'
        END                                                  AS sh4_category
    FROM sh27_exp p
    LEFT JOIN ncm n ON p."CO_NCM" = n.codigo_ncm
    ORDER BY p.vl_fob DESC
    LIMIT 20;
"""

df_sh27_exp = pd.read_sql(query_sh27_exp, engine)
df_sh27_exp['vl_fob_bn']      = df_sh27_exp['vl_fob'] / 1e9
df_sh27_exp['cumulative_pct'] = df_sh27_exp['pct_of_total'].cumsum().round(1)

sh27_exp_total   = df_sh27_exp['vl_fob'].sum()
sh27_exp_summary = (
    df_sh27_exp.groupby('sh4_category')['vl_fob']
    .sum()
    .sort_values(ascending=False)
)

#print(f'SH27 export composition by category, {MAX_YEAR} (${sh27_exp_total/1e9:.2f}bn total):')
#for cat, val in sh27_exp_summary.items():
#    print(f'  {cat}: ${val/1e9:.2f}bn ({100*val/sh27_exp_total:.1f}%)')

## Styled tables: SH27 export composition — summary + NCM detail

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

## Category colour map — crude = amber (raw), refined = blue (processed), gas = green
CAT_COLOURS = {
    'Crude Oil'       : AMBER,
    'Refined Products': '#1565c0',
    'LNG / Gas'       : GREEN,
    'Other SH27'      : GREY_TEXT,
}

## ── Table 1: category summary ─────────────────────────────────────────────────
df_sh27_exp_summary = sh27_exp_summary.reset_index().rename(columns={
    'sh4_category': 'Category',
    'vl_fob'      : 'raw',
})
df_sh27_exp_summary['Exports (USD bn)'] = df_sh27_exp_summary['raw'].apply(
    lambda x: f'${x/1e9:.2f}bn'
)
df_sh27_exp_summary['Share (%)'] = df_sh27_exp_summary['raw'].apply(
    lambda x: f'{100*x/sh27_exp_total:.1f}%'
)
df_sh27_exp_summary = df_sh27_exp_summary.drop(columns='raw')

def colour_category(val):
    c = CAT_COLOURS.get(val, GREY_TEXT)
    return f'color: {c}; font-weight: 700;'

def colour_share_summary(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 50.0:
        return f'color: {RED}; font-weight: 700;'
    elif v >= 20.0:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

display(
    df_sh27_exp_summary.style
    .set_caption(
        f'Brazil — SH27 Export Composition by Category, {MAX_YEAR} '
        f'| Total: ${sh27_exp_total/1e9:.2f}bn'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_category,       subset=['Category'])
    .map(colour_share_summary,  subset=['Share (%)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
            ('margin-bottom', '28px'),
        ]},
    ])
    .hide(axis='index')
)

Category,Exports (USD bn),Share (%)
Crude Oil,$44.54bn,79.6%
Refined Products,$10.33bn,18.5%
Other SH27,$1.01bn,1.8%
LNG / Gas,$0.06bn,0.1%


**Table 2 — SH27 Export NCM Detail:** Full NCM-level breakdown. Row tints indicate category: amber = crude oil, blue = refined products, green = LNG. Identifies the specific crude grades and refined product types within the export mix.

In [73]:
## ── Table 2: NCM detail ───────────────────────────────────────────────────────
df_sh27_exp_display = df_sh27_exp[[
    'CO_NCM', 'ncm_description', 'sh4_category', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'CO_NCM'         : 'NCM',
    'ncm_description': 'Product Description',
    'sh4_category'   : 'Category',
    'vl_fob_bn'      : 'Exports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_sh27_exp_display.insert(0, 'Rank', range(1, len(df_sh27_exp_display) + 1))

df_sh27_exp_display['Exports (USD bn)'] = df_sh27_exp_display['Exports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_sh27_exp_display['Share (%)'] = df_sh27_exp_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_sh27_exp_display['Cumulative (%)'] = df_sh27_exp_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

## Row-level highlight by category
def highlight_category_rows(row):
    cat = row['Category']
    tints = {
        'Crude Oil'       : '#fff8e6',
        'Refined Products': '#e8f0fb',
        'LNG / Gas'       : '#eaf5ee',
        'Other SH27'      : '',
    }
    bg = tints.get(cat, '')
    return [f'background-color: {bg};' if bg else ''] * len(row)

display(
    df_sh27_exp_display.style
    .set_caption(
        f'Brazil — SH27 Export NCM Detail, {MAX_YEAR} '
        f'| Crude Oil | Refined Products | LNG / Gas '
        f'| Top 20 NCM codes by value'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_category_rows, axis=1)
    .map(colour_share,      subset=['Share (%)'])
    .map(colour_cumulative, subset=['Cumulative (%)'])
    .map(colour_category,   subset=['Category'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## NCM code
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Product description
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Category
            ('text-align', 'center'),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Exports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

Rank,NCM,Product Description,Category,Exports (USD bn),Share (%),Cumulative (%)
1,27090010,Óleos brutos de petróleo,Crude Oil,$44.536bn,79.6%,79.6%
2,27101922,Fuel oil,Refined Products,$7.007bn,12.5%,92.1%
3,27101911,Querosenes de aviação,Refined Products,$2.028bn,3.6%,95.7%
4,27101259,"Outras gasolinas, exceto para aviação",Refined Products,$0.593bn,1.1%,96.8%
5,27131100,Coque de petróleo não calcinado,Other SH27,$0.435bn,0.8%,97.6%
6,27101921,Gasóleo (óleo diesel),Refined Products,$0.410bn,0.7%,98.3%
7,27160000,Energia elétrica,Other SH27,$0.311bn,0.6%,98.9%
8,27131200,Coque de petróleo calcinado,Other SH27,$0.190bn,0.3%,99.2%
9,27101932,Óleos lubrificantes com aditivos,Refined Products,$0.169bn,0.3%,99.5%
10,27111100,Gás natural liquefeito,LNG / Gas,$0.062bn,0.1%,99.6%


**Table 3 — SH27 Import vs Export Composition (side-by-side):** The Gap column (Imports − Exports per category) directly quantifies the refining capacity deficit. A positive gap on Refined Products is the key figure — it represents the USD value of refined petroleum Brazil imports that it cannot produce domestically at sufficient scale.

In [74]:
## SH27 import composition — mirror query

query_sh27_imp = f"""
    WITH sh27_imp AS (
        SELECT
            i."CO_NCM",
            SUM(i."VL_FOB")     AS vl_fob,
            SUM(i."KG_LIQUIDO") AS kg_liq
        FROM imp i
        WHERE i."CO_ANO" = {MAX_YEAR}
          AND CAST(LEFT(i."CO_NCM"::TEXT, 2) AS INT) = 27
        GROUP BY i."CO_NCM"
    )
    SELECT
        p."CO_NCM",
        n.nome_ncm                                           AS ncm_description,
        p.vl_fob,
        p.kg_liq,
        ROUND(100.0 * p.vl_fob / SUM(p.vl_fob) OVER (), 2) AS pct_of_total,
        CASE
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 2709 THEN 'Crude Oil'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 2710 THEN 'Refined Products'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 2711 THEN 'LNG / Gas'
            ELSE 'Other SH27'
        END                                                  AS sh4_category
    FROM sh27_imp p
    LEFT JOIN ncm n ON p."CO_NCM" = n.codigo_ncm
    ORDER BY p.vl_fob DESC
    LIMIT 20;
"""

df_sh27_imp = pd.read_sql(query_sh27_imp, engine)
df_sh27_imp['vl_fob_bn']      = df_sh27_imp['vl_fob'] / 1e9
df_sh27_imp['cumulative_pct'] = df_sh27_imp['pct_of_total'].cumsum().round(1)

sh27_imp_total   = df_sh27_imp['vl_fob'].sum()
sh27_imp_summary = (
    df_sh27_imp.groupby('sh4_category')['vl_fob']
    .sum()
    .sort_values(ascending=False)
)

## Refining gap: refined imports minus refined exports
refined_imp = sh27_imp_summary.get('Refined Products', 0)
refined_exp = sh27_exp_summary.get('Refined Products', 0)
refining_gap = refined_imp - refined_exp

#print(f'SH27 import composition by category, {MAX_YEAR} (${sh27_imp_total/1e9:.2f}bn total):')
#for cat, val in sh27_imp_summary.items():
#    print(f'  {cat}: ${val/1e9:.2f}bn ({100*val/sh27_imp_total:.1f}%)')
#print(f'\nRefining gap (refined imports − refined exports): ${refining_gap/1e9:.2f}bn')

## Styled tables: SH27 import composition — summary + NCM detail

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

CAT_COLOURS = {
    'Crude Oil'       : AMBER,
    'Refined Products': '#1565c0',
    'LNG / Gas'       : GREEN,
    'Other SH27'      : GREY_TEXT,
}

## ── Table 1: category summary with export comparison column ───────────────────
df_sh27_imp_summary = sh27_imp_summary.reset_index().rename(columns={
    'sh4_category': 'Category',
    'vl_fob'      : 'raw_imp',
})

## Merge export values for side-by-side comparison
df_sh27_exp_ref = sh27_exp_summary.reset_index().rename(columns={
    'sh4_category': 'Category',
    'vl_fob'      : 'raw_exp',
})
df_sh27_imp_summary = df_sh27_imp_summary.merge(df_sh27_exp_ref, on='Category', how='left').fillna(0)
df_sh27_imp_summary['raw_gap'] = df_sh27_imp_summary['raw_imp'] - df_sh27_imp_summary['raw_exp']

df_sh27_imp_summary['Imports (USD bn)'] = df_sh27_imp_summary['raw_imp'].apply(
    lambda x: f'${x/1e9:.2f}bn'
)
df_sh27_imp_summary['Exports (USD bn)'] = df_sh27_imp_summary['raw_exp'].apply(
    lambda x: f'${x/1e9:.2f}bn'
)
df_sh27_imp_summary['Gap (USD bn)'] = df_sh27_imp_summary['raw_gap'].apply(
    lambda x: f'+${x/1e9:.2f}bn' if x >= 0 else f'-${abs(x)/1e9:.2f}bn'
)
df_sh27_imp_summary['Imp Share (%)'] = df_sh27_imp_summary['raw_imp'].apply(
    lambda x: f'{100*x/sh27_imp_total:.1f}%'
)
df_sh27_imp_summary = df_sh27_imp_summary[[
    'Category', 'Imports (USD bn)', 'Imp Share (%)', 'Exports (USD bn)', 'Gap (USD bn)'
]]

def colour_category(val):
    c = CAT_COLOURS.get(val, GREY_TEXT)
    return f'color: {c}; font-weight: 700;'

def colour_gap(val):
    if val.startswith('+'):
        return f'color: {RED}; font-weight: 700;'   ## import surplus = dependency
    elif val.startswith('-'):
        return f'color: {GREEN}; font-weight: 600;'  ## export surplus = capacity
    return f'color: {GREY_TEXT};'

def colour_share_summary(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 50.0:
        return f'color: {RED}; font-weight: 700;'
    elif v >= 20.0:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

display(
    df_sh27_imp_summary.style
    .set_caption(
        f'Brazil — SH27 Import vs Export Composition by Category, {MAX_YEAR} '
        f'| Imports: \\${sh27_imp_total/1e9:.2f}bn '
        f'| Refining gap: \\${refining_gap/1e9:.2f}bn '
        f'| Gap = Imports − Exports per category'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_category,      subset=['Category'])
    .map(colour_share_summary, subset=['Imp Share (%)'])
    .map(colour_gap,           subset=['Gap (USD bn)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'2px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
            ('margin-bottom', '28px'),
        ]},
    ])
    .hide(axis='index')
)

Category,Imports (USD bn),Imp Share (%),Exports (USD bn),Gap (USD bn)
Refined Products,$15.23bn,51.1%,$10.33bn,+$4.90bn
Crude Oil,$6.61bn,22.2%,$44.54bn,-$37.92bn
Other SH27,$4.60bn,15.4%,$1.01bn,+$3.59bn
LNG / Gas,$3.39bn,11.4%,$0.06bn,+$3.33bn


**Table 4 — SH27 Import NCM Detail:** Mirror of Table 2 for the import side. Identifies the specific refined product types (gasoline, diesel, aviation fuel) driving the \$30.53bn import figure.

In [75]:
## ── Table 2: NCM detail ───────────────────────────────────────────────────────
df_sh27_imp_display = df_sh27_imp[[
    'CO_NCM', 'ncm_description', 'sh4_category', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'CO_NCM'         : 'NCM',
    'ncm_description': 'Product Description',
    'sh4_category'   : 'Category',
    'vl_fob_bn'      : 'Imports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_sh27_imp_display.insert(0, 'Rank', range(1, len(df_sh27_imp_display) + 1))

df_sh27_imp_display['Imports (USD bn)'] = df_sh27_imp_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_sh27_imp_display['Share (%)'] = df_sh27_imp_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_sh27_imp_display['Cumulative (%)'] = df_sh27_imp_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

def highlight_category_rows(row):
    tints = {
        'Crude Oil'       : '#fff8e6',
        'Refined Products': '#e8f0fb',
        'LNG / Gas'       : '#eaf5ee',
        'Other SH27'      : '',
    }
    bg = tints.get(row['Category'], '')
    return [f'background-color: {bg};' if bg else ''] * len(row)

display(
    df_sh27_imp_display.style
    .set_caption(
        f'Brazil — SH27 Import NCM Detail, {MAX_YEAR} '
        f'| Crude Oil | Refined Products | LNG / Gas '
        f'| Top 20 NCM codes by value'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_category_rows, axis=1)
    .map(colour_share,      subset=['Share (%)'])
    .map(colour_cumulative, subset=['Cumulative (%)'])
    .map(colour_category,   subset=['Category'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [
            ('text-align', 'center'),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

Rank,NCM,Product Description,Category,Imports (USD bn),Share (%),Cumulative (%)
1,27101921,Gasóleo (óleo diesel),Refined Products,$9.492bn,31.1%,31.1%
2,27090010,Óleos brutos de petróleo,Crude Oil,$6.612bn,21.7%,52.8%
3,27011200,"Hulha betuminosa, não aglomerada",Other SH27,$2.190bn,7.2%,59.9%
4,27101241,Naftas para petroquimica,Refined Products,$1.972bn,6.5%,66.4%
5,27111100,Gás natural liquefeito,LNG / Gas,$1.645bn,5.4%,71.8%
6,27101259,"Outras gasolinas, exceto para aviação",Refined Products,$1.594bn,5.2%,77.0%
7,27112100,Gás natural no estado gasoso,LNG / Gas,$1.023bn,3.4%,80.4%
8,27101911,Querosenes de aviação,Refined Products,$0.927bn,3.0%,83.4%
9,27160000,Energia elétrica,Other SH27,$0.911bn,3.0%,86.4%
10,27040012,Coques com granulometria inferior a 80 mm,Other SH27,$0.676bn,2.2%,88.6%


#### Overview

Brazil's petroleum trade structure confirms the commodity-without-refining-capacity thesis. The country exports \\$44.54bn in crude oil (79.6% of SH27 exports) and simultaneously imports \\$9.49bn in diesel — its single largest import line item across all sectors nationally.

The export side is straightforward. A single NCM code — 27090010 (crude petroleum oils) — accounts for \\$44.54bn, or 79.6% of Brazil's \\$55.93bn in SH27 exports. The remaining 20.4% is refined products (\\$10.33bn) and minor categories. Within refined product exports, fuel oil (27101922, \\$7.01bn, 12.5%) dominates — a low-value heavy residual product that Petrobras's refineries produce as a byproduct of crude processing, not a high-margin refined product. Aviation kerosene (\\$2.03bn, 3.6%) and gasoline (\\$0.59bn, 1.1%) follow. The overall picture is a country that extracts crude at scale, exports the majority unprocessed, and exports the lower-value residuals from what domestic refining capacity it does operate.

**The import side inverts this structure precisely**. Diesel (27101921, \\$9.49bn, 31.1%) is the single largest import item — Brazil cannot refine sufficient diesel domestically to meet national demand, which is structurally high given the country's road-dominated freight network and agricultural machinery base. Crude oil imports (\\$6.61bn, 21.7%) appear at rank 2 — Brazil imports specific crude grades its refineries are configured to process that differ from the pre-salt crude its offshore fields predominantly produce. Petrochemical naphtha (27101241, $1.97bn, 6.5%) and gasoline (27101259, \\$1.59bn, 5.2%) follow, confirming import dependency across the core refined product categories.

**The refining gap table quantifies the structural deficit directly**. Brazil imports \\$15.23bn in refined products and exports \\$10.33bn — a net refined product deficit of \\$4.90bn. On LNG and gas, the gap is \\$3.33bn (imports \\$3.39bn, exports \\$0.06bn). The crude oil position is the inverse: a \\$37.92bn net surplus (exports \\$44.54bn, imports \\$6.61bn). The aggregate SH27 balance is therefore strongly positive — Brazil is a large net petroleum exporter in value terms — but the composition reveals a country that monetises its hydrocarbon endowment primarily through crude export rather than through domestic value-added processing.

**The LNG import profile warrants attention**. Natural gas imports total \\$3.39bn (11.4% of SH27 imports), led by liquefied natural gas (27111100, \\$1.65bn), pipeline gas (27112100, \\$1.02bn), propane (27111290, \\$0.49bn), and butane (27111300, $0.23bn). The end-use split between thermoelectric generation, industrial consumption, and distribution cannot be determined from this data.

Bituminous coal imports total \\$2.19bn (27011200, rank 3). Brazil has no significant domestic coking coal reserves and the product is classified as metallurgical grade, making steel production the most plausible end-use, though buyer-level confirmation is not available from this dataset.

#### Business Relevance
The \\$4.90bn refined product gap is the commercially actionable figure. Brazil is a net importer of the refined products its economy most depends on — diesel for freight and agriculture, naphtha for petrochemicals, gasoline for passenger transport — despite sitting on one of the world's largest offshore crude reserves. This gap is a function of refinery configuration, capacity constraints, and the product slate mismatch between pre-salt crude grades and existing refinery inputs. For energy sector participants, the diesel import dependency (\\$9.49bn) is the primary procurement market — the scale and structural nature of the shortfall makes it a durable commercial opportunity rather than a cyclical one. For logistics and infrastructure providers, the LNG import volume (\\$3.39bn) identifies the receiving terminal and regasification infrastructure as a growth segment as Brazil expands gas-to-power capacity. The crude import figure (\\$6.61bn) reflects refinery feedstock optimisation rather than production insufficiency and is less commercially significant than the refined product gap as a demand signal.

---
---


### 4.2.2 — Ferro-Niobium (NCM 72029300): National Value and Destination Markets

**Open flag:** CBMM (Companhia Brasileira de Metalurgia e Mineração), headquartered in Araxá (MG), produces approximately 80% of the world's niobium supply. The combined figure across Minas Gerais, Goiás and Amazonas was identified as \\$2.657bn in Step 2. Ferro-niobium does not appear in any state's top China export sectors — confirming destination markets answers where the world's primary niobium supply actually routes.


**Output — National Total, State Breakdown and Destination Markets:** The state breakdown confirms Minas Gerais (Araxá/CBMM) as the dominant registrant. The destination market ranking is the primary finding — niobium's absence from state-level China top-partner lists suggests non-China routing; confirm here. Expected destinations: Europe, Japan, South Korea, United States (high-strength steel for automotive and construction).

In [76]:
## Ferro-niobium: national total, state breakdown, destination markets
## NCM 72029300 | SG_UF_NCM joins to uf.sigla for state name

query_niobium = f"""
    SELECT
        e."SG_UF_NCM"       AS sg_uf,
        e."CO_PAIS",
        p.nome_pais         AS partner_country,
        SUM(e."VL_FOB")     AS vl_fob,
        SUM(e."KG_LIQUIDO") AS kg_liq
    FROM exp e
    LEFT JOIN pais p ON e."CO_PAIS" = p.codigo_pais
    WHERE e."CO_NCM" = 72029300
      AND e."CO_ANO" = {MAX_YEAR}
    GROUP BY e."SG_UF_NCM", e."CO_PAIS", p.nome_pais
    ORDER BY vl_fob DESC;
"""

df_niobium = pd.read_sql(query_niobium, engine)
df_niobium['vl_fob_bn'] = df_niobium['vl_fob'] / 1e9
niobium_total = df_niobium['vl_fob'].sum()

niobium_by_state   = df_niobium.groupby('sg_uf')['vl_fob'].sum().sort_values(ascending=False)
niobium_by_partner = df_niobium.groupby('partner_country')['vl_fob'].sum().sort_values(ascending=False).head(15)

print(f'NCM 72029300 (ferro-niobium) — national total: ${niobium_total/1e9:.3f}bn')

## Styled tables: ferro-niobium state breakdown + destination markets

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

TABLE_STYLES = [
    {'selector': 'caption', 'props': [
        ('caption-side', 'top'),
        ('font-size', '14px'),
        ('font-weight', '700'),
        ('font-family', 'Helvetica Neue, Arial, sans-serif'),
        ('color', DARK_BLUE),
        ('padding', '0 0 10px 0'),
        ('text-align', 'left'),
    ]},
    {'selector': 'thead th', 'props': [
        ('background-color', DARK_BLUE),
        ('color', WHITE),
        ('font-size', '12px'),
        ('font-weight', '600'),
        ('text-transform', 'uppercase'),
        ('letter-spacing', '0.04em'),
        ('padding', '10px 14px'),
        ('border', 'none'),
    ]},
    {'selector': 'thead tr:last-child th', 'props': [
        ('border-bottom', f'2px solid {MID_BLUE}'),
    ]},
    {'selector': 'tbody tr:nth-child(odd)', 'props': [
        ('background-color', WHITE),
    ]},
    {'selector': 'tbody tr:nth-child(even)', 'props': [
        ('background-color', STRIPE),
    ]},
    {'selector': 'tbody tr:hover', 'props': [
        ('background-color', LIGHT_BLUE),
    ]},
    {'selector': 'table', 'props': [
        ('border-collapse', 'collapse'),
        ('width', '100%'),
        ('border', f'1px solid {LIGHT_BLUE}'),
        ('border-radius', '6px'),
        ('overflow', 'hidden'),
        ('margin-bottom', '28px'),
    ]},
]

BASE_PROPS = {
    'font-family'  : 'Helvetica Neue, Arial, sans-serif',
    'font-size'    : '13px',
    'text-align'   : 'left',
    'padding'      : '8px 14px',
    'border-bottom': f'1px solid {LIGHT_BLUE}',
}

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 50.0:
        return f'color: {RED}; font-weight: 700;'
    elif v >= 20.0:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

## ── Table 1: state breakdown ──────────────────────────────────────────────────
df_state_display = niobium_by_state.reset_index().rename(columns={
    'sg_uf'  : 'State',
    'vl_fob' : 'raw',
})
df_state_display.insert(0, 'Rank', range(1, len(df_state_display) + 1))
df_state_display['Exports (USD bn)'] = df_state_display['raw'].apply(
    lambda x: f'${x/1e9:.3f}bn'
)
df_state_display['Share (%)'] = df_state_display['raw'].apply(
    lambda x: f'{100*x/niobium_total:.1f}%'
)
df_state_display = df_state_display[['Rank', 'State', 'Exports (USD bn)', 'Share (%)']]

display(
    df_state_display.style
    .set_caption(
        f'NCM 72029300 — Ferro-Niobium Export by State, {MAX_YEAR} '
        f'| National total: ${niobium_total/1e9:.3f}bn'
    )
    .set_properties(**BASE_PROPS)
    .map(colour_share, subset=['Share (%)'])
    .set_table_styles(TABLE_STYLES + [
        {'selector': 'tbody td:nth-child(1)', 'props': [
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('font-family', 'monospace'),
            ('font-weight', '700'),
            ('color', DARK_BLUE),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
    ])
    .hide(axis='index')
)

## ── Table 2: destination markets ──────────────────────────────────────────────
df_partner_display = niobium_by_partner.reset_index().rename(columns={
    'partner_country': 'Destination Market',
    'vl_fob'         : 'raw',
})
df_partner_display.insert(0, 'Rank', range(1, len(df_partner_display) + 1))
df_partner_display['Exports (USD bn)'] = df_partner_display['raw'].apply(
    lambda x: f'${x/1e9:.3f}bn'
)
df_partner_display['Share (%)'] = df_partner_display['raw'].apply(
    lambda x: f'{100*x/niobium_total:.1f}%'
)
df_partner_display['Cumulative (%)'] = df_partner_display['raw'].cumsum().apply(
    lambda x: f'{100*x/niobium_total:.1f}%'
)
df_partner_display = df_partner_display[[
    'Rank', 'Destination Market', 'Exports (USD bn)', 'Share (%)', 'Cumulative (%)'
]]

## Flag China row for visual identification
china_keywords = ['china', 'china,', 'chinese']

def highlight_china(row):
    if any(kw in str(row['Destination Market']).lower() for kw in china_keywords):
        return [f'background-color: #fdf0ee; font-weight: 600;'] * len(row)
    return [''] * len(row)

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

display(
    df_partner_display.style
    .set_caption(
        f'NCM 72029300 — Ferro-Niobium Destination Markets, {MAX_YEAR} '
        f'| National total: ${niobium_total/1e9:.3f}bn '
        f'| Top 15 partners | China row highlighted if present'
    )
    .set_properties(**BASE_PROPS)
    .apply(highlight_china, axis=1)
    .map(colour_share,      subset=['Share (%)'])
    .map(colour_cumulative, subset=['Cumulative (%)'])
    .set_table_styles(TABLE_STYLES + [
        {'selector': 'tbody td:nth-child(1)', 'props': [
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
    ])
    .hide(axis='index')
)

NCM 72029300 (ferro-niobium) — national total: $2.658bn


Rank,State,Exports (USD bn),Share (%)
1,MG,$2.126bn,80.0%
2,GO,$0.424bn,16.0%
3,AM,$0.107bn,4.0%
4,PR,$0.000bn,0.0%
5,SP,$0.000bn,0.0%
6,ND,$0.000bn,0.0%


Rank,Destination Market,Exports (USD bn),Share (%),Cumulative (%)
1,China,$1.323bn,49.8%,49.8%
2,Países Baixos (Holanda),$0.461bn,17.3%,67.1%
3,Estados Unidos,$0.214bn,8.1%,75.2%
4,Singapura,$0.194bn,7.3%,82.4%
5,Coreia do Sul,$0.176bn,6.6%,89.1%
6,Japão,$0.155bn,5.8%,94.9%
7,Índia,$0.030bn,1.1%,96.0%
8,Canadá,$0.020bn,0.7%,96.8%
9,Arábia Saudita,$0.019bn,0.7%,97.5%
10,Turquia,$0.014bn,0.5%,98.0%


#### Overview

Brazil's near-monopoly on global niobium supply routes predominantly to China, but with a materially more diversified destination market than the commodity export pattern established elsewhere in this analysis.

Minas Gerais accounts for \\$2.13bn (80.0%) of the \\$2.66bn national total, confirming CBMM's Araxá operation as the overwhelmingly dominant registrant. Goiás adds \\$0.42bn (16.0%) and Amazonas contributes \\$0.11bn (4.0%). The three-state concentration at 100.0% of meaningful export value reflects the geological reality: economic niobium deposits in Brazil are confined to two carbonatite complexes, Araxá and Catalão, with the Amazonas figure a distant third.

**China leads destination markets at \\$1.32bn (49.8%) but does not dominate in the way it does for iron ore, soybeans, or cotton**. The finding from Step 2 — that ferro-niobium does not appear in any state's top China export sectors — is explained here: at 49.8%, China's share is significant but not structurally dominant. The remaining 50.2% is distributed across a genuinely global customer base in a way that no other major Brazilian commodity export achieves. The Netherlands at \\$0.46bn (17.3%) is the second largest destination — consistent with Rotterdam as a trading and distribution hub for niobium bound for European steel mills rather than reflecting Dutch domestic consumption. The United States (\\$0.21bn, 8.1%), Singapore (\\$0.19bn, 7.3%), South Korea (\\$0.18bn, 6.6%), and Japan (\\$0.16bn, 5.8%) follow, completing a top-6 that spans Asia, Europe, and North America at meaningful individual shares.

**The cumulative concentration profile is distinctive**. The top 6 destinations account for 94.9% of exports, a relatively concentrated routing by number of partners, but spread across four distinct economic blocs (China, Europe via Netherlands, North America, and Northeast Asia). This reflects the end-use of ferro-niobium: it is a high-strength low-alloy steel additive used in automotive, construction, pipeline, and infrastructure steel manufacturing, industries concentrated in precisely these markets. A country without a significant steel industry does not appear in this ranking.

**Singapore at 7.3% warrants interpretation**. Singapore has no meaningful domestic steel industry at the scale implied by $0.19bn in ferro-niobium imports.

**The prior Step 2 expectation that ferro-niobium routes primarily to Europe, Japan, South Korea and the United States is partially confirmed but requires revision**. China at 49.8% is the single largest destination by a significant margin. A finding that contradicts the assumption that niobium's absence from state-level China top-partner lists indicated non-China routing. The correct interpretation is that niobium's China share (49.8%) is high in absolute terms but low relative to China's share of other commodities (soybeans 75%+, iron ore 63%+), which is why it does not surface in state-level China concentration rankings.

#### Business Relevance

Ferro-niobium's destination market structure is the most geographically resilient of any major Brazilian commodity export. A 49.8% China share with meaningful volumes to the US, Europe, Japan, South Korea and Singapore means that demand disruption in any single market, including China, does not threaten the full export volume in the way it would for soy or iron ore. For CBMM and CMOC as producers, this diversification is a strategic asset. For Brazil as a whole, the near-monopoly supply position combined with diversified demand geography creates a structurally different risk profile from the commodity sectors examined elsewhere in this analysis. The commercial implication for any participant in the specialty metals or steel industry supply chain is that Brazilian niobium pricing and availability is a global, not a bilateral, question and that Singapore's 7.3% share identifies commodity trading intermediation as a meaningful channel alongside direct OEM procurement relationships.


---
---


## 4.3 — Agricultural Commodities and Pharmaceuticals: Partner-Level Resolution

Three partner-level investigations where the destination market composition is unknown. Cotton and tobacco are established export commodities with high national concentration in specific states but unconfirmed routing beyond China's known low share. Goiás pharmaceutical imports are structurally anomalous for an agricultural state and require NCM-level product identification before the Anápolis industrial profile can be characterised.


### 4.3.1 — Cotton Destination Markets: Mato Grosso (SH52)

**Open flag:** Mato Grosso accounts for 62.0% of national cotton exports. Only 13.9% routes to China — well below China's national average share across all commodities. The remaining 86.1% of Mato Grosso cotton export destinations are unconfirmed.


**Output — Partner Ranking with Cumulative Share:** Identifies the primary non-China destinations for Mato Grosso cotton. Only 13.9% routes to China nationally — the remaining 86.1% is the open question. Likely destinations: Bangladesh, Pakistan, Vietnam and other textile-manufacturing economies in South and Southeast Asia.

In [77]:
## Cotton (SH52) — Mato Grosso partner-level breakdown
## SG_UF_NCM = 'MT' for Mato Grosso

query_cotton_partners = f"""
    WITH mt_cotton AS (
        SELECT
            e."CO_PAIS",
            SUM(e."VL_FOB")     AS vl_fob,
            SUM(e."KG_LIQUIDO") AS kg_liq
        FROM exp e
        WHERE e."CO_ANO"    = {MAX_YEAR}
          AND e."SG_UF_NCM" = 'MT'
          AND CAST(LEFT(e."CO_NCM"::TEXT, 2) AS INT) = 52
        GROUP BY e."CO_PAIS"
    )
    SELECT
        c."CO_PAIS",
        p.nome_pais                                          AS partner_country,
        c.vl_fob,
        ROUND(100.0 * c.vl_fob / SUM(c.vl_fob) OVER (), 2) AS pct_of_total
    FROM mt_cotton c
    LEFT JOIN pais p ON c."CO_PAIS" = p.codigo_pais
    ORDER BY c.vl_fob DESC
    LIMIT 20;
"""

df_cotton = pd.read_sql(query_cotton_partners, engine)
df_cotton['vl_fob_bn']      = df_cotton['vl_fob'] / 1e9
df_cotton['cumulative_pct'] = df_cotton['pct_of_total'].cumsum().round(1)

cotton_total     = df_cotton['vl_fob'].sum()
cotton_china_pct = df_cotton[df_cotton['CO_PAIS'] == CHINA_CODE]['pct_of_total'].sum()

#print(f'Mato Grosso cotton (SH52) — destination markets, {MAX_YEAR}')
#print(f'Total: ${cotton_total/1e9:.2f}bn | China share: {cotton_china_pct:.1f}%')

## Styled table: Mato Grosso cotton destination markets
DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

df_cotton_display = df_cotton[[
    'partner_country', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'partner_country': 'Destination Market',
    'vl_fob_bn'      : 'Exports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_cotton_display.insert(0, 'Rank', range(1, len(df_cotton_display) + 1))

df_cotton_display['Exports (USD bn)'] = df_cotton_display['Exports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_cotton_display['Share (%)'] = df_cotton_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_cotton_display['Cumulative (%)'] = df_cotton_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

## Flag China row
def highlight_china(row):
    if df_cotton.loc[row.name, 'CO_PAIS'] == CHINA_CODE:
        return [f'background-color: #fdf0ee; font-weight: 600;'] * len(row)
    return [''] * len(row)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

styler = (
    df_cotton_display.style
    .set_caption(
        f'Mato Grosso — Cotton (SH52) Destination Markets, {MAX_YEAR} '
        f'| Total: ${cotton_total/1e9:.2f}bn '
        f'| China share: {cotton_china_pct:.1f}% '
        f'| China row highlighted'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_china,  axis=1)
    .map(colour_share,       subset=['Share (%)'])
    .map(colour_cumulative,  subset=['Cumulative (%)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## Destination
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Exports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Rank,Destination Market,Exports (USD bn),Share (%),Cumulative (%)
1,Bangladesh,$0.539bn,17.2%,17.2%
2,Turquia,$0.505bn,16.1%,33.3%
3,Vietnã,$0.476bn,15.2%,48.5%
4,Paquistão,$0.451bn,14.4%,62.9%
5,China,$0.437bn,13.9%,76.8%
6,Índia,$0.273bn,8.7%,85.5%
7,Indonésia,$0.163bn,5.2%,90.7%
8,Egito,$0.110bn,3.5%,94.2%
9,Malásia,$0.068bn,2.2%,96.4%
10,Coreia do Sul,$0.058bn,1.8%,98.2%


#### Overview

Mato Grosso cotton routes almost exclusively to textile manufacturing economies in South and Southeast Asia, with no single country dominant and China playing a secondary role at 13.9% structurally the inverse of Brazil's soy and iron ore export pattern.

The top four destinations, Bangladesh (\\$0.54bn, 17.2%), Turkey (\\$0.51bn, 16.1%), Vietnam (\\$0.48bn, 15.2%), and Pakistan (\\$0.45bn, 14.4%), together account for 62.9% of the \\$3.13bn total. China at \\$0.44bn (13.9%) ranks fifth, behind all four. The top 6 destinations reach 85.5%, and all six are major global textile and apparel producers. The product maps directly onto the end-use industry concentrated in precisely these countries.

The four-country Southeast Asia cluster Vietnam (\\$0.48bn, 15.2%), Indonesia ($0.16bn, 5.2%), Malaysia (\\$0.07bn, 2.2%), Thailand (\\$0.02bn, 0.5%), collectively accounts for \\$0.72bn (23.1%).

**China's 13.9% share ranks fifth**. At \\$0.44bn, China remains a significant buyer in absolute terms but is not the structurally dominant destination it represents for soybeans, iron ore, or cotton at the national level. The 13.9% figure reflects a genuinely diversified buyer base rather than a bilateral dependency.

**The tail of the distribution is notably clean**. Ranks 8 through 20: Egypt, Malaysia, South Korea, Thailand, and smaller markets collectively contribute only 13.2% with no single entry exceeding 2.5%. The destination structure is transparent: buyers are identifiable textile-producing economies throughout.

#### Business Relevance

The destination market structure of Mato Grosso cotton is the clearest example in this dataset of a Brazilian commodity that has achieved genuine demand-side diversification without China dependency. Four countries each absorbing 14–17% of total exports produces a risk profile that is structurally more resilient than any other major commodity examined in this analysis. For participants in agricultural trade finance, commodity brokerage, or logistics serving the MT cotton export corridor, the six distinct major destination geographies demand disruption in any single market does not threaten the overall export volume in the way it would for a China-concentrated commodity.

---
---


### 4.3.2 — Tobacco Destination Markets: Rio Grande do Sul (NCM 24012030)

**Open flag:** Rio Grande do Sul accounts for 89.9% of national tobacco exports. Only 18.2% routes to China. The remaining 81.8% of RS tobacco export destinations are unconfirmed.


**Output — Partner Ranking with Cumulative Share:** Identifies the primary non-China destinations for RS tobacco (NCM 24012030). Only 18.2% routes to China — the remaining 81.8% is the open question. Expected primary markets: European cigarette manufacturers (Germany, Belgium, Netherlands) and the United States.

In [78]:
## Tobacco (NCM 24012030) — Rio Grande do Sul partner-level breakdown
## SG_UF_NCM = 'RS'

query_tobacco = f"""
    WITH rs_tobacco AS (
        SELECT
            e."CO_PAIS",
            SUM(e."VL_FOB")     AS vl_fob,
            SUM(e."KG_LIQUIDO") AS kg_liq
        FROM exp e
        WHERE e."CO_ANO"    = {MAX_YEAR}
          AND e."SG_UF_NCM" = 'RS'
          AND e."CO_NCM"    = 24012030
        GROUP BY e."CO_PAIS"
    )
    SELECT
        t."CO_PAIS",
        p.nome_pais                                          AS partner_country,
        t.vl_fob,
        ROUND(100.0 * t.vl_fob / SUM(t.vl_fob) OVER (), 2) AS pct_of_total
    FROM rs_tobacco t
    LEFT JOIN pais p ON t."CO_PAIS" = p.codigo_pais
    ORDER BY t.vl_fob DESC
    LIMIT 20;
"""

df_tobacco = pd.read_sql(query_tobacco, engine)
df_tobacco['vl_fob_bn']      = df_tobacco['vl_fob'] / 1e9
df_tobacco['cumulative_pct'] = df_tobacco['pct_of_total'].cumsum().round(1)

tobacco_total     = df_tobacco['vl_fob'].sum()
tobacco_china_pct = df_tobacco[df_tobacco['CO_PAIS'] == CHINA_CODE]['pct_of_total'].sum()
top3_pct          = df_tobacco['pct_of_total'].head(3).sum()

#print(f'Rio Grande do Sul tobacco (NCM 24012030) — destination markets, {MAX_YEAR}')
#print(f'Total: ${tobacco_total/1e9:.2f}bn | China share: {tobacco_china_pct:.1f}% | Top 3 combined: {top3_pct:.1f}%')

## Styled table: Rio Grande do Sul tobacco destination markets

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

df_tobacco_display = df_tobacco[[
    'partner_country', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'partner_country': 'Destination Market',
    'vl_fob_bn'      : 'Exports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_tobacco_display.insert(0, 'Rank', range(1, len(df_tobacco_display) + 1))

df_tobacco_display['Exports (USD bn)'] = df_tobacco_display['Exports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_tobacco_display['Share (%)'] = df_tobacco_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_tobacco_display['Cumulative (%)'] = df_tobacco_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

def highlight_china(row):
    if df_tobacco.loc[row.name, 'CO_PAIS'] == CHINA_CODE:
        return [f'background-color: #fdf0ee; font-weight: 600;'] * len(row)
    return [''] * len(row)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

styler = (
    df_tobacco_display.style
    .set_caption(
        f'Rio Grande do Sul — Tobacco (NCM 24012030) Destination Markets, {MAX_YEAR} '
        f'| Total: ${tobacco_total/1e9:.2f}bn '
        f'| China share: {tobacco_china_pct:.1f}% '
        f'| Top 3 combined: {top3_pct:.1f}% '
        f'| China row highlighted'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_china,  axis=1)
    .map(colour_share,       subset=['Share (%)'])
    .map(colour_cumulative,  subset=['Cumulative (%)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## Destination
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Exports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Rank,Destination Market,Exports (USD bn),Share (%),Cumulative (%)
1,Bélgica,$0.568bn,23.0%,23.0%
2,China,$0.552bn,22.4%,45.4%
3,Indonésia,$0.227bn,9.2%,54.6%
4,Vietnã,$0.142bn,5.8%,60.4%
5,Estados Unidos,$0.122bn,4.9%,65.3%
6,Turquia,$0.081bn,3.3%,68.6%
7,Polônia,$0.070bn,2.9%,71.4%
8,Emirados Árabes Unidos,$0.070bn,2.8%,74.3%
9,Suíça,$0.068bn,2.8%,77.0%
10,Paraguai,$0.063bn,2.6%,79.6%


#### Overview

Rio Grande do Sul tobacco (NCM 24012030) routes to a genuinely global set of destinations with no single country dominant, though Belgium and China together account for 45.4% of the \\$2.31bn total.

Belgium leads at \\$0.57bn (23.0%). Belgium is the registered location of several major multinational tobacco processing and trading companies, making it a procurement and re-export hub rather than a final consumption market at this scale. China follows at \\$0.55bn (22.4%) — the closest to a bilateral dependency in this ranking, though still below the level that would constitute structural concentration. Indonesia ranks third at \\$0.23bn (9.2%), completing a top three that accounts for 54.6% of total exports.

**The geographic spread below rank 3 is broad and commercially diverse**. Vietnam (\\$0.14bn, 5.8%), the United States (\\$0.12bn, 4.9%), Turkey (\\$0.08bn, 3.3%), Poland (\\$0.07bn, 2.9%), UAE (\\$0.07bn, 2.8%), and Switzerland (\\$0.07bn, 2.8%) each represent distinct regional markets across Southeast Asia, North America, Europe, and the Middle East. The top 20 destinations reach 93.7% cumulative share across a set of countries that spans every major world region, confirming that RS tobacco exports are not structurally dependent on any single trade corridor.

**Switzerland at 2.8% and Paraguay at 2.6% warrant note**. Switzerland is the headquarters location of major multinational tobacco companies including Philip Morris International and Japan Tobacco International. Its appearance at rank 9 is consistent with procurement routing through Swiss holding structures rather than Swiss domestic consumption. Paraguay at \\$0.06bn (2.6%) is a known re-export market given its position as a regional tobacco trading hub, particularly for the informal market across the Southern Cone.

China's 22.4% share sits above the cotton figure (13.9%) but below the commodity averages for soybeans and iron ore. The destination structure across the top 20 reaches 93.7% cumulative share and no single entry in ranks 6 through 20 exceeds 3.3%.


#### Business Relevance

The tobacco destination market structure is more concentrated than cotton but more diversified than soybeans or iron ore. Belgium's 23.0% share, if driven by multinational tobacco company procurement hubs as the data suggests, means that a significant portion of the $2.31bn flows through a small number of large corporate buyers with centralised purchasing functions rather than through a distributed set of sovereign market relationships. For trade finance and logistics participants serving the RS tobacco export corridor, this implies a counterparty base that is narrower than the destination country count suggests the number of distinct purchasing entities is likely materially smaller than the number of destination markets. The Switzerland and UAE entries reinforce this: both are consistent with multinational company procurement routing that consolidates purchasing across multiple end markets through a single registered entity.


---
---


### 4.3.3 — Goiás Pharmaceutical Imports: API Manufacturing Input or Distribution Entry Point?

**Open flag:** \\$1.94bn in pharmaceutical imports (SH30) account for 36.3% of Goiás state imports — the top import sector for an otherwise agricultural commodity state. This is structurally anomalous and inconsistent with Goiás's established export profile. Anápolis was confirmed in Step 3 as the primary import registration municipality (40.3% of state imports).

**⚠️ Note:** Anápolis has an established pharmaceutical industrial district (Daia — Distrito Agroindustrial de Anápolis) that produces medicines for domestic and export markets. If imports are API-driven, this is consistent with that profile.


**Output — SH30 NCM Breakdown:** Identifies whether Goiás pharmaceutical imports are driven by active pharmaceutical ingredients (API, SH2902–SH2941 range) or finished medicines (SH3003/SH3004). API dominance confirms Anápolis's Daia industrial district as a manufacturing input cluster. Finished medicine dominance indicates a national distribution entry point.

In [79]:
## Goiás SH30 (pharmaceuticals) — NCM-level breakdown
## SG_UF_NCM = 'GO'

query_goias_pharma = f"""
    WITH go_pharma AS (
        SELECT
            i."CO_NCM",
            SUM(i."VL_FOB")     AS vl_fob,
            SUM(i."KG_LIQUIDO") AS kg_liq
        FROM imp i
        WHERE i."CO_ANO"    = {MAX_YEAR}
          AND i."SG_UF_NCM" = 'GO'
          AND CAST(LEFT(i."CO_NCM"::TEXT, 2) AS INT) = 30
        GROUP BY i."CO_NCM"
    )
    SELECT
        p."CO_NCM",
        n.nome_ncm                                           AS ncm_description,
        p.vl_fob,
        p.kg_liq,
        ROUND(100.0 * p.vl_fob / SUM(p.vl_fob) OVER (), 2) AS pct_of_total,
        CASE
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 2) AS INT) IN (29) THEN 'Active Pharma Ingredient'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) IN (3003, 3004) THEN 'Finished Medicine'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 3002 THEN 'Biological / Vaccine'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 3006 THEN 'Pharma Preparations'
            ELSE 'Other SH30'
        END                                                  AS pharma_type
    FROM go_pharma p
    LEFT JOIN ncm n ON p."CO_NCM" = n.codigo_ncm
    ORDER BY p.vl_fob DESC
    LIMIT 20;
"""

df_goias_pharma = pd.read_sql(query_goias_pharma, engine)
df_goias_pharma['vl_fob_mn']      = df_goias_pharma['vl_fob'] / 1e6
df_goias_pharma['cumulative_pct'] = df_goias_pharma['pct_of_total'].cumsum().round(1)

pharma_total = df_goias_pharma['vl_fob'].sum()

## Summary by pharma type
pharma_summary = (
    df_goias_pharma.groupby('pharma_type')['vl_fob']
    .sum()
    .sort_values(ascending=False)
)

## API vs finished split — key hypothesis test
api_total      = pharma_summary.get('Active Pharma Ingredient', 0)
finished_total = pharma_summary.get('Finished Medicine', 0)

#print(f'Goiás pharmaceutical imports (SH30) — NCM breakdown, {MAX_YEAR}')
#print(f'Total: ${pharma_total/1e6:.1f}m')
#print(f'API: ${api_total/1e6:.1f}m ({100*api_total/pharma_total:.1f}%) | '
#      f'Finished medicines: ${finished_total/1e6:.1f}m ({100*finished_total/pharma_total:.1f}%)')
#print(f'Hypothesis: {"Manufacturing cluster (API-driven)" if api_total > finished_total else "Distribution hub (finished-driven)"}')

## Styled tables: Goiás pharmaceutical imports — summary + NCM detail
DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

PHARMA_COLOURS = {
    'Active Pharma Ingredient': '#1565c0',
    'Finished Medicine'       : GREEN,
    'Biological / Vaccine'    : '#6a1b9a',
    'Pharma Preparations'     : AMBER,
    'Other SH30'              : GREY_TEXT,
}

## ── Table 1: pharma type summary ──────────────────────────────────────────────
df_pharma_summary_display = pharma_summary.reset_index().rename(columns={
    'pharma_type': 'Type',
    'vl_fob'     : 'raw',
})
df_pharma_summary_display['Imports (USD m)'] = df_pharma_summary_display['raw'].apply(
    lambda x: f'${x/1e6:.1f}m'
)
df_pharma_summary_display['Share (%)'] = df_pharma_summary_display['raw'].apply(
    lambda x: f'{100*x/pharma_total:.1f}%'
)

## Hypothesis badge per type
def get_hypothesis(type_val):
    if type_val == 'Active Pharma Ingredient':
        return 'Manufacturing'
    elif type_val == 'Finished Medicine':
        return 'Distribution'
    elif type_val == 'Biological / Vaccine':
        return 'Healthcare'
    return '—'

df_pharma_summary_display['Implication'] = df_pharma_summary_display['Type'].apply(
    get_hypothesis
)
df_pharma_summary_display = df_pharma_summary_display[[
    'Type', 'Imports (USD m)', 'Share (%)', 'Implication'
]]

def colour_pharma_type(val):
    c = PHARMA_COLOURS.get(val, GREY_TEXT)
    return f'color: {c}; font-weight: 700;'

def colour_implication(val):
    if val == 'Manufacturing':
        return f'color: #1565c0; font-weight: 700;'
    elif val == 'Distribution':
        return f'color: {GREEN}; font-weight: 700;'
    elif val == 'Healthcare':
        return f'color: #6a1b9a; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_share_summary(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 50.0:
        return f'color: {RED}; font-weight: 700;'
    elif v >= 20.0:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

## Overall hypothesis for caption
overall_hypothesis = (
    'Manufacturing cluster (API-driven)'
    if api_total > finished_total
    else 'Distribution hub (finished medicine-driven)'
)

display(
    df_pharma_summary_display.style
    .set_caption(
        f'Goiás — SH30 Pharmaceutical Import Composition, {MAX_YEAR} '
        f'| Total: ${pharma_total/1e6:.1f}m '
        f'| Hypothesis: {overall_hypothesis}'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_pharma_type,   subset=['Type'])
    .map(colour_share_summary, subset=['Share (%)'])
    .map(colour_implication,   subset=['Implication'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [
            ('text-align', 'center'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
            ('margin-bottom', '28px'),
        ]},
    ])
    .hide(axis='index')
)

## ── Table 2: full NCM detail ──────────────────────────────────────────────────
df_goias_ncm_display = df_goias_pharma[[
    'CO_NCM', 'ncm_description', 'pharma_type', 'vl_fob_mn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'CO_NCM'         : 'NCM',
    'ncm_description': 'Product Description',
    'pharma_type'    : 'Type',
    'vl_fob_mn'      : 'Imports (USD m)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_goias_ncm_display.insert(0, 'Rank', range(1, len(df_goias_ncm_display) + 1))

df_goias_ncm_display['Imports (USD m)'] = df_goias_ncm_display['Imports (USD m)'].apply(
    lambda x: f'${x:.1f}m'
)
df_goias_ncm_display['Share (%)'] = df_goias_ncm_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_goias_ncm_display['Cumulative (%)'] = df_goias_ncm_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

def highlight_type_rows(row):
    tints = {
        'Active Pharma Ingredient': '#e8f0fb',
        'Finished Medicine'       : '#eaf5ee',
        'Biological / Vaccine'    : '#f3e5f5',
        'Pharma Preparations'     : '#fff8e6',
        'Other SH30'              : '',
    }
    bg = tints.get(row['Type'], '')
    return [f'background-color: {bg};' if bg else ''] * len(row)

display(
    df_goias_ncm_display.style
    .set_caption(
        f'Goiás — SH30 Pharmaceutical NCM Detail, {MAX_YEAR} '
        f'| Source: GO state-level imp table '
        f'| Total: ${pharma_total/1e6:.1f}m '
        f'| Top 20 NCM codes by value'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_type_rows,  axis=1)
    .map(colour_share,           subset=['Share (%)'])
    .map(colour_cumulative,      subset=['Cumulative (%)'])
    .map(colour_pharma_type,     subset=['Type'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## NCM code
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Product description
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Type
            ('text-align', 'center'),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

Type,Imports (USD m),Share (%),Implication
Biological / Vaccine,$1311.7m,70.4%,Healthcare
Finished Medicine,$522.2m,28.0%,Distribution
Pharma Preparations,$29.9m,1.6%,—


Rank,NCM,Product Description,Type,Imports (USD m),Share (%),Cumulative (%)
1,30021590,"Outros produtos imunológicos, apresentados em doses ou acondicionados para venda a retalho",Biological / Vaccine,$1231.8m,63.4%,63.4%
2,30049069,"Outros medicamentos contendo compostos heterocíclicos heteroátomos nitrogenados, em doses",Finished Medicine,$159.9m,8.2%,71.6%
3,30049079,"Outros medicamentos com compostos heterocíclicos, etc, em doses",Finished Medicine,$102.7m,5.3%,76.9%
4,30021520,Basiliximab (DCI); bevacizumab (DCI); daclizumab (DCI); etanercept (DCI); gemtuzumab ozogamicin (DCI); oprelvekin (DCI); rituximab (DCI); trastuzumab (DCI),Biological / Vaccine,$43.3m,2.2%,79.1%
5,30049049,"Outros medicamentos com compostos de função carboxiamida, etc, em doses",Finished Medicine,$40.1m,2.1%,81.2%
6,30049039,"Outros medicam.c/compostos de funcao amina,etc.em doses",Finished Medicine,$34.1m,1.8%,82.9%
7,30024129,"Outras vacinas para medicina humana, em doses",Biological / Vaccine,$29.9m,1.5%,84.5%
8,30066000,"Prepars.quims.contraceptivas, de hormônios/espermicidas",Pharma Preparations,$29.9m,1.5%,86.0%
9,30049059,"Outros medicamentos contendo produtos das posições 2930 a 2932, etc, em doses",Finished Medicine,$28.9m,1.5%,87.5%
10,30049035,"Medicamento contendo levodopa ou alfa-metildopa, em doses",Finished Medicine,$27.0m,1.4%,88.9%


#### Overview

The pharmaceutical import profile of Goiás is dominated by a single NCM code that reframes the entire section. NCM 30021590 (other immunological products presented in doses or packaged for retail) accounts for \\$1,231.8m, or 63.4% of the \\$1,863.8m total. This is not an API manufacturing input. It is a finished biological product, most likely monoclonal antibodies, immunotherapies, or other high-value biological medicines procured for distribution into the Brazilian healthcare system.

The hypothesis generated by the query classification ("Distribution hub, finished medicine-driven") requires revision in light of the NCM detail. The Biological / Vaccine category at 70.4% (\\$1,311.7m) is driven overwhelmingly by NCM 30021590, which covers immunological products rather than conventional vaccines. The secondary biological entry, NCM 30021520 (\\$43.3m, 2.2%), names specific monoclonal antibodies including bevacizumab, rituximab, trastuzumab, and etanercept, which are oncology and autoimmune biologics, not vaccines. NCM 30024129 ($29.9m, 1.5%) covers human medicine vaccines. The biological category is therefore dominated by high-value therapeutic biologics, not immunisation products, and the distribution implication is to hospital and specialist healthcare settings rather than to primary care or retail pharmacy.

**The finished medicine component (\\$522.2m, 28.0%) is concentrated in complex pharmaceutical classes**. The leading finished medicine codes include heterocyclic compounds (NCM 30049069, \\$159.9m and 30049079, \\$102.7m combined at 13.5%), levodopa and alpha-methyldopa (30049035, \\$27.0m), amoxicillin (30041012, \\$24.8m), amphotericin B in liposomes (30042095, \\$21.9m), and cephalosporins (30042059, \\$13.7m). This is a specialist, hospital-oriented product mix rather than a general consumer medicine portfolio.

**The Anápolis / Daia pharmaceutical district hypothesis is not supported by the NCM composition**. API manufacturing inputs, including chemical precursors and bulk active ingredients imported for tableting or capsule filling, do not appear in the top 20. The imports are finished doses and packaged biological products, which are end-point procurement items for the healthcare system, not manufacturing inputs. The $1,863.8m figure reflects Goiás as a registered procurement point for the Brazilian public and private healthcare supply chain, consistent with the presence of large pharmaceutical distributors and potentially the central procurement function of state or federal health authorities registered in Goiás.

**The scale of NCM 30021590 at \\$1,231.8m requires contextualisation**. A single immunological product category accounting for 63.4% of a state's pharmaceutical imports at this absolute value is consistent with centralised procurement of a high-cost biological therapy rather than a broad distribution function. At \\$1,231.8m for immunological products in one state, the volume implies either national-level procurement routing through a Goiás-registered entity, or the presence of a major distributor with exclusive or dominant market position in this category.

#### Business Relevance

The Goiás pharmaceutical import profile identifies the state as a registered procurement point for high-value biological medicines and specialist finished pharmaceuticals rather than a manufacturing input cluster. The practical commercial implication differs materially from the Anápolis industrial district characterisation carried in prior steps: the counterparties registered in Goiás importing \\$1.86bn in pharmaceuticals are distributors, procurement entities, or healthcare system intermediaries handling finished therapeutic products, not manufacturers importing API. For participants in pharmaceutical logistics, cold chain infrastructure, or healthcare procurement, Goiás represents a significant registered demand node for biologics and specialist medicines. The \\$1,231.8m in immunological products alone warrants entity-level investigation to identify the specific importers, as the concentration of this value in a single NCM code suggests a small number of large buyers rather than a distributed pharmaceutical retail or distribution network.

---
---


## 4.4 — State Product Concentration: Piauí, Rio de Janeiro and Rio Grande do Norte

Three state-level investigations where the product identity behind a structural indicator is unconfirmed. Piauí and Rio de Janeiro carry the two highest product HHI values in the dataset — the dominant NCM codes driving each state's concentration have not been identified. Rio Grande do Norte's Panama top-partner anomaly tests whether Panama is acting as a transit hub and, if so, for what product.

### 4.4.1 — Piauí and Rio de Janeiro: Dominant NCM Codes (Product HHI Follow-Up)

**Open flag:** Step 3 (Finding 23) identified Piauí (product HHI 0.688) and Rio de Janeiro (product HHI 0.631) as the two most product-concentrated states in the dataset. Both carry significant single-sector export dependency that is not visible from geographic HHI alone. The dominant NCM codes driving each state's HHI have not been identified.


**Output — Top 10 NCM Export Codes per State with HHI Contribution:** Run for both Piauí (HHI 0.688) and Rio de Janeiro (HHI 0.631). The partial HHI from the top-10 codes reveals how much of total concentration is explained by a small number of products. Expected: RJ dominated by crude oil (SH2709); Piauí by a single agricultural or extractive commodity.

In [80]:
## Piauí (PI) and Rio de Janeiro (RJ): top NCM export codes to identify HHI drivers
## Finding 18 (Step 3): PI HHI = 0.688, RJ HHI = 0.631 — highest two states

for sg_uf, state_name in [('PI', 'Piauí'), ('RJ', 'Rio de Janeiro')]:

    query_hhi_ncm = f"""
        WITH state_exp AS (
            SELECT
                e."CO_NCM",
                SUM(e."VL_FOB") AS vl_fob
            FROM exp e
            WHERE e."CO_ANO"    = {MAX_YEAR}
              AND e."SG_UF_NCM" = '{sg_uf}'
            GROUP BY e."CO_NCM"
        ),
        total AS (
            SELECT SUM(vl_fob) AS total_fob FROM state_exp
        )
        SELECT
            s."CO_NCM",
            n.nome_ncm                                              AS ncm_description,
            s.vl_fob,
            ROUND(100.0 * s.vl_fob / t.total_fob, 2)              AS pct_of_total,
            ROUND(
                POWER(s.vl_fob::NUMERIC / t.total_fob, 2), 6
            )                                                       AS hhi_contribution
        FROM state_exp s
        CROSS JOIN total t
        LEFT JOIN ncm n ON s."CO_NCM" = n.codigo_ncm
        ORDER BY s.vl_fob DESC
        LIMIT 10;
    """

    df_hhi = pd.read_sql(query_hhi_ncm, engine)
    df_hhi['vl_fob_bn']      = df_hhi['vl_fob'] / 1e9
    df_hhi['cumulative_pct'] = df_hhi['pct_of_total'].cumsum()
    implied_hhi              = df_hhi['hhi_contribution'].sum()

## Piauí (PI) and Rio de Janeiro (RJ): top NCM export codes to identify HHI drivers
## Finding 18 (Step 3): PI HHI = 0.688, RJ HHI = 0.631 — highest two states

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

## Known full HHI values from Step 3 for caption reference
KNOWN_HHI = {'PI': 0.688, 'RJ': 0.631}

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

TABLE_STYLES = [
    {'selector': 'caption', 'props': [
        ('caption-side', 'top'),
        ('font-size', '14px'),
        ('font-weight', '700'),
        ('font-family', 'Helvetica Neue, Arial, sans-serif'),
        ('color', DARK_BLUE),
        ('padding', '0 0 10px 0'),
        ('text-align', 'left'),
    ]},
    {'selector': 'thead th', 'props': [
        ('background-color', DARK_BLUE),
        ('color', WHITE),
        ('font-size', '12px'),
        ('font-weight', '600'),
        ('text-transform', 'uppercase'),
        ('letter-spacing', '0.04em'),
        ('padding', '10px 14px'),
        ('border', 'none'),
    ]},
    {'selector': 'thead tr:last-child th', 'props': [
        ('border-bottom', f'2px solid {MID_BLUE}'),
    ]},
    {'selector': 'tbody tr:nth-child(odd)', 'props': [
        ('background-color', WHITE),
    ]},
    {'selector': 'tbody tr:nth-child(even)', 'props': [
        ('background-color', STRIPE),
    ]},
    {'selector': 'tbody tr:hover', 'props': [
        ('background-color', LIGHT_BLUE),
    ]},
    {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
        ('text-align', 'center'),
        ('font-weight', '600'),
        ('color', GREY_TEXT),
        ('font-size', '12px'),
    ]},
    {'selector': 'tbody td:nth-child(2)', 'props': [   ## NCM code
        ('font-family', 'monospace'),
        ('font-size', '12px'),
        ('color', GREY_TEXT),
        ('text-align', 'center'),
    ]},
    {'selector': 'tbody td:nth-child(3)', 'props': [   ## Description
        ('font-weight', '500'),
        ('color', DARK_BLUE),
    ]},
    {'selector': 'tbody td:nth-child(4)', 'props': [   ## Exports
        ('text-align', 'right'),
        ('font-variant-numeric', 'tabular-nums'),
    ]},
    {'selector': 'tbody td:nth-child(5)', 'props': [   ## Share
        ('text-align', 'right'),
        ('font-variant-numeric', 'tabular-nums'),
    ]},
    {'selector': 'tbody td:nth-child(6)', 'props': [   ## Cumulative
        ('text-align', 'right'),
        ('font-variant-numeric', 'tabular-nums'),
        ('border-left', f'1px solid {LIGHT_BLUE}'),
    ]},
    {'selector': 'tbody td:nth-child(7)', 'props': [   ## HHI contribution
        ('text-align', 'right'),
        ('font-variant-numeric', 'tabular-nums'),
        ('border-left', f'1px solid {LIGHT_BLUE}'),
        ('color', GREY_TEXT),
        ('font-size', '12px'),
    ]},
    {'selector': 'table', 'props': [
        ('border-collapse', 'collapse'),
        ('width', '100%'),
        ('border', f'1px solid {LIGHT_BLUE}'),
        ('border-radius', '6px'),
        ('overflow', 'hidden'),
        ('margin-bottom', '28px'),
    ]},
]

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

def colour_hhi_contrib(val):
    try:
        v = float(val)
    except (ValueError, AttributeError):
        return ''
    if v >= 0.10:
        return f'color: {RED}; font-weight: 700;'
    elif v >= 0.01:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

for sg_uf, state_name in [('PI', 'Piauí'), ('RJ', 'Rio de Janeiro')]:

    query_hhi_ncm = f"""
        WITH state_exp AS (
            SELECT
                e."CO_NCM",
                SUM(e."VL_FOB") AS vl_fob
            FROM exp e
            WHERE e."CO_ANO"    = {MAX_YEAR}
              AND e."SG_UF_NCM" = '{sg_uf}'
            GROUP BY e."CO_NCM"
        ),
        total AS (
            SELECT SUM(vl_fob) AS total_fob FROM state_exp
        )
        SELECT
            s."CO_NCM",
            n.nome_ncm                                          AS ncm_description,
            s.vl_fob,
            ROUND(100.0 * s.vl_fob / t.total_fob, 2)          AS pct_of_total,
            ROUND(
                POWER(s.vl_fob::NUMERIC / t.total_fob, 2), 6
            )                                                   AS hhi_contribution
        FROM state_exp s
        CROSS JOIN total t
        LEFT JOIN ncm n ON s."CO_NCM" = n.codigo_ncm
        ORDER BY s.vl_fob DESC
        LIMIT 10;
    """

    df_hhi = pd.read_sql(query_hhi_ncm, engine)
    df_hhi['vl_fob_bn']      = df_hhi['vl_fob'] / 1e9
    df_hhi['cumulative_pct'] = df_hhi['pct_of_total'].cumsum().round(1)
    implied_hhi              = df_hhi['hhi_contribution'].sum()
    full_hhi                 = KNOWN_HHI.get(sg_uf, None)
    explained_pct            = (implied_hhi / full_hhi * 100) if full_hhi else None

    df_display = df_hhi[[
        'CO_NCM', 'ncm_description', 'vl_fob_bn', 'pct_of_total',
        'cumulative_pct', 'hhi_contribution'
    ]].rename(columns={
        'CO_NCM'          : 'NCM',
        'ncm_description' : 'Product Description',
        'vl_fob_bn'       : 'Exports (USD bn)',
        'pct_of_total'    : 'Share (%)',
        'cumulative_pct'  : 'Cumulative (%)',
        'hhi_contribution': 'HHI Contribution',
    }).copy()

    df_display.insert(0, 'Rank', range(1, len(df_display) + 1))

    df_display['Exports (USD bn)']  = df_display['Exports (USD bn)'].apply(
        lambda x: f'${x:.3f}bn'
    )
    df_display['Share (%)']         = df_display['Share (%)'].apply(
        lambda x: f'{x:.1f}%'
    )
    df_display['Cumulative (%)']    = df_display['Cumulative (%)'].apply(
        lambda x: f'{x:.1f}%'
    )
    df_display['HHI Contribution']  = df_display['HHI Contribution'].apply(
        lambda x: f'{x:.6f}'
    )

    explained_str = (
        f'| Top-10 explains {explained_pct:.1f}% of full HHI ({full_hhi})'
        if explained_pct else ''
    )

    display(
        df_display.style
        .set_caption(
            f'{state_name} ({sg_uf}) — Top 10 NCM Export Codes, {MAX_YEAR} '
            f'| Full product HHI: {full_hhi} '
            f'| Partial HHI (top-10): {implied_hhi:.4f} '
            f'{explained_str}'
        )
        .set_properties(**{
            'font-family'  : 'Helvetica Neue, Arial, sans-serif',
            'font-size'    : '13px',
            'text-align'   : 'left',
            'padding'      : '8px 14px',
            'border-bottom': f'1px solid {LIGHT_BLUE}',
        })
        .map(colour_share,       subset=['Share (%)'])
        .map(colour_cumulative,  subset=['Cumulative (%)'])
        .map(colour_hhi_contrib, subset=['HHI Contribution'])
        .set_table_styles(TABLE_STYLES)
        .hide(axis='index')
    )

Rank,NCM,Product Description,Exports (USD bn),Share (%),Cumulative (%),HHI Contribution
1,12019000,"Soja, mesmo triturada, exceto para semeadura",$0.993bn,82.7%,82.7%,0.683304
2,52010020,"Algodão não cardado nem penteado, simplesmente debulhado",$0.043bn,3.6%,86.3%,0.001303
3,15211000,Ceras vegetais,$0.036bn,3.0%,89.3%,0.000915
4,10059010,"Milho em grão, exceto para semeadura",$0.036bn,3.0%,92.3%,0.000887
5,26011100,"Minérios de ferro e seus concentrados, exceto as piritas de ferro ustuladas (cinzas de piritas), não aglomerados",$0.030bn,2.5%,94.8%,0.000615
6,4090000,Mel natural,$0.022bn,1.8%,96.6%,0.000326
7,23040090,"Bagaços e outros resíduos sólidos, da extração do óleo de soja",$0.021bn,1.8%,98.3%,0.000307
8,15200010,Glicerol em bruto,$0.004bn,0.4%,98.7%,0.000014
9,29397931,"Pilocarpina, seu nitrato e seu cloridrato",$0.004bn,0.3%,99.0%,0.000008
10,41053000,"Peles curtidas ou crust de ovinos, depiladas, mesmo divididas, mas não preparadas de outro modo, no estado seco (crust)",$0.002bn,0.2%,99.2%,0.000004


Rank,NCM,Product Description,Exports (USD bn),Share (%),Cumulative (%),HHI Contribution
1,27090010,Óleos brutos de petróleo,$38.815bn,79.2%,79.2%,0.628131
2,72071200,"Outros produtos semimanufaturados de ferro ou aço não ligado, de seção transversal retangular, que contenham, em peso, menos de 0,25 % de carbono",$1.574bn,3.2%,82.5%,0.001033
3,26011100,"Minérios de ferro e seus concentrados, exceto as piritas de ferro ustuladas (cinzas de piritas), não aglomerados",$1.541bn,3.1%,85.6%,0.000991
4,27101922,Fuel oil,$1.204bn,2.5%,88.1%,0.000604
5,84119100,Partes de turborreatores ou de turbopropulsores,$0.422bn,0.9%,88.9%,0.000074
6,84818099,"Torneiras, e dispositivos semelhantes, para canalizações",$0.397bn,0.8%,89.7%,0.000066
7,27101911,Querosenes de aviação,$0.349bn,0.7%,90.4%,0.000051
8,87032310,"Automóveis com motor explosão, 1500 < cm3 <= 3000, até 6 passageiros",$0.280bn,0.6%,91.0%,0.000033
9,87032100,"Automóveis com motor explosão, de cilindrada não superior a 1.000 cm3",$0.243bn,0.5%,91.5%,0.000025
10,27101259,"Outras gasolinas, exceto para aviação",$0.242bn,0.5%,92.0%,0.000025


#### Overview

Both states confirm the single-product concentration pattern their HHI values predicted, through entirely different structural mechanisms.

**Piauí: soybean concentration at 82.7%**
NCM 12019000 (soybeans, not for sowing) accounts for $0.993bn and 82.7% of Piauí's registered exports. Its HHI contribution of 0.6833 out of a full state HHI of 0.688 means soybeans alone explain 99.3% of the state's product concentration. The remaining nine codes cotton (3.6%), vegetable waxes (3.0%), corn (3.0%), iron ore (2.5%), honey (1.8%), soy residues (1.8%), and three minor items together contribute 0.005 to the HHI. The top 10 explains 100.0% of the full HHI.

**Rio de Janeiro: petroleum registration at 79.2%**
NCM 27090010 (crude petroleum oils) accounts for $38.815bn and 79.2% of Rio de Janeiro's registered exports, with an HHI contribution of 0.6281 out of a full state HHI of 0.631. As established in Section 4.1.3, this reflects registration of offshore petroleum exports through Rio de Janeiro legal addresses. Semi-manufactured steel (3.2%), iron ore (3.1%), and fuel oil (2.5%) follow as the next largest entries. The top 10 reaches 92.0% cumulative share and explains 100.0% of the full HHI.

**The two states share a high HHI for entirely different reasons**. Piauí's concentration reflects genuine single-commodity agricultural dependency. Rio de Janeiro's reflects administrative registration concentration of petroleum export contracts. The HHI metric correctly identifies both as highly concentrated, but the underlying drivers are structurally distinct.

---
---


### 4.4.2 — Rio Grande do Norte / Panama

**Open flag:** Panama is Rio Grande do Norte's top export partner — anomalous because Panama is a known transit and re-export hub rather than a final demand market. The product composition of RN exports to Panama is unconfirmed. If Panama is acting as transit, identifying the product will allow the likely end market to be inferred.

**⚠️ Note:** Panama does not have significant domestic demand for most Brazilian commodity exports. Re-export routing through the Panama Canal or Colón Free Trade Zone is the standard commercial explanation for Panama appearing as a top partner in Latin American trade data.


**Step 1 — Confirm Panama Country Code:** Verifies the `CO_PAIS` value for Panama from the `pais` table before filtering. Panama does not have significant domestic demand for most Brazilian commodity exports — re-export routing through the Panama Canal or Colón Free Trade Zone is the standard commercial explanation.

In [81]:
with engine.connect() as conn:
    df_panama_code = pd.read_sql(
        text("SELECT codigo_pais, nome_pais FROM pais WHERE nome_pais ILIKE '%panam%';"),
        conn
    )

print('Panama entries in pais table:')
print(df_panama_code)

PANAMA_CODE = int(df_panama_code['codigo_pais'].iloc[0])

Panama entries in pais table:
   codigo_pais                nome_pais
0          580                   Panamá
1          895  Zona do Canal do Panamá


**Table 1 — RN Full Partner Ranking:** Confirms Panama's position in Rio Grande do Norte's export partner list and its share of total state exports. Provides the broader partner context before drilling into the Panama-specific flow.

In [82]:
## RN full partner ranking
## SG_UF_NCM = 'RN'

query_rn_partners = f"""
    WITH rn_exp AS (
        SELECT
            e."CO_PAIS",
            SUM(e."VL_FOB") AS vl_fob
        FROM exp e
        WHERE e."CO_ANO"    = {MAX_YEAR}
          AND e."SG_UF_NCM" = 'RN'
        GROUP BY e."CO_PAIS"
    )
    SELECT
        r."CO_PAIS",
        p.nome_pais                                          AS partner_country,
        r.vl_fob,
        ROUND(100.0 * r.vl_fob / SUM(r.vl_fob) OVER (), 2) AS pct_of_total
    FROM rn_exp r
    LEFT JOIN pais p ON r."CO_PAIS" = p.codigo_pais
    ORDER BY r.vl_fob DESC
    LIMIT 15;
"""

df_rn_partners = pd.read_sql(query_rn_partners, engine)
df_rn_partners['vl_fob_mn']      = df_rn_partners['vl_fob'] / 1e6
df_rn_partners['cumulative_pct'] = df_rn_partners['pct_of_total'].cumsum().round(1)

rn_total      = df_rn_partners['vl_fob'].sum()
rn_china_pct  = df_rn_partners[df_rn_partners['CO_PAIS'] == CHINA_CODE]['pct_of_total'].sum()
top1_partner  = df_rn_partners.iloc[0]['partner_country']
top1_pct      = df_rn_partners.iloc[0]['pct_of_total']

## Panama flag — check if Panama is in top 5
panama_rows   = df_rn_partners[
    df_rn_partners['partner_country'].str.contains('panam', case=False, na=False)
]
panama_rank   = panama_rows.index[0] + 1 if len(panama_rows) > 0 else None
panama_pct    = panama_rows['pct_of_total'].iloc[0] if len(panama_rows) > 0 else 0

#print(f'Rio Grande do Norte — export partner ranking, {MAX_YEAR}')
#print(f'Total: ${rn_total/1e6:.1f}m | Top partner: {top1_partner} ({top1_pct:.1f}%)')
#if panama_rank:
#    print(f'Panama: rank {panama_rank} ({panama_pct:.1f}%) — transit hub investigation triggered')

## Styled table: Rio Grande do Norte export partner ranking

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'
PANAMA_TINT = '#fff3e0'   ## amber tint for Panama row
CHINA_TINT  = '#fdf0ee'   ## red tint for China row

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

df_rn_display = df_rn_partners[[
    'partner_country', 'vl_fob_mn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'partner_country': 'Destination Market',
    'vl_fob_mn'      : 'Exports (USD m)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_rn_display.insert(0, 'Rank', range(1, len(df_rn_display) + 1))

df_rn_display['Exports (USD m)'] = df_rn_display['Exports (USD m)'].apply(
    lambda x: f'${x:.1f}m'
)
df_rn_display['Share (%)'] = df_rn_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_rn_display['Cumulative (%)'] = df_rn_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

## Flag rows: Panama = amber tint, China = red tint
def highlight_special_rows(row):
    co_pais = df_rn_partners.loc[row.name, 'CO_PAIS']
    partner = str(df_rn_partners.loc[row.name, 'partner_country']).lower()
    if 'panam' in partner:
        return [f'background-color: {PANAMA_TINT}; font-weight: 600;'] * len(row)
    if co_pais == CHINA_CODE:
        return [f'background-color: {CHINA_TINT}; font-weight: 600;'] * len(row)
    return [''] * len(row)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

panama_note = (
    f'| Panama rank {panama_rank} ({panama_pct:.1f}%) — transit hub flag'
    if panama_rank else '| Panama not in top 15'
)

styler = (
    df_rn_display.style
    .set_caption(
        f'Rio Grande do Norte — Export Partner Ranking, {MAX_YEAR} '
        f'| Total: ${rn_total/1e6:.1f}m '
        f'| China: {rn_china_pct:.1f}% '
        f'{panama_note} '
        f'| Panama = amber | China = red'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_special_rows, axis=1)
    .map(colour_share,             subset=['Share (%)'])
    .map(colour_cumulative,        subset=['Cumulative (%)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## Destination
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Exports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Rank,Destination Market,Exports (USD m),Share (%),Cumulative (%)
1,Panamá,$488.0m,43.3%,43.3%
2,Países Baixos (Holanda),$147.2m,13.1%,56.3%
3,Canadá,$100.2m,8.9%,65.2%
4,Estados Unidos,$91.3m,8.1%,73.3%
5,Reino Unido,$67.8m,6.0%,79.3%
6,Espanha,$62.9m,5.6%,84.9%
7,Porto Rico,$21.7m,1.9%,86.8%
8,Portugal,$17.4m,1.5%,88.4%
9,Nigéria,$14.1m,1.2%,89.6%
10,Togo,$13.5m,1.2%,90.8%


**Table 2 — RN Exports to Panama, NCM Detail:** Identifies the specific product(s) routing through Panama. If the dominant product is a commodity consistent with RN's known export base (petroleum, salt, cashews, ferroalloys), Panama is acting as a transit point. The SH2 description column provides the sector-level context.

In [83]:
## RN exports to Panama — NCM-level product decomposition

query_rn_panama_ncm = f"""
    WITH rn_panama AS (
        SELECT
            e."CO_NCM",
            SUM(e."VL_FOB")     AS vl_fob,
            SUM(e."KG_LIQUIDO") AS kg_liq
        FROM exp e
        WHERE e."CO_ANO"    = {MAX_YEAR}
          AND e."SG_UF_NCM" = 'RN'
          AND e."CO_PAIS"   = {PANAMA_CODE}
        GROUP BY e."CO_NCM"
    )
    SELECT
        p."CO_NCM",
        n.nome_ncm                                           AS ncm_description,
        sh.descricao_sh2                                     AS sh2_description,
        p.vl_fob,
        ROUND(100.0 * p.vl_fob / SUM(p.vl_fob) OVER (), 2) AS pct_of_total
    FROM rn_panama p
    LEFT JOIN ncm n  ON p."CO_NCM"     = n.codigo_ncm
    LEFT JOIN ncm_sh sh ON n.codigo_sh6 = sh.codigo_sh6
    ORDER BY p.vl_fob DESC
    LIMIT 15;
"""

df_rn_panama = pd.read_sql(query_rn_panama_ncm, engine)
df_rn_panama['vl_fob_mn']      = df_rn_panama['vl_fob'] / 1e6
df_rn_panama['cumulative_pct'] = df_rn_panama['pct_of_total'].cumsum().round(1)

rn_panama_total = df_rn_panama['vl_fob'].sum()
top_product     = df_rn_panama.iloc[0]['ncm_description']
top_product_pct = df_rn_panama.iloc[0]['pct_of_total']
top_sh2         = df_rn_panama.iloc[0]['sh2_description']

#print(f'RN exports to Panama — NCM breakdown, {MAX_YEAR}')
#print(f'Total: ${rn_panama_total/1e6:.1f}m | Dominant product: {top_product} ({top_product_pct:.1f}%)')
#print(f'SH2 sector: {top_sh2}')

## Styled table: RN exports to Panama — NCM breakdown

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

df_rn_panama_display = df_rn_panama[[
    'CO_NCM', 'ncm_description', 'sh2_description', 'vl_fob_mn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'CO_NCM'         : 'NCM',
    'ncm_description': 'Product Description',
    'sh2_description': 'Sector (SH2)',
    'vl_fob_mn'      : 'Exports (USD m)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_rn_panama_display.insert(0, 'Rank', range(1, len(df_rn_panama_display) + 1))

df_rn_panama_display['Exports (USD m)'] = df_rn_panama_display['Exports (USD m)'].apply(
    lambda x: f'${x:.2f}m'
)
df_rn_panama_display['Share (%)'] = df_rn_panama_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_rn_panama_display['Cumulative (%)'] = df_rn_panama_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

## Highlight top product row — it is the transit clue
def highlight_top_row(row):
    if row.name == 0:
        return [f'background-color: {LIGHT_BLUE}; font-weight: 600;'] * len(row)
    return [''] * len(row)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

styler = (
    df_rn_panama_display.style
    .set_caption(
        f'Rio Grande do Norte → Panama — NCM Export Breakdown, {MAX_YEAR} '
        f'| Total: ${rn_panama_total/1e6:.1f}m '
        f'| Dominant product: {top_product_pct:.1f}% '
        f'| Top product row highlighted — identifies likely transit commodity'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_top_row, axis=1)
    .map(colour_share,        subset=['Share (%)'])
    .map(colour_cumulative,   subset=['Cumulative (%)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## NCM code
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Product description
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Sector SH2
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Exports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [   ## Cumulative
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Rank,NCM,Product Description,Sector (SH2),Exports (USD m),Share (%),Cumulative (%)
1,27101922,Fuel oil,"Combustíveis minerais, óleos minerais e produtos da sua destilação; matérias betuminosas; ceras minerais",$487.98m,100.0%,100.0%
2,84251100,"Talhas, cadernais e moitões, de motor elétrico","Reatores nucleares, caldeiras, máquinas, aparelhos e instrumentos mecânicos, e suas partes",$0.03m,0.0%,100.0%
3,41015010,"Couros e peles em bruto, de bovinos, inteiros, de peso unitário superior a 16 kg, sem dividir","Peles, exceto as peles com pelo, e couros",$0.03m,0.0%,100.0%


#### Overview

Panama's 43.3% share of Rio Grande do Norte's exports is explained by a single NCM code. NCM 27101922 (fuel oil) accounts for \\$487.98m, or 100.0% of the $488.0m registered to Panama. The remaining two NCM entries, electric hoists (\\$0.03m) and bovine hides (\\$0.03m) are negligible. Panama is not a demand market for Brazilian fuel oil at this scale; the registration is consistent with transit routing, most plausibly through the Colón Free Trade Zone or onward shipping via the Panama Canal to final destination markets.

The broader partner ranking provides the necessary context. The Netherlands ranks second at \\$147.2m (13.1%), followed by Canada (\\$100.2m, 8.9%), the United States (\\$91.3m, 8.1%), the United Kingdom (\\$67.8m, 6.0%), and Spain (\\$62.9m, 5.6%). These five destinations are consistent with genuine fuel oil demand markets; refining, bunkering, and industrial energy consumers across North America and Europe. China ranks eleventh at \\$13.3m (1.2%), confirming that RN's fuel oil exports do not route predominantly to Asia. The Panama entry at rank 1 with 43.3% represents a single transit flow that distorts the apparent partner concentration; removing it would place the Netherlands as the leading destination at 23.0% of the residual total, with the remaining partners distributed across North America and Western Europe.

Two country codes exist for Panama in the pais table: código 580 (Panamá) and código 895 (Zona do Canal do Panamá). The $488.0m is registered under código 580. The Canal Zone entry (895) records no material export value in this ranking. Whether the flow represents final demand or transit cannot be determined from registration data alone; the general Panama country code is consistent with both direct procurement and Colón Free Trade Zone intermediation.

#### Business Relevance

The investigation closes the RN Panama flag. The $488.0m is a single fuel oil transit flow, not a demand signal from Panama. The true destination market structure for RN fuel oil exports is the partner ranking with Panama removed: North America and Western Europe account for the substantive demand base, with the Netherlands, Canada, the United States, the United Kingdom, and Spain collectively representing 41.7% of total state exports. China's 1.2% share confirms RN fuel oil does not route to Asia at meaningful scale through any registered destination in this dataset.


---
---


## 4.5 — National Sector Studies: Vehicle Import Composition and Manufactured Goods Destinations

Two investigations at national level that require product-level data unavailable from state-level aggregates. Vehicle import composition (finished vs components) determines whether Brazil's \$22.73bn SH87 import deficit reflects consumer demand or manufacturing input dependency. Manufactured goods destination markets establish where Brazil's industrial exports route — the Step 2 China dependency analysis confirmed low China share for these sectors but left the primary markets unconfirmed.

### 4.5.1 — Vehicle Import Composition: Finished Vehicles vs Components (National)

**Open flag:** Brazil imported \\$22.73bn in SH87 (vehicles and parts) in 2025. The national composition — whether dominated by finished vehicles (ready for consumer sale) or components (for assembly in Brazil) — determines the commercial and structural character of this deficit.

**SH4 classification:** 8703 = Passenger Vehicles | 8704 = Trucks/Goods Vehicles | 8708 = Parts & Accessories | 8701 = Tractors | 8702 = Buses


**Output — National SH87 Category Summary and Top NCM Codes:** Reports the finished vehicles vs components split at national level. Brazil has an established domestic automotive sector (Stellantis, Volkswagen, GM, Toyota), implying significant component imports for domestic assembly. The finished/components ratio determines the commercial and structural character of the \$22.73bn import figure.

In [84]:
## National SH87 import decomposition: finished vehicles vs parts

query_sh87_imp = f"""
    WITH sh87_imp AS (
        SELECT
            i."CO_NCM",
            SUM(i."VL_FOB")     AS vl_fob,
            SUM(i."KG_LIQUIDO") AS kg_liq
        FROM imp i
        WHERE i."CO_ANO" = {MAX_YEAR}
          AND CAST(LEFT(i."CO_NCM"::TEXT, 2) AS INT) = 87
        GROUP BY i."CO_NCM"
    )
    SELECT
        p."CO_NCM",
        n.nome_ncm                                           AS ncm_description,
        p.vl_fob,
        ROUND(100.0 * p.vl_fob / SUM(p.vl_fob) OVER (), 2) AS pct_of_total,
        CASE
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 8703 THEN 'Passenger Vehicles'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 8704 THEN 'Trucks / Goods Vehicles'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 8708 THEN 'Parts & Accessories'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 8701 THEN 'Tractors'
            WHEN CAST(LEFT(p."CO_NCM"::TEXT, 4) AS INT) = 8702 THEN 'Buses'
            ELSE 'Other SH87'
        END                                                  AS sh4_category
    FROM sh87_imp p
    LEFT JOIN ncm n ON p."CO_NCM" = n.codigo_ncm
    ORDER BY p.vl_fob DESC
    LIMIT 30;
"""

df_sh87_imp = pd.read_sql(query_sh87_imp, engine)
df_sh87_imp['vl_fob_bn']      = df_sh87_imp['vl_fob'] / 1e9
df_sh87_imp['cumulative_pct'] = df_sh87_imp['pct_of_total'].cumsum().round(1)

sh87_summary = (
    df_sh87_imp.groupby('sh4_category')['vl_fob']
    .sum()
    .sort_values(ascending=False)
)
sh87_total = sh87_summary.sum()

## Finished vs components split
finished_cats   = ['Passenger Vehicles', 'Trucks / Goods Vehicles', 'Buses']
components_cats = ['Parts & Accessories']
finished_total  = sh87_summary[sh87_summary.index.isin(finished_cats)].sum()
components_total = sh87_summary.get('Parts & Accessories', 0)

#print(f'National SH87 import composition, {MAX_YEAR} (${sh87_total/1e9:.2f}bn total):')
#for cat, val in sh87_summary.items():
#    print(f'  {cat}: ${val/1e9:.2f}bn ({100*val/sh87_total:.1f}%)')
#print(f'\nFinished vehicles combined: ${finished_total/1e9:.2f}bn ({100*finished_total/sh87_total:.1f}%)')
#print(f'Parts & Accessories:        ${components_total/1e9:.2f}bn ({100*components_total/sh87_total:.1f}%)')

## Styled tables: national SH87 import composition — summary + NCM detail

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

CAT_COLOURS = {
    'Passenger Vehicles'    : RED,
    'Trucks / Goods Vehicles': RED,
    'Buses'                 : RED,
    'Tractors'              : AMBER,
    'Parts & Accessories'   : GREEN,
    'Other SH87'            : GREY_TEXT,
}

## ── Table 1: category summary ─────────────────────────────────────────────────
df_sh87_summary_display = sh87_summary.reset_index().rename(columns={
    'sh4_category': 'Category',
    'vl_fob'      : 'raw',
})

def get_vehicle_class(cat):
    if cat in ('Passenger Vehicles', 'Trucks / Goods Vehicles', 'Buses'):
        return 'Finished'
    elif cat == 'Parts & Accessories':
        return 'Components'
    elif cat == 'Tractors':
        return 'Mixed'
    return '—'

df_sh87_summary_display['Imports (USD bn)'] = df_sh87_summary_display['raw'].apply(
    lambda x: f'${x/1e9:.2f}bn'
)
df_sh87_summary_display['Share (%)'] = df_sh87_summary_display['raw'].apply(
    lambda x: f'{100*x/sh87_total:.1f}%'
)
df_sh87_summary_display['Classification'] = df_sh87_summary_display['Category'].apply(
    get_vehicle_class
)
df_sh87_summary_display = df_sh87_summary_display[[
    'Category', 'Imports (USD bn)', 'Share (%)', 'Classification'
]]

def colour_category(val):
    c = CAT_COLOURS.get(val, GREY_TEXT)
    return f'color: {c}; font-weight: 700;'

def colour_class(val):
    if val == 'Finished':
        return f'color: {RED}; font-weight: 700;'
    elif val == 'Components':
        return f'color: {GREEN}; font-weight: 700;'
    elif val == 'Mixed':
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_share_summary(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 50.0:
        return f'color: {RED}; font-weight: 700;'
    elif v >= 20.0:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

## Verdict for caption
if finished_total > components_total:
    verdict = f'Consumer market dominant — finished vehicles {100*finished_total/sh87_total:.1f}%'
else:
    verdict = f'Manufacturing input dominant — components {100*components_total/sh87_total:.1f}%'

display(
    df_sh87_summary_display.style
    .set_caption(
        f'Brazil — National SH87 Import Composition, {MAX_YEAR} '
        f'| Total: ${sh87_total/1e9:.2f}bn '
        f'| {verdict}'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_category,      subset=['Category'])
    .map(colour_share_summary, subset=['Share (%)'])
    .map(colour_class,         subset=['Classification'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [
            ('text-align', 'center'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
            ('margin-bottom', '28px'),
        ]},
    ])
    .hide(axis='index')
)

## ── Table 2: NCM detail — top 15 ─────────────────────────────────────────────
df_sh87_ncm_display = df_sh87_imp[[
    'CO_NCM', 'ncm_description', 'sh4_category', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].head(15).rename(columns={
    'CO_NCM'         : 'NCM',
    'ncm_description': 'Product Description',
    'sh4_category'   : 'Category',
    'vl_fob_bn'      : 'Imports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_sh87_ncm_display.insert(0, 'Rank', range(1, len(df_sh87_ncm_display) + 1))

df_sh87_ncm_display['Imports (USD bn)'] = df_sh87_ncm_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_sh87_ncm_display['Share (%)'] = df_sh87_ncm_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_sh87_ncm_display['Cumulative (%)'] = df_sh87_ncm_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

def highlight_category_rows(row):
    tints = {
        'Passenger Vehicles'    : '#fdf0ee',
        'Trucks / Goods Vehicles': '#fdf0ee',
        'Buses'                 : '#fdf0ee',
        'Tractors'              : '#fff8e6',
        'Parts & Accessories'   : '#eaf5ee',
        'Other SH87'            : '',
    }
    bg = tints.get(row['Category'], '')
    return [f'background-color: {bg};' if bg else ''] * len(row)

display(
    df_sh87_ncm_display.style
    .set_caption(
        f'Brazil — National SH87 Import NCM Detail, {MAX_YEAR} '
        f'| Finished vehicles = red | Parts = green | Top 15 NCM codes'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_category_rows, axis=1)
    .map(colour_share,              subset=['Share (%)'])
    .map(colour_cumulative,         subset=['Cumulative (%)'])
    .map(colour_category,           subset=['Category'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [
            ('text-align', 'center'),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

Category,Imports (USD bn),Share (%),Classification
Parts & Accessories,$7.51bn,37.1%,Components
Passenger Vehicles,$7.06bn,34.9%,Finished
Trucks / Goods Vehicles,$4.10bn,20.3%,Finished
Other SH87,$1.23bn,6.1%,—
Buses,$0.21bn,1.0%,Finished
Tractors,$0.13bn,0.6%,Mixed


Rank,NCM,Product Description,Category,Imports (USD bn),Share (%),Cumulative (%)
1,87042190,"Outros veículos automóveis com motor diesel, para carga <= 5 toneladas",Trucks / Goods Vehicles,$3.349bn,14.5%,14.5%
2,87036000,"Outros veículos, equipados para propulsão, simultaneamente, com um motor de pistão alternativo de ignição por centelha (faísca) e um motor elétrico, suscetíveis de serem carregados por conexão a uma fonte externa de energia elétrica",Passenger Vehicles,$2.311bn,10.0%,24.5%
3,87084080,Outras caixas de marchas,Parts & Accessories,$2.227bn,9.7%,34.2%
4,87082999,Outras partes e acessórios de carrocerias para veículos automóveis,Parts & Accessories,$1.186bn,5.1%,39.3%
5,87089990,Outras partes e acessórios para tratores e veículos automóveis,Parts & Accessories,$1.160bn,5.0%,44.4%
6,87141000,Partes e acessórios de motocicletas (inclusive ciclomotores),Other SH87,$1.038bn,4.5%,48.9%
7,87038000,"Outros veículos, equipados unicamente com motor elétrico para propulsão",Passenger Vehicles,$0.998bn,4.3%,53.2%
8,87034000,"Outros veículos, equipados para propulsão, simultaneamente, com um motor de pistão alternativo de ignição por centelha (faísca) e um motor elétrico, exceto os suscetíveis de serem carregados por conexão a uma fonte externa de energia elétrica",Passenger Vehicles,$0.983bn,4.3%,57.4%
9,87032210,"Automóveis com motor explosão, de cilindrada superior a 1.000 cm3, mas não superior a 1.500 cm3, com capacidade de transporte de pessoas sentadas inferior ou igual a seis, incluindo o motorista",Passenger Vehicles,$0.718bn,3.1%,60.6%
10,87032310,"Automóveis com motor explosão, 1500 < cm3 <= 3000, até 6 passageiros",Passenger Vehicles,$0.647bn,2.8%,63.4%


#### Overview

Brazil's \\$20.23bn in SH87 imports splits into two structurally distinct flows: finished vehicles (56.2%, \\$11.37bn combined across passenger vehicles, trucks, and buses) and parts and accessories (37.1%, \\$7.51bn). The finished vehicle majority confirms that Brazil's vehicle import register is consumer and commercial fleet demand driven rather than assembly-input driven at the national level.

The leading NCM code is a commercial vehicle, not a passenger car. NCM 87042190 (diesel trucks, payload up to 5 tonnes) leads at $3.349bn (14.5%), ahead of all passenger vehicle entries. The scale of this single code at 14.5% of total SH87 imports is the a commercially significant single-line finding in the table.

Electrified passenger vehicles account for a combined \\$4.29bn (21.2%) across three NCM codes. Plug-in hybrids (87036000, \\$2.311bn, 10.0%), battery electric vehicles (87038000, \\$0.998bn, 4.3%), and non-plug-in hybrids (87034000, \\$0.983bn, 4.3%) together represent the largest passenger vehicle cluster in the ranking. Conventional internal combustion passenger vehicles across NCM codes 87032210, 87032310, 87032100, and 87033390 contribute a combined \\$2.582bn (11.2%). Electrified vehicles therefore exceed conventional ICE passenger vehicle imports by \\$1.71bn at the national level.

Gearboxes (87084080) at \\$2.227bn (9.7%) are the dominant parts and accessories entry. This is the third largest NCM code in the entire SH87 ranking, behind only diesel trucks and plug-in hybrids. At \\$2.227bn for a single component category, this implies either large-volume supply to domestic vehicle assembly operations or significant aftermarket distribution. Body parts and accessories (87082999, \\$1.186bn, 5.1%) and general vehicle parts (87089990, \\$1.160bn, 5.0%) follow.

Motorcycle parts (87141000, \\$1.038bn, 4.5%) rank sixth nationally. The end-use split between assembly supply and aftermarket distribution is not determinable from national aggregate data and warrants entity-level investigation.

The top 15 NCM codes reach 74.7% cumulative share, leaving 25.3% distributed across the remaining SH87 codes not shown in the table.

#### Business Relevance

The finished vehicle majority at 56.2% confirms that SH87 import demand at the national level is led by vehicle procurement rather than component sourcing. The diesel truck lead at \\$3.349bn identifies commercial fleet replacement as the primary single-product demand driver. The \\$4.29bn electrified passenger vehicle cluster confirms that EV and hybrid imports have reached a scale where they constitute the largest passenger vehicle import category nationally, exceeding conventional ICE imports by a material margin. For participants in automotive distribution, financing, or aftersales, the parts and accessories total at \\$7.51bn represents a substantial secondary market alongside the finished vehicle flow, with gearboxes alone at \\$2.227bn indicating either assembly-line supply or large-scale aftermarket replacement demand that warrants entity-level investigation to determine the split.

---
---


### 4.5.2 — Manufactured Goods Destination Markets

**Open flag:** Brazil exports \\$15.08bn (vehicles), \\$13.89bn (machinery), \\$4.89bn (aircraft) and \\$5.10bn (electrical equipment) in manufactured goods. The destination market breakdown for each sector is unconfirmed at the national level. The Step 2 China dependency analysis established that China accounts for < 5% of these sectors — but where the majority of manufactured goods exports go is the open question.


**Output — Top 10 Destination Markets per Sector:** Run for SH87 (vehicles), SH84 (machinery), SH85 (electrical equipment) and SH88 (aircraft). The key question is whether Brazil's manufactured goods route predominantly to regional neighbours (Mercosul, South America) or have meaningful global reach. Aircraft (SH88) is expected to show the broadest geographic distribution; vehicles (SH87) the most regional concentration.

In [85]:
## Destination market breakdown for key manufactured export sectors
## SH2 codes: 87 (vehicles), 84 (machinery), 85 (electrical equipment), 88 (aircraft)

manufactured_sectors = {
    87: 'Vehicles (SH87)',
    84: 'Machinery (SH84)',
    85: 'Electrical Equipment (SH85)',
    88: 'Aircraft (SH88)',
}

for sh2_code, label in manufactured_sectors.items():

    query_manuf = f"""
        WITH sector_exp AS (
            SELECT
                e."CO_PAIS",
                SUM(e."VL_FOB") AS vl_fob
            FROM exp e
            WHERE e."CO_ANO" = {MAX_YEAR}
              AND CAST(LEFT(e."CO_NCM"::TEXT, 2) AS INT) = {sh2_code}
            GROUP BY e."CO_PAIS"
        )
        SELECT
            s."CO_PAIS",
            p.nome_pais                                          AS partner_country,
            s.vl_fob,
            ROUND(100.0 * s.vl_fob / SUM(s.vl_fob) OVER (), 2) AS pct_of_total
        FROM sector_exp s
        LEFT JOIN pais p ON s."CO_PAIS" = p.codigo_pais
        ORDER BY s.vl_fob DESC
        LIMIT 10;
    """

    df_mp = pd.read_sql(query_manuf, engine)
    df_mp['vl_fob_bn'] = df_mp['vl_fob'] / 1e9

#    print(f'\n{label} — Top destination markets, {MAX_YEAR}')

## Destination market breakdown for key manufactured export sectors
## SH2 codes: 87 (vehicles), 84 (machinery), 85 (electrical equipment), 88 (aircraft)

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

TABLE_STYLES = [
    {'selector': 'caption', 'props': [
        ('caption-side', 'top'),
        ('font-size', '14px'),
        ('font-weight', '700'),
        ('font-family', 'Helvetica Neue, Arial, sans-serif'),
        ('color', DARK_BLUE),
        ('padding', '0 0 10px 0'),
        ('text-align', 'left'),
    ]},
    {'selector': 'thead th', 'props': [
        ('background-color', DARK_BLUE),
        ('color', WHITE),
        ('font-size', '12px'),
        ('font-weight', '600'),
        ('text-transform', 'uppercase'),
        ('letter-spacing', '0.04em'),
        ('padding', '10px 14px'),
        ('border', 'none'),
    ]},
    {'selector': 'thead tr:last-child th', 'props': [
        ('border-bottom', f'2px solid {MID_BLUE}'),
    ]},
    {'selector': 'tbody tr:nth-child(odd)', 'props': [
        ('background-color', WHITE),
    ]},
    {'selector': 'tbody tr:nth-child(even)', 'props': [
        ('background-color', STRIPE),
    ]},
    {'selector': 'tbody tr:hover', 'props': [
        ('background-color', LIGHT_BLUE),
    ]},
    {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
        ('text-align', 'center'),
        ('font-weight', '600'),
        ('color', GREY_TEXT),
        ('font-size', '12px'),
    ]},
    {'selector': 'tbody td:nth-child(2)', 'props': [   ## Partner
        ('font-weight', '500'),
        ('color', DARK_BLUE),
    ]},
    {'selector': 'tbody td:nth-child(3)', 'props': [   ## Exports
        ('text-align', 'right'),
        ('font-variant-numeric', 'tabular-nums'),
    ]},
    {'selector': 'tbody td:nth-child(4)', 'props': [   ## Share
        ('text-align', 'right'),
        ('font-variant-numeric', 'tabular-nums'),
    ]},
    {'selector': 'tbody td:nth-child(5)', 'props': [   ## Cumulative
        ('text-align', 'right'),
        ('font-variant-numeric', 'tabular-nums'),
        ('border-left', f'1px solid {LIGHT_BLUE}'),
    ]},
    {'selector': 'tbody td:nth-child(6)', 'props': [   ## Geography badge
        ('text-align', 'center'),
    ]},
    {'selector': 'table', 'props': [
        ('border-collapse', 'collapse'),
        ('width', '100%'),
        ('border', f'1px solid {LIGHT_BLUE}'),
        ('border-radius', '6px'),
        ('overflow', 'hidden'),
        ('margin-bottom', '28px'),
    ]},
]

BASE_PROPS = {
    'font-family'  : 'Helvetica Neue, Arial, sans-serif',
    'font-size'    : '13px',
    'text-align'   : 'left',
    'padding'      : '8px 14px',
    'border-bottom': f'1px solid {LIGHT_BLUE}',
}

## Region classification for geography badge
LATAM = {
    'argentina', 'mexico', 'chile', 'colombia', 'peru', 'venezuela',
    'equador', 'bolívia', 'paraguai', 'uruguai', 'costa rica',
    'guatemala', 'el salvador', 'honduras', 'panamá', 'cuba',
}

def get_geography(country):
    if pd.isna(country):
        return '—'
    c = str(country).lower()
    if 'estados unidos' in c or 'eua' in c:
        return 'North America'
    elif any(la in c for la in LATAM):
        return 'Latin America'
    elif any(eu in c for eu in ['alemanha', 'reino unido', 'frança', 'itália', 'espanha',
                                 'países baixos', 'bélgica', 'suécia', 'noruega']):
        return 'Europe'
    elif any(as_ in c for as_ in ['china', 'japão', 'coreia', 'índia', 'cingapura',
                                   'tailândia', 'vietnã', 'malásia']):
        return 'Asia'
    elif any(me in c for me in ['emirados', 'arábia', 'turquia', 'israel']):
        return 'Middle East'
    return 'Other'

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

GEO_COLOURS = {
    'Latin America' : '#ad1457',
    'North America' : '#1565c0',
    'Europe'        : '#2e7d32',
    'Asia'          : '#e65100',
    'Middle East'   : '#6a1b9a',
    'Other'         : GREY_TEXT,
}

def colour_geo(val):
    c = GEO_COLOURS.get(val, GREY_TEXT)
    return f'color: {c}; font-weight: 600;'

def highlight_china(row):
    if df_mp.loc[row.name, 'CO_PAIS'] == CHINA_CODE:
        return [f'background-color: #fdf0ee; font-weight: 600;'] * len(row)
    return [''] * len(row)

manufactured_sectors = {
    87: 'Vehicles (SH87)',
    84: 'Machinery (SH84)',
    85: 'Electrical Equipment (SH85)',
    88: 'Aircraft (SH88)',
}

for sh2_code, label in manufactured_sectors.items():

    query_manuf = f"""
        WITH sector_exp AS (
            SELECT
                e."CO_PAIS",
                SUM(e."VL_FOB") AS vl_fob
            FROM exp e
            WHERE e."CO_ANO" = {MAX_YEAR}
              AND CAST(LEFT(e."CO_NCM"::TEXT, 2) AS INT) = {sh2_code}
            GROUP BY e."CO_PAIS"
        )
        SELECT
            s."CO_PAIS",
            p.nome_pais                                          AS partner_country,
            s.vl_fob,
            ROUND(100.0 * s.vl_fob / SUM(s.vl_fob) OVER (), 2) AS pct_of_total
        FROM sector_exp s
        LEFT JOIN pais p ON s."CO_PAIS" = p.codigo_pais
        ORDER BY s.vl_fob DESC
        LIMIT 10;
    """

    df_mp = pd.read_sql(query_manuf, engine)
    df_mp['vl_fob_bn']      = df_mp['vl_fob'] / 1e9
    df_mp['cumulative_pct'] = df_mp['pct_of_total'].cumsum().round(1)

    sector_total  = df_mp['vl_fob'].sum()
    top3_pct      = df_mp['pct_of_total'].head(3).sum()
    china_pct     = df_mp[df_mp['CO_PAIS'] == CHINA_CODE]['pct_of_total'].sum()
    latam_pct     = df_mp[df_mp['partner_country'].str.lower().apply(
        lambda x: any(la in str(x) for la in LATAM) if pd.notna(x) else False
    )]['pct_of_total'].sum()

    df_display = df_mp[[
        'partner_country', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
    ]].rename(columns={
        'partner_country': 'Destination Market',
        'vl_fob_bn'      : 'Exports (USD bn)',
        'pct_of_total'   : 'Share (%)',
        'cumulative_pct' : 'Cumulative (%)',
    }).copy()

    df_display.insert(0, 'Rank', range(1, len(df_display) + 1))

    df_display['Geography'] = df_mp['partner_country'].apply(get_geography)

    df_display['Exports (USD bn)'] = df_display['Exports (USD bn)'].apply(
        lambda x: f'${x:.2f}bn'
    )
    df_display['Share (%)'] = df_display['Share (%)'].apply(
        lambda x: f'{x:.1f}%'
    )
    df_display['Cumulative (%)'] = df_display['Cumulative (%)'].apply(
        lambda x: f'{x:.1f}%'
    )

    display(
        df_display.style
        .set_caption(
            f'{label} — Export Destination Markets, {MAX_YEAR} '
            f'| Total: ${sector_total/1e9:.2f}bn '
            f'| Top 3: {top3_pct:.1f}% '
            f'| China: {china_pct:.1f}% '
            f'| Latin America: {latam_pct:.1f}% '
            f'| China row highlighted'
        )
        .set_properties(**BASE_PROPS)
        .apply(highlight_china,   axis=1)
        .map(colour_share,        subset=['Share (%)'])
        .map(colour_cumulative,   subset=['Cumulative (%)'])
        .map(colour_geo,          subset=['Geography'])
        .set_table_styles(TABLE_STYLES)
        .hide(axis='index')
    )

Rank,Destination Market,Exports (USD bn),Share (%),Cumulative (%),Geography
1,Argentina,$7.83bn,52.0%,52.0%,Latin America
2,México,$1.35bn,8.9%,60.9%,Other
3,Chile,$1.16bn,7.7%,68.6%,Latin America
4,Colômbia,$0.92bn,6.1%,74.7%,Other
5,Peru,$0.83bn,5.5%,80.2%,Latin America
6,Uruguai,$0.61bn,4.0%,84.2%,Latin America
7,Estados Unidos,$0.55bn,3.6%,87.8%,North America
8,Paraguai,$0.40bn,2.6%,90.4%,Latin America
9,África do Sul,$0.19bn,1.3%,91.7%,Other
10,Equador,$0.12bn,0.8%,92.5%,Latin America


Rank,Destination Market,Exports (USD bn),Share (%),Cumulative (%),Geography
1,Estados Unidos,$3.33bn,24.0%,24.0%,North America
2,Argentina,$1.89bn,13.6%,37.5%,Latin America
3,Singapura,$1.39bn,10.0%,47.6%,Other
4,México,$0.95bn,6.8%,54.4%,Other
5,Paraguai,$0.57bn,4.1%,58.5%,Latin America
6,Alemanha,$0.50bn,3.6%,62.1%,Europe
7,Chile,$0.46bn,3.3%,65.4%,Latin America
8,Peru,$0.39bn,2.8%,68.2%,Latin America
9,Colômbia,$0.33bn,2.4%,70.6%,Other
10,Canadá,$0.25bn,1.8%,72.4%,Other


Rank,Destination Market,Exports (USD bn),Share (%),Cumulative (%),Geography
1,Estados Unidos,$1.56bn,30.5%,30.5%,North America
2,Argentina,$0.76bn,14.9%,45.5%,Latin America
3,México,$0.29bn,5.7%,51.1%,Other
4,Chile,$0.28bn,5.4%,56.6%,Latin America
5,Paraguai,$0.23bn,4.6%,61.1%,Latin America
6,Colômbia,$0.15bn,3.0%,64.1%,Other
7,Alemanha,$0.14bn,2.7%,66.8%,Europe
8,Canadá,$0.10bn,2.0%,68.8%,Other
9,Peru,$0.09bn,1.8%,70.6%,Latin America
10,China,$0.09bn,1.8%,72.3%,Asia


Rank,Destination Market,Exports (USD bn),Share (%),Cumulative (%),Geography
1,Estados Unidos,$3.05bn,62.4%,62.4%,North America
2,Irlanda,$0.24bn,4.9%,67.3%,Other
3,Canadá,$0.23bn,4.6%,71.9%,Other
4,Portugal,$0.21bn,4.4%,76.3%,Other
5,México,$0.20bn,4.2%,80.4%,Other
6,Espanha,$0.18bn,3.6%,84.0%,Europe
7,Hungria,$0.14bn,3.0%,87.0%,Other
8,Países Baixos (Holanda),$0.11bn,2.2%,89.3%,Europe
9,França,$0.10bn,2.1%,91.4%,Europe
10,Paraguai,$0.07bn,1.5%,92.8%,Latin America


#### Overview

The four manufactured export sectors reveal distinct destination market structures that segment cleanly into two geographic orientations: regional Latin American markets (vehicles, electrical equipment) and global developed economy markets (machinery, aircraft).

Vehicles (SH87, \\$13.95bn) are structurally concentrated in Latin America at 72.6%. Argentina alone accounts for \\$7.83bn (52.0%) of total vehicle exports, making it by far the largest single bilateral destination for any manufactured sector in this table. Chile (\\$1.16bn, 7.7%), Peru (\\$0.83bn, 5.5%), Uruguay (\\$0.61bn, 4.0%), and Paraguay ($0.40bn, 2.6%) add a further 19.8%. China accounts for 0.0% of vehicle exports. The United States registers at only $0.55bn (3.6%). The vehicle export market is a regional Mercosul and South American market with negligible exposure to global developed economies or Asia.

Machinery (SH84, \\$10.06bn) shows a materially different structure. The United States leads at \\$3.33bn (24.0%), followed by Argentina (\\$1.89bn, 13.6%) and Singapore (\\$1.39bn, 10.0%). Latin America collectively accounts for 23.8%, less than a third of the vehicle sector's regional concentration. Singapore at 10.0% is not a plausible domestic demand market for $1.39bn in Brazilian machinery at this scale. The destination warrants entity-level investigation before a commercial interpretation is drawn. China registers at 0.0% in the top 10. Germany (\\$0.50bn, 3.6%) and Canada (\\$0.25bn, 1.8%) are the only European entries of note.

Electrical equipment (SH85, $3.69bn) mirrors the machinery geographic orientation but with higher US concentration. The United States leads at \\$1.56bn (30.5%), with Argentina second at \\$0.76bn (14.9%). Latin America accounts for 26.7% combined. China appears at rank 10 with \\$0.09bn (1.8%), the only instance of China appearing in the top 10 of any manufactured sector destination ranking in this table. Germany (\\$0.14bn, 2.7%) and Canada (\\$0.10bn, 2.0%) represent modest European and North American secondary destinations.

Aircraft (SH88, \\$4.54bn) is the most globally oriented and most US-concentrated manufactured sector. The United States accounts for \\$3.05bn (62.4%) of total aircraft exports, a concentration level that exceeds even Argentina's 52.0% share of vehicle exports in absolute percentage terms. Ireland (\\$0.24bn, 4.9%), Canada (\\$0.23bn, 4.6%), Portugal (\\$0.21bn, 4.4%), and Mexico (\\$0.20bn, 4.2%) follow. European destinations including Spain (\\$0.18bn), Hungary (\\$0.14bn), the Netherlands ($0.11bn), and France (\\$0.10bn) collectively account for 10.9%. Latin America contributes only 1.5% via Paraguay alone. China registers at 0.0%. The aircraft sector's US concentration at 62.4% reflects the bilateral structure of commercial aircraft procurement rather than geographic diversification.

China's absence across all four sectors is the cross-cutting finding. China accounts for 0.0% of vehicle exports, 0.0% of machinery exports in the top 10, 1.8% of electrical equipment exports, and 0.0% of aircraft exports. This stands in direct contrast to China's dominant position as a destination for Brazilian commodity exports across soybeans, iron ore, crude oil, cotton, and ferro-niobium. Brazil's manufactured export relationship with China is negligible relative to its commodity export relationship.

#### Business Relevance

The two-cluster structure has direct commercial implications. Vehicle exports are a regional market story: Argentina's \\$7.83bn bilateral flow means that Brazilian automotive export revenue is heavily exposed to Argentine economic conditions, exchange rate movements, and trade policy. For machinery and electrical equipment, the United States at 24.0% and 30.5% respectively establishes North America as the primary developed-economy destination, with Latin America as a secondary regional market. Aircraft at 62.4% US concentration reflects the procurement geography of the global commercial aviation market rather than a Brazil-specific trade relationship. Singapore's 10.0% share of machinery exports warrants entity-level investigation to determine whether this represents genuine capital goods demand or trading intermediation routing to undisclosed final markets.

---
---


## 4.6 — Paraná Trade Balance Reversal: Product and Partner Analysis

**Open flag:** Paraná's trade balance shifted from -\\$1.51bn (2011) to +\\$3.50bn (2025) — a \\$5.01bn swing. The mechanism has not been identified at product level. Three hypotheses were proposed in Step 2: agricultural export expansion outpacing import growth; import contraction in specific sectors; or logistics and port infrastructure improvements enabling better export conditions.

This chapter isolates which SH2 sectors drove the shift by comparing export and import composition at the deficit peak (2011) against MAX_YEAR, and identifies the primary new export markets if any emerged. Paraná's high overlap score (4/5) and co-location of export and import municipalities from Step 3 make it one of the most commercially well-structured states for follow-on analysis.


**Table 1 — Annual Trade Balance, 2009–2025:** Establishes the deficit peak year (2011, -\\$1.51bn) and the trajectory to surplus (2025, +\\$3.50bn). Identifies intermediate years where the balance crossed zero — useful for pinpointing when the structural shift occurred.

In [86]:
## Paraná: SH2-level export and import time series, 2009–MAX_YEAR
## SG_UF_NCM = 'PR'

query_pr_exp_sh2 = f"""
    SELECT
        e."CO_ANO",
        sh.codigo_sh2,
        sh.descricao_sh2                AS sh2_description,
        SUM(e."VL_FOB")                 AS exp_fob
    FROM exp e
    LEFT JOIN ncm n   ON e."CO_NCM"     = n.codigo_ncm
    LEFT JOIN ncm_sh sh ON n.codigo_sh6 = sh.codigo_sh6
    WHERE e."SG_UF_NCM" = 'PR'
      AND e."CO_ANO" BETWEEN 2009 AND {MAX_YEAR}
    GROUP BY e."CO_ANO", sh.codigo_sh2, sh.descricao_sh2
    ORDER BY e."CO_ANO", exp_fob DESC;
"""

query_pr_imp_sh2 = f"""
    SELECT
        i."CO_ANO",
        sh.codigo_sh2,
        sh.descricao_sh2                AS sh2_description,
        SUM(i."VL_FOB")                 AS imp_fob
    FROM imp i
    LEFT JOIN ncm n   ON i."CO_NCM"     = n.codigo_ncm
    LEFT JOIN ncm_sh sh ON n.codigo_sh6 = sh.codigo_sh6
    WHERE i."SG_UF_NCM" = 'PR'
      AND i."CO_ANO" BETWEEN 2009 AND {MAX_YEAR}
    GROUP BY i."CO_ANO", sh.codigo_sh2, sh.descricao_sh2
    ORDER BY i."CO_ANO", imp_fob DESC;
"""

df_pr_exp = pd.read_sql(query_pr_exp_sh2, engine)
df_pr_imp = pd.read_sql(query_pr_imp_sh2, engine)

pr_exp_annual = df_pr_exp.groupby('CO_ANO')['exp_fob'].sum() / 1e9
pr_imp_annual = df_pr_imp.groupby('CO_ANO')['imp_fob'].sum() / 1e9
pr_balance    = (pr_exp_annual - pr_imp_annual).round(2)

## Identify reversal year — first year with positive balance
reversal_year = pr_balance[pr_balance > 0].index.min()
deficit_peak  = pr_balance.idxmin()

#print(f'Paraná annual trade balance, 2009–{MAX_YEAR}')
#print(f'Deficit peak: {deficit_peak} (${pr_balance[deficit_peak]:.2f}bn)')
#print(f'Reversal year: {reversal_year}')

## Styled table: Paraná annual trade balance time series

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

df_pr_ts = pd.DataFrame({
    'CO_ANO'    : pr_exp_annual.index,
    'exp_bn'    : pr_exp_annual.values.round(2),
    'imp_bn'    : pr_imp_annual.values.round(2),
    'balance_bn': pr_balance.values,
}).reset_index(drop=True)

df_pr_display = df_pr_ts.rename(columns={
    'CO_ANO'    : 'Year',
    'exp_bn'    : 'Exports (USD bn)',
    'imp_bn'    : 'Imports (USD bn)',
    'balance_bn': 'Balance (USD bn)',
}).copy()

df_pr_display['Exports (USD bn)'] = df_pr_display['Exports (USD bn)'].apply(
    lambda x: f'${x:.2f}bn'
)
df_pr_display['Imports (USD bn)'] = df_pr_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.2f}bn'
)
df_pr_display['Balance (USD bn)'] = df_pr_display['Balance (USD bn)'].apply(
    lambda x: f'+${x:.2f}bn' if x >= 0 else f'-${abs(x):.2f}bn'
)

def colour_year(val):
    if val == deficit_peak:
        return f'color: {RED}; font-weight: 700;'
    if val == reversal_year:
        return f'color: {GREEN}; font-weight: 700;'
    if val in [2009, 2011, 2015, 2020, MAX_YEAR]:
        return f'color: {DARK_BLUE}; font-weight: 700;'
    return f'color: {GREY_TEXT};'

def colour_balance(val):
    if val.startswith('+'):
        try:
            v = float(val.replace('+$', '').replace('bn', ''))
        except (ValueError, AttributeError):
            return f'color: {GREEN}; font-weight: 600;'
        if v >= 3.0:
            return f'color: {GREEN}; font-weight: 700;'
        return f'color: {GREEN}; font-weight: 600;'
    elif val.startswith('-'):
        try:
            v = abs(float(val.replace('-$', '').replace('bn', '')))
        except (ValueError, AttributeError):
            return f'color: {RED}; font-weight: 600;'
        if v >= 1.0:
            return f'color: {RED}; font-weight: 700;'
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {GREY_TEXT};'

styler = (
    df_pr_display.style
    .set_caption(
        f'Paraná (PR) — Annual Trade Balance, 2009–{MAX_YEAR} '
        f'| Deficit peak: {deficit_peak} (${pr_balance[deficit_peak]:.2f}bn) '
        f'| Reversal: {reversal_year} '
        f'| Deficit peak = red | Reversal year = green'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'right',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_year,    subset=['Year'])
    .map(colour_balance, subset=['Balance (USD bn)'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Year
            ('text-align', 'left'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## Exports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## Balance
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'2px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
            ('margin-bottom', '28px'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Year,Exports (USD bn),Imports (USD bn),Balance (USD bn)
2009,$11.13bn,$9.64bn,+$1.49bn
2010,$14.04bn,$13.96bn,+$0.08bn
2011,$17.29bn,$18.80bn,-$1.51bn
2012,$17.62bn,$19.49bn,-$1.87bn
2013,$18.10bn,$19.43bn,-$1.33bn
2014,$16.24bn,$17.33bn,-$1.09bn
2015,$14.83bn,$12.49bn,+$2.34bn
2016,$15.01bn,$11.17bn,+$3.85bn
2017,$17.93bn,$12.68bn,+$5.25bn
2018,$18.10bn,$14.10bn,+$4.00bn


**Table 2 — Top SH2 Export and Import Sectors: 2011 vs 2025:** The comparative sector table is the primary diagnostic. Sectors that appear in 2025 but not 2011 exports (or grew substantially) are the likely drivers of the surplus. Sectors that contracted on the import side also contribute to the balance reversal.

In [87]:
## Compare top SH2 export sectors: 2011 (deficit peak) vs MAX_YEAR (surplus)

for year in [2011, MAX_YEAR]:
    df_exp_yr = df_pr_exp[df_pr_exp['CO_ANO'] == year].sort_values('exp_fob', ascending=False).head(10)
    df_imp_yr = df_pr_imp[df_pr_imp['CO_ANO'] == year].sort_values('imp_fob', ascending=False).head(5)
    df_exp_yr = df_exp_yr.copy()
    df_exp_yr['exp_fob_bn'] = df_exp_yr['exp_fob'] / 1e9
    df_imp_yr = df_imp_yr.copy()
    df_imp_yr['imp_fob_bn'] = df_imp_yr['imp_fob'] / 1e9

## Compare top SH2 export and import sectors: 2011 (deficit peak) vs MAX_YEAR (surplus)

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

TABLE_STYLES = [
    {'selector': 'caption', 'props': [
        ('caption-side', 'top'),
        ('font-size', '14px'),
        ('font-weight', '700'),
        ('font-family', 'Helvetica Neue, Arial, sans-serif'),
        ('color', DARK_BLUE),
        ('padding', '0 0 10px 0'),
        ('text-align', 'left'),
    ]},
    {'selector': 'thead th', 'props': [
        ('background-color', DARK_BLUE),
        ('color', WHITE),
        ('font-size', '12px'),
        ('font-weight', '600'),
        ('text-transform', 'uppercase'),
        ('letter-spacing', '0.04em'),
        ('padding', '10px 14px'),
        ('border', 'none'),
    ]},
    {'selector': 'thead tr:last-child th', 'props': [
        ('border-bottom', f'2px solid {MID_BLUE}'),
    ]},
    {'selector': 'tbody tr:nth-child(odd)', 'props': [
        ('background-color', WHITE),
    ]},
    {'selector': 'tbody tr:nth-child(even)', 'props': [
        ('background-color', STRIPE),
    ]},
    {'selector': 'tbody tr:hover', 'props': [
        ('background-color', LIGHT_BLUE),
    ]},
    {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
        ('text-align', 'center'),
        ('font-weight', '600'),
        ('color', GREY_TEXT),
        ('font-size', '12px'),
    ]},
    {'selector': 'tbody td:nth-child(2)', 'props': [   ## SH2 code
        ('font-family', 'monospace'),
        ('font-size', '12px'),
        ('color', GREY_TEXT),
        ('text-align', 'center'),
    ]},
    {'selector': 'tbody td:nth-child(3)', 'props': [   ## Description
        ('font-weight', '500'),
        ('color', DARK_BLUE),
    ]},
    {'selector': 'tbody td:nth-child(4)', 'props': [   ## Value
        ('text-align', 'right'),
        ('font-variant-numeric', 'tabular-nums'),
    ]},
    {'selector': 'tbody td:nth-child(5)', 'props': [   ## Change badge
        ('text-align', 'center'),
    ]},
    {'selector': 'table', 'props': [
        ('border-collapse', 'collapse'),
        ('width', '100%'),
        ('border', f'1px solid {LIGHT_BLUE}'),
        ('border-radius', '6px'),
        ('overflow', 'hidden'),
        ('margin-bottom', '28px'),
    ]},
]

BASE_PROPS = {
    'font-family'  : 'Helvetica Neue, Arial, sans-serif',
    'font-size'    : '13px',
    'text-align'   : 'left',
    'padding'      : '8px 14px',
    'border-bottom': f'1px solid {LIGHT_BLUE}',
}

## Pre-compute 2011 and MAX_YEAR sector totals for comparison badges
def get_sector_totals(df_exp, df_imp, year):
    exp = df_exp[df_exp['CO_ANO'] == year].set_index('codigo_sh2')['exp_fob'].to_dict()
    imp = df_imp[df_imp['CO_ANO'] == year].set_index('codigo_sh2')['imp_fob'].to_dict()
    return exp, imp

exp_2011, imp_2011 = get_sector_totals(df_pr_exp, df_pr_imp, 2011)
exp_curr, imp_curr = get_sector_totals(df_pr_exp, df_pr_imp, MAX_YEAR)

def get_change_badge(sh2, val_2011, val_curr):
    old = val_2011.get(sh2, 0)
    new = val_curr.get(sh2, 0)
    if old == 0:
        return 'New'
    pct = (new - old) / old * 100
    if pct >= 100:
        return f'+{pct:.0f}%'
    elif pct >= 0:
        return f'+{pct:.0f}%'
    else:
        return f'{pct:.0f}%'

def colour_change(val):
    if val == 'New':
        return f'color: #1565c0; font-weight: 700;'
    try:
        v = float(val.replace('%', '').replace('+', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 100:
        return f'color: {GREEN}; font-weight: 700;'
    elif v >= 0:
        return f'color: {GREEN}; font-weight: 500;'
    else:
        return f'color: {RED}; font-weight: 600;'

def colour_value(val):
    try:
        v = float(val.replace('$', '').replace('bn', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 5.0:
        return f'color: {DARK_BLUE}; font-weight: 700;'
    elif v >= 1.0:
        return f'color: {DARK_BLUE}; font-weight: 500;'
    return f'color: {GREY_TEXT};'

for year in [2011, MAX_YEAR]:

    year_label   = f'{year} (Deficit Peak)' if year == 2011 else f'{year} (Surplus)'
    pr_bal_year  = pr_balance.get(year, 0)
    bal_str      = f'+${pr_bal_year:.2f}bn' if pr_bal_year >= 0 else f'-${abs(pr_bal_year):.2f}bn'

    ## ── Export table ──────────────────────────────────────────────────────────
    df_exp_yr = (
        df_pr_exp[df_pr_exp['CO_ANO'] == year]
        .sort_values('exp_fob', ascending=False)
        .head(10)
        .copy()
    )
    df_exp_yr['exp_fob_bn'] = df_exp_yr['exp_fob'] / 1e9
    exp_total_yr = df_exp_yr['exp_fob'].sum()

    df_exp_display = df_exp_yr[[
        'codigo_sh2', 'sh2_description', 'exp_fob_bn'
    ]].rename(columns={
        'codigo_sh2'     : 'SH2',
        'sh2_description': 'Sector',
        'exp_fob_bn'     : 'Exports (USD bn)',
    }).copy()

    df_exp_display.insert(0, 'Rank', range(1, len(df_exp_display) + 1))
    df_exp_display['Exports (USD bn)'] = df_exp_display['Exports (USD bn)'].apply(
        lambda x: f'${x:.2f}bn'
    )

    ## Change badge vs the other year
    other_val = exp_2011 if year == MAX_YEAR else exp_curr
    df_exp_display['vs ' + ('2011' if year == MAX_YEAR else str(MAX_YEAR))] = (
        df_exp_yr['codigo_sh2'].apply(
            lambda sh2: get_change_badge(sh2, exp_2011, exp_curr)
            if year == MAX_YEAR
            else get_change_badge(sh2, exp_curr, exp_2011)
        )
    )

    change_col = 'vs ' + ('2011' if year == MAX_YEAR else str(MAX_YEAR))

    display(
        df_exp_display.style
        .set_caption(
            f'Paraná — Top 10 Export Sectors, {year_label} '
            f'| Total shown: ${exp_total_yr/1e9:.2f}bn '
            f'| Balance: {bal_str}'
        )
        .set_properties(**BASE_PROPS)
        .map(colour_value,  subset=['Exports (USD bn)'])
        .map(colour_change, subset=[change_col])
        .set_table_styles(TABLE_STYLES)
        .hide(axis='index')
    )

    ## ── Import table ──────────────────────────────────────────────────────────
    df_imp_yr = (
        df_pr_imp[df_pr_imp['CO_ANO'] == year]
        .sort_values('imp_fob', ascending=False)
        .head(5)
        .copy()
    )
    df_imp_yr['imp_fob_bn'] = df_imp_yr['imp_fob'] / 1e9
    imp_total_yr = df_imp_yr['imp_fob'].sum()

    df_imp_display = df_imp_yr[[
        'codigo_sh2', 'sh2_description', 'imp_fob_bn'
    ]].rename(columns={
        'codigo_sh2'     : 'SH2',
        'sh2_description': 'Sector',
        'imp_fob_bn'     : 'Imports (USD bn)',
    }).copy()

    df_imp_display.insert(0, 'Rank', range(1, len(df_imp_display) + 1))
    df_imp_display['Imports (USD bn)'] = df_imp_display['Imports (USD bn)'].apply(
        lambda x: f'${x:.2f}bn'
    )

    df_imp_display['vs ' + ('2011' if year == MAX_YEAR else str(MAX_YEAR))] = (
        df_imp_yr['codigo_sh2'].apply(
            lambda sh2: get_change_badge(sh2, imp_2011, imp_curr)
            if year == MAX_YEAR
            else get_change_badge(sh2, imp_curr, imp_2011)
        )
    )

    display(
        df_imp_display.style
        .set_caption(
            f'Paraná — Top 5 Import Sectors, {year_label} '
            f'| Total shown: ${imp_total_yr/1e9:.2f}bn'
        )
        .set_properties(**BASE_PROPS)
        .map(colour_value,  subset=['Imports (USD bn)'])
        .map(colour_change, subset=[change_col])
        .set_table_styles(TABLE_STYLES)
        .hide(axis='index')
    )

Rank,SH2,Sector,Exports (USD bn),vs 2025
1,12,"Sementes e frutos oleaginosos; grãos, sementes e frutos diversos; plantas industriais ou medicinais; palhas e forragens",$3.38bn,-27%
2,2,"Carnes e miudezas, comestíveis",$2.18bn,-50%
3,87,"Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$1.84bn,-13%
4,17,Açúcares e produtos de confeitaria,$1.51bn,+23%
5,23,Resíduos e desperdícios das indústrias alimentares; alimentos preparados para animais,$1.41bn,+3%
6,84,"Reatores nucleares, caldeiras, máquinas, aparelhos e instrumentos mecânicos, e suas partes",$0.92bn,-28%
7,15,Gorduras e óleos animais ou vegetais; produtos da sua dissociação; gorduras alimentares elaboradas; ceras de origem animal ou vegetal,$0.79bn,+37%
8,10,Cereais,$0.66bn,-44%
9,44,"Madeira, carvão vegetal e obras de madeira",$0.64bn,-41%
10,48,"Papel e cartão; obras de pasta de celulose, de papel ou de cartão",$0.46bn,-45%


Rank,SH2,Sector,Imports (USD bn),vs 2025
1,87,"Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$3.74bn,+91%
2,84,"Reatores nucleares, caldeiras, máquinas, aparelhos e instrumentos mecânicos, e suas partes",$3.10bn,+1%
3,27,"Combustíveis minerais, óleos minerais e produtos da sua destilação; matérias betuminosas; ceras minerais",$2.55bn,+50%
4,31,Adubos (fertilizantes),$1.68bn,-44%
5,85,"Máquinas, aparelhos e materiais elétricos, e suas partes; aparelhos de gravação ou de reprodução de som, aparelhos de gravação ou de reprodução de imagens e de som em televisão, e suas partes e acessórios",$1.67bn,+12%


Rank,SH2,Sector,Exports (USD bn),vs 2011
1,12,"Sementes e frutos oleaginosos; grãos, sementes e frutos diversos; plantas industriais ou medicinais; palhas e forragens",$4.64bn,+37%
2,2,"Carnes e miudezas, comestíveis",$4.39bn,+101%
3,87,"Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$2.13bn,+15%
4,23,Resíduos e desperdícios das indústrias alimentares; alimentos preparados para animais,$1.36bn,-3%
5,84,"Reatores nucleares, caldeiras, máquinas, aparelhos e instrumentos mecânicos, e suas partes",$1.29bn,+40%
6,17,Açúcares e produtos de confeitaria,$1.22bn,-19%
7,10,Cereais,$1.18bn,+80%
8,44,"Madeira, carvão vegetal e obras de madeira",$1.08bn,+69%
9,48,"Papel e cartão; obras de pasta de celulose, de papel ou de cartão",$0.84bn,+82%
10,27,"Combustíveis minerais, óleos minerais e produtos da sua destilação; matérias betuminosas; ceras minerais",$0.76bn,+167%


Rank,SH2,Sector,Imports (USD bn),vs 2011
1,84,"Reatores nucleares, caldeiras, máquinas, aparelhos e instrumentos mecânicos, e suas partes",$3.07bn,-1%
2,31,Adubos (fertilizantes),$3.02bn,+80%
3,87,"Veículos automóveis, tratores, ciclos e outros veículos terrestres, suas partes e acessórios",$1.96bn,-48%
4,27,"Combustíveis minerais, óleos minerais e produtos da sua destilação; matérias betuminosas; ceras minerais",$1.70bn,-33%
5,85,"Máquinas, aparelhos e materiais elétricos, e suas partes; aparelhos de gravação ou de reprodução de som, aparelhos de gravação ou de reprodução de imagens e de som em televisão, e suas partes e acessórios",$1.50bn,-10%


#### Overview

Paraná's trade balance shifted from a deficit peak of -\\$1.87bn in 2012 to a surplus of +\\$3.50bn in 2025, a swing of \\$5.37bn. The time series identifies two distinct deficit years (2011 at -\\$1.51bn and 2012 at -\\$1.87bn) followed by a sustained surplus from 2015 onward, with the exception of 2022 (-\\$0.27bn). The 2023 surplus of +$7.10bn is the highest in the series.

The balance reversal is driven by export growth rather than import contraction. Total exports grew from \\$17.29bn (2011) to \\$23.65bn (2025), a 37% increase. Total imports grew from \\$18.80bn (2011) to \\$20.15bn (2025), a 7% increase. The surplus emerged because export growth outpaced import growth across the period rather than through any structural reduction in import volumes.

The sector comparison identifies meat (SH2 02) as the primary driver of the export-side shift. Meat and edible offal exports grew from \\$2.18bn in 2011 to \\$4.39bn in 2025, a 101% increase and the largest absolute gain in the export ranking. Oilseeds (SH2 12) grew from \\$3.38bn to \\$4.64bn (+37%), retaining the top export rank. Cereals (SH2 10) grew from \\$0.66bn to \\$1.18bn (+80%), timber (SH2 44) from \\$0.64bn to \\$1.08bn (+69%), and paper and cardboard (SH2 48) from \\$0.46bn to \\$0.84bn (+82%). Mineral fuels (SH2 27) grew from a negligible export base to \\$0.76bn (+167%), the highest percentage growth in the table though from a low base. These five sectors together added approximately \\$3.60bn in export value between 2011 and 2025.

On the import side, vehicles (SH87) show the most significant contraction. Vehicle imports fell from \\$3.74bn (2011) to \\$1.96bn (2025), a 48% reduction and the largest absolute import decline in the table. Mineral fuel imports fell from \\$2.55bn to \\$1.70bn (-33%). Fertiliser imports (SH2 31) are the exception, growing from \\$1.68bn to \\$3.02bn (+80%), consistent with the expansion of Paraná's agricultural output base requiring increased input volumes. Machinery imports (SH2 84) remained flat at approximately $3.07bn across the period.

The 2022 deficit (-\\$0.27bn) is an interruption rather than a structural reversal. Imports reached \\$22.40bn in 2022, the highest in the series, while exports reached \\$22.13bn. The balance returned to surplus in 2023 at +\\$7.10bn as exports grew to \\$25.28bn and imports fell to \\$18.18bn, confirming 2022 as a single-year anomaly rather than a renewed structural deficit.

#### Business Relevance

The Paraná balance reversal is an agricultural export growth story combined with a vehicle import contraction. The meat sector doubling from \\$2.18bn to \\$4.39bn is the single largest contributor to the surplus shift. Fertiliser imports growing 80% in parallel confirms that agricultural output expansion is input-intensive, creating a structural import dependency on fertilisers that partially offsets the export gains. Vehicle imports fell from \\$3.74bn (2011) to \\$1.96bn (2025). Whether this reflects domestic production substitution, demand contraction, or a registration shift to another state has not been determined and the contraction should not be attributed to any single cause without NCM-level follow-up. The meat sector growth from \\$2.18bn to \\$4.39bn is the most clearly supported driver of the surplus shift, it is directly observable in the export data and requires no further decomposition to confirm its directional contribution.

---
---


## 4.7 — Remaining Open Flags: Anchieta, Aracruz, Marabá and Post-2015 Deficit Convergence

Three municipality-level investigations and one time-series analysis that did not fit the thematic clusters of earlier chapters. Anchieta and Aracruz test whether Espírito Santo's ore and pulp exports route to China at below-average rates due to product-mix differences. Marabá's unexpectedly low China share requires product confirmation. The post-2015 deficit convergence analysis tests whether multiple municipalities entering deficit positions in the same window as the Step 2c regression weakening is systematic or coincidental.


### 4.7.1 — Anchieta and Aracruz (Espírito Santo): Export Destination Markets

**Open flag:** Step 2 identified that Espírito Santo's ore (9.6%) and pulp (17.5%) China routing is below the national average for comparable states. Anchieta and Aracruz are the primary municipalities registering these exports. Their destination market composition is unconfirmed — whether the below-average China share reflects genuine destination diversification or a product-mix distinction within the ore and pulp categories.

**CO_MUN:** Anchieta = 3200904 | Aracruz = 3200607


**Output — SH4 Export Profile and Destination Markets per Municipality:** Run for both Anchieta (CO_MUN 3200904) and Aracruz (CO_MUN 3200607). Anchieta is expected to show iron ore / steel products (SH26/SH72); Aracruz is expected to show eucalyptus pulp (SH47) with Europe as primary destination. Confirm or revise the 9.6% (ore) and 17.5% (pulp) China share figures from Step 2.

In [2]:
## Anchieta and Aracruz: SH4-level export profile + partner breakdown from exp_mun
## CO_MUN: Anchieta = 3200904, Aracruz = 3200607

from IPython.display import display

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

TABLE_STYLES = [
    {'selector': 'caption', 'props': [
        ('caption-side', 'top'),
        ('font-size', '14px'),
        ('font-weight', '700'),
        ('font-family', 'Helvetica Neue, Arial, sans-serif'),
        ('color', DARK_BLUE),
        ('padding', '0 0 10px 0'),
        ('text-align', 'left'),
    ]},
    {'selector': 'thead th', 'props': [
        ('background-color', DARK_BLUE),
        ('color', WHITE),
        ('font-size', '12px'),
        ('font-weight', '600'),
        ('text-transform', 'uppercase'),
        ('letter-spacing', '0.04em'),
        ('padding', '10px 14px'),
        ('border', 'none'),
    ]},
    {'selector': 'thead tr:last-child th', 'props': [
        ('border-bottom', f'2px solid {MID_BLUE}'),
    ]},
    {'selector': 'tbody tr:nth-child(odd)', 'props': [
        ('background-color', WHITE),
    ]},
    {'selector': 'tbody tr:nth-child(even)', 'props': [
        ('background-color', STRIPE),
    ]},
    {'selector': 'tbody tr:hover', 'props': [
        ('background-color', LIGHT_BLUE),
    ]},
    {'selector': 'table', 'props': [
        ('border-collapse', 'collapse'),
        ('width', '100%'),
        ('border', f'1px solid {LIGHT_BLUE}'),
        ('border-radius', '6px'),
        ('overflow', 'hidden'),
        ('margin-bottom', '20px'),
    ]},
]

BASE_PROPS = {
    'font-family'  : 'Helvetica Neue, Arial, sans-serif',
    'font-size'    : '13px',
    'text-align'   : 'left',
    'padding'      : '8px 14px',
    'border-bottom': f'1px solid {LIGHT_BLUE}',
}

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

for mun_code, mun_name in [(3200904, 'Anchieta'), (3200607, 'Aracruz')]:

    ## ── Queries ───────────────────────────────────────────────────────────────
    query_mun_sh4 = f"""
        WITH mun_exp AS (
            SELECT
                em."SH4",
                SUM(em."VL_FOB") AS vl_fob
            FROM exp_mun em
            WHERE em."CO_MUN" = {mun_code}
              AND em."CO_ANO" = {MAX_YEAR}
            GROUP BY em."SH4"
        ),
        ncm_sh_dedup AS (
            SELECT DISTINCT ON (codigo_sh4)
                codigo_sh4,
                descricao_sh4,
                descricao_sh2,
                codigo_sh2
            FROM ncm_sh
            ORDER BY codigo_sh4
        )
        SELECT
            m."SH4",
            sh.descricao_sh4                                         AS sh4_description,
            sh.descricao_sh2                                         AS sh2_description,
            m.vl_fob,
            ROUND(100.0 * m.vl_fob / SUM(m.vl_fob) OVER (), 2)     AS pct_of_total
        FROM mun_exp m
        LEFT JOIN ncm_sh_dedup sh ON m."SH4" = sh.codigo_sh4
        ORDER BY m.vl_fob DESC
        LIMIT 10;
    """

    query_mun_partners = f"""
        WITH mun_partners AS (
            SELECT
                em."CO_PAIS",
                SUM(em."VL_FOB") AS vl_fob
            FROM exp_mun em
            WHERE em."CO_MUN" = {mun_code}
              AND em."CO_ANO" = {MAX_YEAR}
            GROUP BY em."CO_PAIS"
        )
        SELECT
            m."CO_PAIS",
            p.nome_pais                                              AS partner_country,
            m.vl_fob,
            ROUND(100.0 * m.vl_fob / SUM(m.vl_fob) OVER (), 2)     AS pct_of_total
        FROM mun_partners m
        LEFT JOIN pais p ON m."CO_PAIS" = p.codigo_pais
        ORDER BY m.vl_fob DESC
        LIMIT 10;
    """

    df_sh4   = pd.read_sql(query_mun_sh4, engine)
    df_parts = pd.read_sql(query_mun_partners, engine)

    df_sh4['vl_fob_bn']        = df_sh4['vl_fob'] / 1e9
    df_sh4['cumulative_pct']   = df_sh4['pct_of_total'].cumsum().round(1)
    df_parts['vl_fob_bn']      = df_parts['vl_fob'] / 1e9
    df_parts['cumulative_pct'] = df_parts['pct_of_total'].cumsum().round(1)

    mun_total    = df_sh4['vl_fob'].sum()
    china_pct    = df_parts[df_parts['CO_PAIS'] == CHINA_CODE]['pct_of_total'].sum()
    top_product  = df_sh4.iloc[0]['sh4_description'] if len(df_sh4) > 0 else 'N/A'
    top_prod_pct = df_sh4.iloc[0]['pct_of_total']    if len(df_sh4) > 0 else 0

    ## ── SH4 profile table ────────────────────────────────────────────────────
    df_sh4_display = df_sh4[[
        'SH4', 'sh4_description', 'sh2_description', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
    ]].rename(columns={
        'SH4'            : 'SH4',
        'sh4_description': 'Product (SH4)',
        'sh2_description': 'Sector (SH2)',
        'vl_fob_bn'      : 'Exports (USD bn)',
        'pct_of_total'   : 'Share (%)',
        'cumulative_pct' : 'Cumulative (%)',
    }).copy()

    df_sh4_display.insert(0, 'Rank', range(1, len(df_sh4_display) + 1))

    df_sh4_display['Exports (USD bn)'] = df_sh4_display['Exports (USD bn)'].apply(
        lambda x: f'${x:.3f}bn'
    )
    df_sh4_display['Share (%)'] = df_sh4_display['Share (%)'].apply(
        lambda x: f'{x:.1f}%'
    )
    df_sh4_display['Cumulative (%)'] = df_sh4_display['Cumulative (%)'].apply(
        lambda x: f'{x:.1f}%'
    )

    display(
        df_sh4_display.style
        .set_caption(
            f'{mun_name} (ES) — SH4 Export Profile, {MAX_YEAR} '
            f'| CO_MUN {mun_code} '
            f'| Total: ${mun_total/1e9:.2f}bn '
            f'| Dominant: {top_product} ({top_prod_pct:.1f}%)'
        )
        .set_properties(**BASE_PROPS)
        .map(colour_share,      subset=['Share (%)'])
        .map(colour_cumulative, subset=['Cumulative (%)'])
        .set_table_styles(TABLE_STYLES + [
            {'selector': 'tbody td:nth-child(1)', 'props': [
                ('text-align', 'center'),
                ('font-weight', '600'),
                ('color', GREY_TEXT),
                ('font-size', '12px'),
            ]},
            {'selector': 'tbody td:nth-child(2)', 'props': [
                ('font-family', 'monospace'),
                ('font-size', '12px'),
                ('color', GREY_TEXT),
                ('text-align', 'center'),
            ]},
            {'selector': 'tbody td:nth-child(3)', 'props': [
                ('font-weight', '500'),
                ('color', DARK_BLUE),
            ]},
            {'selector': 'tbody td:nth-child(4)', 'props': [
                ('color', GREY_TEXT),
                ('font-size', '12px'),
            ]},
            {'selector': 'tbody td:nth-child(5)', 'props': [
                ('text-align', 'right'),
                ('font-variant-numeric', 'tabular-nums'),
            ]},
            {'selector': 'tbody td:nth-child(6)', 'props': [
                ('text-align', 'right'),
                ('font-variant-numeric', 'tabular-nums'),
            ]},
            {'selector': 'tbody td:nth-child(7)', 'props': [
                ('text-align', 'right'),
                ('font-variant-numeric', 'tabular-nums'),
                ('border-left', f'1px solid {LIGHT_BLUE}'),
            ]},
        ])
        .hide(axis='index')
    )

    ## ── Partner table ─────────────────────────────────────────────────────────
    df_parts_display = df_parts[[
        'partner_country', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
    ]].rename(columns={
        'partner_country': 'Destination Market',
        'vl_fob_bn'      : 'Exports (USD bn)',
        'pct_of_total'   : 'Share (%)',
        'cumulative_pct' : 'Cumulative (%)',
    }).copy()

    df_parts_display.insert(0, 'Rank', range(1, len(df_parts_display) + 1))

    df_parts_display['Exports (USD bn)'] = df_parts_display['Exports (USD bn)'].apply(
        lambda x: f'${x:.3f}bn'
    )
    df_parts_display['Share (%)'] = df_parts_display['Share (%)'].apply(
        lambda x: f'{x:.1f}%'
    )
    df_parts_display['Cumulative (%)'] = df_parts_display['Cumulative (%)'].apply(
        lambda x: f'{x:.1f}%'
    )

    def highlight_china_parts(row):
        if df_parts.loc[row.name, 'CO_PAIS'] == CHINA_CODE:
            return [f'background-color: #fdf0ee; font-weight: 600;'] * len(row)
        return [''] * len(row)

    display(
        df_parts_display.style
        .set_caption(
            f'{mun_name} (ES) — Export Destination Markets, {MAX_YEAR} '
            f'| Total: ${mun_total/1e9:.2f}bn '
            f'| China share: {china_pct:.1f}% '
            f'| China row highlighted'
        )
        .set_properties(**BASE_PROPS)
        .apply(highlight_china_parts, axis=1)
        .map(colour_share,            subset=['Share (%)'])
        .map(colour_cumulative,       subset=['Cumulative (%)'])
        .set_table_styles(TABLE_STYLES + [
            {'selector': 'tbody td:nth-child(1)', 'props': [
                ('text-align', 'center'),
                ('font-weight', '600'),
                ('color', GREY_TEXT),
                ('font-size', '12px'),
            ]},
            {'selector': 'tbody td:nth-child(2)', 'props': [
                ('font-weight', '500'),
                ('color', DARK_BLUE),
            ]},
            {'selector': 'tbody td:nth-child(3)', 'props': [
                ('text-align', 'right'),
                ('font-variant-numeric', 'tabular-nums'),
            ]},
            {'selector': 'tbody td:nth-child(4)', 'props': [
                ('text-align', 'right'),
                ('font-variant-numeric', 'tabular-nums'),
            ]},
            {'selector': 'tbody td:nth-child(5)', 'props': [
                ('text-align', 'right'),
                ('font-variant-numeric', 'tabular-nums'),
                ('border-left', f'1px solid {LIGHT_BLUE}'),
            ]},
        ])
        .hide(axis='index')
    )

Rank,SH4,Product (SH4),Sector (SH2),Exports (USD bn),Share (%),Cumulative (%)
1,6802,"Pedras de cantaria ou de construção (exceto de ardósia) trabalhadas e obras destas pedras, exceto as da posição 6801; cubos, pastilhas e artigos semelhantes, para mosaicos, de pedra natural (incluída a ardósia), mesmo com suporte; grânulos, fragmentos e","Obras de pedra, gesso, cimento, amianto, mica ou de matérias semelhantes",$0.110bn,70.8%,70.8%
2,2516,"Granito, pórfiro, basalto, arenito e outras pedras de cantaria ou de construção, mesmo desbastados ou simplesmente cortados à serra ou por outro meio, em blocos ou placas de forma quadrada ou rectangular","Sal; enxofre; terras e pedras; gesso, cal e cimento",$0.042bn,27.1%,97.9%
3,2506,"Quartzo (exceto areias naturais); quartzites, mesmo desbastadas ou simplesmente cortadas à serra ou por outro meio, em blocos ou placas de forma quadrada ou rectangular","Sal; enxofre; terras e pedras; gesso, cal e cimento",$0.003bn,2.0%,99.9%
4,2529,Feldspato; leucite; nefelina e nefelina-sienite; espatoflúor,"Sal; enxofre; terras e pedras; gesso, cal e cimento",$0.000bn,0.0%,99.9%
5,2515,"Mármores, travertinos, granitos belgas e outras pedras calcárias de cantaria ou de construção, de densidade aparente igual ou superior a 2,5, e alabastro, mesmo desbastados ou simplesmente cortados à serra ou por outro meio, em blocos ou placas de forma q","Sal; enxofre; terras e pedras; gesso, cal e cimento",$0.000bn,0.0%,100.0%
6,2849,Carbonetos de constituição química definida ou não,"Produtos químicos inorgânicos; compostos inorgânicos ou orgânicos de metais preciosos, de elementos radioativos, de metais das terras raras ou de isótopos",$0.000bn,0.0%,100.0%
7,6804,"Mós e artefactos semelhantes, sem armação, para moer, desfibrar, triturar, amolar, polir, rectificar ou cortar; pedras para amolar ou para polir, manualmente, e suas partes, de pedras naturais, de abrasivos naturais ou artificiais aglomerados ou de cerâmi","Obras de pedra, gesso, cimento, amianto, mica ou de matérias semelhantes",$0.000bn,0.0%,100.0%
8,7103,"Pedras preciosas (exceto diamantes) ou semipreciosas, mesmo trabalhadas ou combinadas, mas não enfiadas, nem montadas, nem engastadas; pedras preciosas (exceto diamantes) ou semipreciosas, não combinadas, enfiadas temporariamente para facilidade de tran","Pérolas naturais ou cultivadas, pedras preciosas ou semipreciosas e semelhantes, metais preciosos, metais folheados ou chapeados de metais preciosos (plaquê), e suas obras; bijuterias; moedas",$0.000bn,0.0%,100.0%
9,4014,"Artigos de higiene ou de farmácia (incluídas as chupetas), de borracha vulcanizada não endurecida, mesmo com partes de borracha endurecida",Borracha e suas obras,$0.000bn,0.0%,100.0%
10,8431,Partes reconhecíveis como exclusiva ou principalmente destinadas às máquinas e aparelhos das posições 8425 a 8430,"Reatores nucleares, caldeiras, máquinas, aparelhos e instrumentos mecânicos, e suas partes",$0.000bn,0.0%,100.0%


Rank,Destination Market,Exports (USD bn),Share (%),Cumulative (%)
1,Estados Unidos,$0.092bn,59.2%,59.2%
2,China,$0.042bn,27.2%,86.4%
3,México,$0.006bn,4.0%,90.4%
4,Itália,$0.003bn,1.6%,92.0%
5,Espanha,$0.002bn,1.3%,93.2%
6,Taiwan (Formosa),$0.002bn,1.0%,94.3%
7,República Dominicana,$0.001bn,0.6%,94.8%
8,Canadá,$0.001bn,0.6%,95.4%
9,Austrália,$0.001bn,0.5%,95.8%
10,Índia,$0.001bn,0.5%,96.3%


Rank,SH4,Product (SH4),Sector (SH2),Exports (USD bn),Share (%),Cumulative (%)
1,4703,"Pastas químicas de madeira, à soda ou ao sulfato, exceto pastas para dissolução",Pastas de madeira ou de outras matérias fibrosas celulósicas; papel ou cartão para reciclar (desperdícios e aparas).,$0.863bn,55.2%,55.2%
2,8479,"Máquinas e aparelhos, mecânicos, com função própria, não especificados nem compreendidos em outras posições deste capítulo","Reatores nucleares, caldeiras, máquinas, aparelhos e instrumentos mecânicos, e suas partes",$0.357bn,22.8%,78.0%
3,8421,"Centrifugadores, incluídos os secadores centrífugos, aparelhos para filtrar ou depurar líquidos ou gases","Reatores nucleares, caldeiras, máquinas, aparelhos e instrumentos mecânicos, e suas partes",$0.162bn,10.4%,88.4%
4,2709,Óleos brutos de petróleo ou de minerais betuminosos,"Combustíveis minerais, óleos minerais e produtos da sua destilação; matérias betuminosas; ceras minerais",$0.129bn,8.3%,96.7%
5,904,"Pimenta (do género Piper); pimentos dos géneros Capsicum ou Pimenta, secos ou triturados ou em pó","Café, chá, mate e especiarias",$0.023bn,1.5%,98.1%
6,8481,"Torneiras, válvulas (incluídas as redutoras de pressão e as termostáticas) e dispositivos semelhantes, para canalizações, caldeiras, reservatórios, cubas e outros recipientes","Reatores nucleares, caldeiras, máquinas, aparelhos e instrumentos mecânicos, e suas partes",$0.014bn,0.9%,99.0%
7,3917,"Tubos e seus acessórios (por exemplo: juntas, cotovelos, flanges, uniões), de plástico",Plásticos e suas obras,$0.003bn,0.2%,99.2%
8,7307,"Acessórios para tubos [por exemplo: uniões, cotovelos, mangas (luvas)], de ferro fundido, ferro ou aço","Obras de ferro fundido, ferro ou aço",$0.002bn,0.1%,99.4%
9,9026,"Instrumentos e aparelhos para medida ou controlo do caudal (vazão), do nível, da pressão ou de outras características variáveis dos líquidos ou gases (por exemplo: medidores de caudal, indicadores de nível, manómetros, contadores de calor), exceto os ins","Instrumentos e aparelhos de óptica, de fotografia, de cinematografia, de medida, de controle ou de precisão; instrumentos e aparelhos médico-cirúrgicos; suas partes e acessórios",$0.002bn,0.1%,99.4%
10,2847,"Peróxido de hidrogênio (água oxigenada), mesmo solidificado com ureia","Produtos químicos inorgânicos; compostos inorgânicos ou orgânicos de metais preciosos, de elementos radioativos, de metais das terras raras ou de isótopos",$0.001bn,0.1%,99.5%


Rank,Destination Market,Exports (USD bn),Share (%),Cumulative (%)
1,Singapura,$0.543bn,34.8%,34.8%
2,Estados Unidos,$0.466bn,29.8%,64.6%
3,China,$0.153bn,9.8%,74.4%
4,Países Baixos (Holanda),$0.131bn,8.3%,82.7%
5,Coreia do Sul,$0.068bn,4.4%,87.1%
6,Taiwan (Formosa),$0.067bn,4.3%,91.4%
7,Turquia,$0.048bn,3.1%,94.4%
8,El Salvador,$0.038bn,2.4%,96.9%
9,Egito,$0.011bn,0.7%,97.6%
10,Itália,$0.008bn,0.5%,98.1%


#### Overview

Anchieta and Aracruz present two structurally distinct export profiles within Espírito Santo, separated by sector, scale, and destination market geography.

**Anchieta ($0.16bn): natural stone exports concentrated in two products and two markets**

The export profile is a natural stone cluster across three SH4 codes. Worked construction and monumental stone (6802) leads at \\$0.110bn (70.8%), followed by granite and construction stone blocks (2516) at \\$0.042bn (27.1%), and quartzite (2506) at \\$0.003bn (2.0%). The remaining seven SH4 codes register negligible values. The two leading codes together account for 97.9% of total exports, confirming a narrow single-sector profile. The destination market is equally concentrated: the United States at \\$0.092bn (59.2%) and China at \\$0.042bn (27.2%) account for 86.4% of the total, with Mexico (4.0%), Italy (1.6%), and Spain (1.3%) as minor secondary markets.

**Aracruz ($1.56bn): chemical wood pulp dominant but with a materially more complex secondary profile than previously indicated**

Chemical wood pulp (4703) leads at \\$0.863bn (55.2%), consistent with Aracruz as a registered location for large-scale eucalyptus pulp operations in Espírito Santo. The corrected table reveals a significant secondary layer not visible in the duplicated output. Mechanical apparatus (8479) registers \\$0.357bn (22.8%) — a substantial figure that warrants NCM-level verification to confirm whether this reflects industrial equipment exports, maintenance-related flows, or a specific registered export activity. Centrifuges and filtration apparatus (8421) add \\$0.162bn (10.4%), and crude petroleum oils (2709) register \\$0.129bn (8.3%). The petroleum entry at rank 4 is unexpected for a municipality whose primary export profile is pulp-related and requires entity-level investigation. Pepper (SH4 904, \\$0.023bn, 1.5%) at rank 5 is a minor but notable agricultural entry.

The destination market for Aracruz is led by Singapore at \\$0.543bn (34.8%), followed by the United States at \\$0.466bn (29.8%). As noted in prior sections, Singapore at 34.8% of a commodity export total is consistent with trading intermediation rather than domestic consumption. China registers \\$0.153bn (9.8%) at rank 3. The Netherlands (\\$0.131bn, 8.3%), South Korea (\\$0.068bn, 4.4%), and Taiwan (\\$0.067bn, 4.3%) follow. El Salvador at \\$0.038bn (2.4%) remains atypical for pulp exports and warrants entity-level verification.

The corrected scale difference between the two municipalities narrows considerably. Aracruz at \\$1.56bn is now 9.8 times the size of Anchieta at $0.16bn, compared to the 5.4x ratio in the duplicated output.

#### Business Relevance

The pulp export (SH4 4703, \\$0.863bn) is the only line item in Aracruz's profile that can be commercially characterised at the SH4 level. The SH4 8479 machinery entry (\\$0.357bn, 22.8%) and the crude petroleum entry (2709, \\$0.129bn, 8.3%) require NCM-level verification before any commercial interpretation is appropriate. Until those two entries are resolved, Aracruz cannot be described as a pure pulp municipality, and the scale of the uncharacterised flows is too large to treat as residual.

---
---


### 4.7.2 — Marabá (Pará): Iron Ore China Share Investigation

**Open flag:** Marabá registered a 9.9% China export share in Step 3 — significantly below the 75.2% at Parauapebas and 63.2% at Canaã dos Carajás, despite all three being Pará iron ore municipalities. The anomaly may reflect a processed steel product component in Marabá's export mix (SH72 steel) that routes to different markets from raw iron ore (SH26).

**CO_MUN:** 1504208


**Output — SH4 + Partner Breakdown with China Share Confirmation:** The SH4 profile distinguishes iron ore (SH26) from processed steel (SH72). If Marabá's mix includes SH72, the lower China share (9.9% vs 63–75% at Parauapebas/Canaã dos Carajás) is explained by product routing differences — processed steel markets differ from raw ore markets. The partner breakdown identifies where non-China exports route.

In [4]:
## Marabá: SH4 export profile and China share
## CO_MUN for Marabá (PA): 1504208

query_maraba_sh4 = f"""
    WITH maraba_exp AS (
        SELECT
            em."SH4",
            em."CO_PAIS",
            SUM(em."VL_FOB")     AS vl_fob,
            SUM(em."KG_LIQUIDO") AS kg_liq
        FROM exp_mun em
        WHERE em."CO_MUN" = 1504208
          AND em."CO_ANO" = {MAX_YEAR}
        GROUP BY em."SH4", em."CO_PAIS"
    ),
    ncm_sh_dedup AS (
        SELECT DISTINCT ON (codigo_sh4)
            codigo_sh4,
            descricao_sh4,
            descricao_sh2,
            codigo_sh2
        FROM ncm_sh
        ORDER BY codigo_sh4
    )
    SELECT
        m."SH4",
        sh.descricao_sh4   AS sh4_description,
        sh.descricao_sh2   AS sh2_description,
        sh.codigo_sh2,
        m."CO_PAIS",
        p.nome_pais        AS partner_country,
        m.vl_fob,
        m.kg_liq,
        ROUND(100.0 * m.vl_fob / SUM(m.vl_fob) OVER (), 2) AS pct_of_total
    FROM maraba_exp m
    LEFT JOIN ncm_sh_dedup sh ON m."SH4" = sh.codigo_sh4
    LEFT JOIN pais p           ON m."CO_PAIS" = p.codigo_pais
    ORDER BY m.vl_fob DESC
    LIMIT 20;
"""

df_maraba = pd.read_sql(query_maraba_sh4, engine)
df_maraba['vl_fob_bn']      = df_maraba['vl_fob'] / 1e9
df_maraba['cumulative_pct'] = df_maraba['pct_of_total'].cumsum().round(1)

maraba_total     = df_maraba['vl_fob'].sum()
maraba_china     = df_maraba[df_maraba['CO_PAIS'] == CHINA_CODE]['vl_fob'].sum()
maraba_china_pct = 100 * maraba_china / maraba_total if maraba_total > 0 else 0

## SH4-level sector summary
maraba_by_sh4 = (
    df_maraba.groupby(['SH4', 'sh4_description', 'sh2_description', 'codigo_sh2'])['vl_fob']
    .sum()
    .reset_index()
    .sort_values('vl_fob', ascending=False)
    .reset_index(drop=True)
)
maraba_by_sh4['pct'] = (maraba_by_sh4['vl_fob'] / maraba_total * 100).round(1)

## SH2 summary to detect steel vs ore split
maraba_by_sh2 = (
    df_maraba.groupby(['codigo_sh2', 'sh2_description'])['vl_fob']
    .sum()
    .reset_index()
    .sort_values('vl_fob', ascending=False)
)
sh26_pct = maraba_by_sh2[maraba_by_sh2['codigo_sh2'] == 26]['vl_fob'].sum() / maraba_total * 100
sh72_pct = maraba_by_sh2[maraba_by_sh2['codigo_sh2'] == 72]['vl_fob'].sum() / maraba_total * 100

## Styled tables: Marabá SH4 sector summary + full SH4/partner detail

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

SHARE_DOM  = 30.0
SHARE_HIGH = 10.0

TABLE_STYLES_BASE = [
    {'selector': 'caption', 'props': [
        ('caption-side', 'top'),
        ('font-size', '14px'),
        ('font-weight', '700'),
        ('font-family', 'Helvetica Neue, Arial, sans-serif'),
        ('color', DARK_BLUE),
        ('padding', '0 0 10px 0'),
        ('text-align', 'left'),
    ]},
    {'selector': 'thead th', 'props': [
        ('background-color', DARK_BLUE),
        ('color', WHITE),
        ('font-size', '12px'),
        ('font-weight', '600'),
        ('text-transform', 'uppercase'),
        ('letter-spacing', '0.04em'),
        ('padding', '10px 14px'),
        ('border', 'none'),
    ]},
    {'selector': 'thead tr:last-child th', 'props': [
        ('border-bottom', f'2px solid {MID_BLUE}'),
    ]},
    {'selector': 'tbody tr:nth-child(odd)', 'props': [
        ('background-color', WHITE),
    ]},
    {'selector': 'tbody tr:nth-child(even)', 'props': [
        ('background-color', STRIPE),
    ]},
    {'selector': 'tbody tr:hover', 'props': [
        ('background-color', LIGHT_BLUE),
    ]},
    {'selector': 'table', 'props': [
        ('border-collapse', 'collapse'),
        ('width', '100%'),
        ('border', f'1px solid {LIGHT_BLUE}'),
        ('border-radius', '6px'),
        ('overflow', 'hidden'),
        ('margin-bottom', '28px'),
    ]},
]

BASE_PROPS = {
    'font-family'  : 'Helvetica Neue, Arial, sans-serif',
    'font-size'    : '13px',
    'text-align'   : 'left',
    'padding'      : '8px 14px',
    'border-bottom': f'1px solid {LIGHT_BLUE}',
}

def colour_share(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= SHARE_DOM:
        return f'color: {RED}; font-weight: 700;'
    elif v >= SHARE_HIGH:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

def colour_china(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 50.0:
        return f'color: {RED}; font-weight: 700;'
    elif v >= 25.0:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREEN}; font-weight: 500;'

def colour_cumulative(val):
    try:
        v = float(val.replace('%', ''))
    except (ValueError, AttributeError):
        return ''
    if v >= 80.0:
        return f'color: {RED}; font-weight: 600;'
    elif v >= 50.0:
        return f'color: {AMBER}; font-weight: 500;'
    return f'color: {DARK_BLUE};'

def highlight_sector_rows(row):
    sh2 = maraba_by_sh4.loc[row.name, 'codigo_sh2']
    if sh2 == 26:
        return ['background-color: #fff8e6;'] * len(row)
    elif sh2 == 72:
        return ['background-color: #e8f0fb;'] * len(row)
    return [''] * len(row)

def highlight_china_rows(row):
    if df_maraba.loc[row.name, 'CO_PAIS'] == CHINA_CODE:
        return ['background-color: #fdf0ee; font-weight: 600;'] * len(row)
    return [''] * len(row)

## ── Table 1: SH4 sector summary ──────────────────────────────────────────────
df_sh4_summary = maraba_by_sh4.copy()
df_sh4_summary['vl_fob_bn'] = df_sh4_summary['vl_fob'] / 1e9
df_sh4_summary['China (%)'] = df_sh4_summary['SH4'].apply(
    lambda sh4: round(
        df_maraba[(df_maraba['SH4'] == sh4) & (df_maraba['CO_PAIS'] == CHINA_CODE)]['vl_fob'].sum()
        / max(df_maraba[df_maraba['SH4'] == sh4]['vl_fob'].sum(), 1) * 100, 1
    )
)

df_sh4_summary_display = df_sh4_summary[[
    'SH4', 'sh4_description', 'sh2_description', 'vl_fob_bn', 'pct', 'China (%)'
]].rename(columns={
    'SH4'            : 'SH4',
    'sh4_description': 'Product (SH4)',
    'sh2_description': 'Sector (SH2)',
    'vl_fob_bn'      : 'Exports (USD bn)',
    'pct'            : 'Share (%)',
    'China (%)'      : 'China (%)',
}).copy()

df_sh4_summary_display.insert(0, 'Rank', range(1, len(df_sh4_summary_display) + 1))
df_sh4_summary_display['Exports (USD bn)'] = df_sh4_summary_display['Exports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_sh4_summary_display['Share (%)']  = df_sh4_summary_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_sh4_summary_display['China (%)']  = df_sh4_summary_display['China (%)'].apply(
    lambda x: f'{x:.1f}%'
)

copper_pct = (
    df_maraba[df_maraba['SH4'] == 2603]['vl_fob'].sum()
    / maraba_total * 100
)

hypothesis_str = (
    'Copper ore dominant — European smelter routing explains low China share'
    if copper_pct >= 50
    else 'Steel (SH72) present — explains lower China routing'
    if sh72_pct >= 10
    else 'Ore-dominant — China routing anomaly unresolved'
)

display(
    df_sh4_summary_display.style
    .set_caption(
        f'Marabá (PA) — SH4 Export Sector Summary, {MAX_YEAR} '
        f'| CO_MUN 1504208 '
        f'| Total: ${maraba_total/1e9:.2f}bn '
        f'| China: {maraba_china_pct:.1f}% (Step 3 ref: 9.9%) '
        f'| {hypothesis_str}'
    )
    .set_properties(**BASE_PROPS)
    .apply(highlight_sector_rows, axis=1)
    .map(colour_share, subset=['Share (%)'])
    .map(colour_china, subset=['China (%)'])
    .set_table_styles(TABLE_STYLES_BASE + [
        {'selector': 'tbody td:nth-child(1)', 'props': [
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
    ])
    .hide(axis='index')
)

## ── Table 2: full SH4 + partner detail ───────────────────────────────────────
df_maraba_display = df_maraba[[
    'SH4', 'sh4_description', 'partner_country', 'vl_fob_bn', 'pct_of_total', 'cumulative_pct'
]].rename(columns={
    'SH4'            : 'SH4',
    'sh4_description': 'Product (SH4)',
    'partner_country': 'Destination',
    'vl_fob_bn'      : 'Exports (USD bn)',
    'pct_of_total'   : 'Share (%)',
    'cumulative_pct' : 'Cumulative (%)',
}).copy()

df_maraba_display.insert(0, 'Rank', range(1, len(df_maraba_display) + 1))

df_maraba_display['Exports (USD bn)'] = df_maraba_display['Exports (USD bn)'].apply(
    lambda x: f'${x:.3f}bn'
)
df_maraba_display['Share (%)'] = df_maraba_display['Share (%)'].apply(
    lambda x: f'{x:.1f}%'
)
df_maraba_display['Cumulative (%)'] = df_maraba_display['Cumulative (%)'].apply(
    lambda x: f'{x:.1f}%'
)

display(
    df_maraba_display.style
    .set_caption(
        f'Marabá (PA) — SH4 Export Detail by Partner, {MAX_YEAR} '
        f'| Top 20 SH4 × partner rows | China rows highlighted'
    )
    .set_properties(**BASE_PROPS)
    .apply(highlight_china_rows, axis=1)
    .map(colour_share,           subset=['Share (%)'])
    .map(colour_cumulative,      subset=['Cumulative (%)'])
    .set_table_styles(TABLE_STYLES_BASE + [
        {'selector': 'tbody td:nth-child(1)', 'props': [
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [
            ('font-family', 'monospace'),
            ('font-size', '12px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'1px solid {LIGHT_BLUE}'),
        ]},
    ])
    .hide(axis='index')
)

Rank,SH4,Product (SH4),Sector (SH2),Exports (USD bn),Share (%),China (%)
1,2603,Minérios de cobre e seus concentrados,"Minerios, escórias e cinzas",$2.833bn,89.6%,7.8%
2,7201,"Ferro fundido bruto e ferro spiegel (especular), em lingotes, linguados ou outras formas primárias","Ferro fundido, ferro e aço",$0.182bn,5.8%,0.0%
3,202,"Carnes de animais da espécie bovina, congeladas","Carnes e miudezas, comestíveis",$0.133bn,4.2%,66.5%
4,201,"Carnes de animais da espécie bovina, frescas ou refrigeradas","Carnes e miudezas, comestíveis",$0.007bn,0.2%,0.0%
5,2601,"Minérios de ferro e seus concentrados, incluídas as pirites de ferro ustuladas (cinzas de pirites)","Minerios, escórias e cinzas",$0.006bn,0.2%,100.0%
6,504,"Tripas, bexigas e estômagos de animais, exceto peixes, inteiros ou em pedaços, frescos, refrigerados, congelados, salgados, secos ou defumados","Outros produtos de origem animal, não especificados nem compreendidos noutros Capítulos",$0.002bn,0.1%,0.0%


Rank,SH4,Product (SH4),Destination,Exports (USD bn),Share (%),Cumulative (%)
1,2603,Minérios de cobre e seus concentrados,Alemanha,$0.546bn,17.2%,17.2%
2,2603,Minérios de cobre e seus concentrados,Polônia,$0.500bn,15.8%,33.0%
3,2603,Minérios de cobre e seus concentrados,Bulgária,$0.451bn,14.2%,47.2%
4,2603,Minérios de cobre e seus concentrados,Malásia,$0.301bn,9.5%,56.7%
5,2603,Minérios de cobre e seus concentrados,Suécia,$0.278bn,8.8%,65.5%
6,2603,Minérios de cobre e seus concentrados,China,$0.220bn,6.9%,72.4%
7,2603,Minérios de cobre e seus concentrados,Taiwan (Formosa),$0.211bn,6.6%,79.1%
8,2603,Minérios de cobre e seus concentrados,Índia,$0.163bn,5.2%,84.2%
9,7201,"Ferro fundido bruto e ferro spiegel (especular), em lingotes, linguados ou outras formas primárias",Estados Unidos,$0.150bn,4.7%,89.0%
10,2603,Minérios de cobre e seus concentrados,Espanha,$0.113bn,3.6%,92.5%


#### Overview

Marabá's \\$3.16bn export register is a copper ore dominated profile with two small secondary sectors. Copper ore (2603) accounts for \\$2.833bn (89.6%), pig iron (7201) for \\$0.182bn (5.8%), and frozen beef (202) for \\$0.133bn (4.2%). Minor entries — fresh beef (201, 0.2%), iron ore (2601, 0.2%), and animal offal (504, 0.1%) — contribute the remaining 0.7%. The top three sectors account for 99.6% of total registered exports.

**Copper ore routes overwhelmingly to European smelters**. Germany (\\$0.546bn, 17.2%), Poland (\\$0.500bn, 15.8%), Bulgaria (\\$0.451bn, 14.2%), Sweden (\\$0.278bn, 8.8%), Spain (\\$0.113bn, 3.6%), and Finland (\\$0.050bn, 1.6%) collectively account for \\$1.938bn across six European destinations. Malaysia (\\$0.301bn, 9.5%), Taiwan (\\$0.211bn, 6.6%), and India (\\$0.163bn, 5.2%) represent the Asian routing. China receives \\$0.220bn (6.9% of total exports, 7.8% of the copper ore sector specifically) — a notably low share for a commodity where China is typically the dominant global buyer.

**Pig iron (7201) routes to the United States (\\$0.150bn) and Italy (\\$0.032bn)**. The corrected total of \\$0.182bn is materially lower than the \\$0.547bn in the duplicated output. The United States and Italy are the only pig iron destinations appearing in the top 20, with the remainder of the \\$0.182bn distributed across partners outside the ranking.

**Frozen beef (202, $0.133bn) routes primarily to China**. China receives \\$0.089bn, representing 66.5% of the sector total as recorded in the summary table. Israel (\\$0.031bn), Indonesia (\\$0.008bn), Philippines (\\$0.004bn), and Saudi Arabia (\\$0.001bn) account for the remainder in the top 20. The minor iron ore entry (2601, \\$0.006bn) also routes entirely to China at 100.0%.

**The caption classification "Ore-dominant — China routing anomaly unresolved" requires revision**. The corrected data confirms that Marabá is copper ore dominant at 89.6%, not iron ore dominant. The low China share of 9.9% is explained by the copper ore destination structure: European smelters absorb the majority of global copper concentrate supply, and Marabá's routing reflects this market structure rather than any anomaly. The flag should be updated to read "Copper ore dominant — European smelter routing explains low China share."

The Step 3 China reference figure of 9.9% now matches exactly, confirming the corrected query produces consistent results with the prior step.

#### Business Relevance

The corrected profile establishes Marabá as a copper ore municipality with a modest beef export base, not the larger and more diversified register suggested by the duplicated output. The copper ore routing to European smelters means Marabá's export flows operate on Atlantic trade lanes rather than the Pacific routes that dominate Brazilian iron ore and soy logistics. The pig iron figure at \\$0.182bn is too small to draw commercial conclusions beyond confirming the presence of a registered iron production and export activity. The beef sector at \\$0.133bn with 66.5% China routing is consistent with the national beef export pattern but at a scale that reflects a small number of registered exporters rather than a significant regional beef processing cluster. Entity-level investigation would be required to identify the specific counterparties behind the copper ore flows given the concentration of value across a small number of European smelter destinations.

---
---


### 4.7.3 — Post-2015 Deficit Convergence: Municipality-Level Time Series

**Open flag:** Multiple municipalities entered deficit positions in the 2015–2025 window, parallel to the post-2015 regression weakening in Step 2c (Regression 1: frequency vs HHI weakened from avg slope ~-0.138 in 1997–2015 to ~-0.089 in 2016–2025). Whether this convergence is systematic — a structural feature of the 2015–2025 period — or coincidental requires a time-series analysis of deficit entry dates.


**Step 1 — Top 20 Deficit Municipalities (2025):** Identifies the universe of municipalities for the time-series analysis. These are the municipalities with the largest negative trade balances in the final year — the cohort most relevant for detecting a systematic structural pattern.

In [90]:
## Step 1: Identify top 20 deficit municipalities in MAX_YEAR

query_top_deficit = f"""
    WITH exp_tot AS (
        SELECT "CO_MUN", SUM("VL_FOB") AS exp_fob
        FROM exp_mun
        WHERE "CO_ANO" = {MAX_YEAR}
        GROUP BY "CO_MUN"
    ),
    imp_tot AS (
        SELECT "CO_MUN", SUM("VL_FOB") AS imp_fob
        FROM imp_mun
        WHERE "CO_ANO" = {MAX_YEAR}
        GROUP BY "CO_MUN"
    )
    SELECT
        COALESCE(e."CO_MUN", m."CO_MUN")                  AS co_mun,
        COALESCE(e.exp_fob, 0)                             AS exp_fob,
        COALESCE(m.imp_fob, 0)                             AS imp_fob,
        COALESCE(e.exp_fob, 0) - COALESCE(m.imp_fob, 0)  AS balance
    FROM exp_tot e
    FULL OUTER JOIN imp_tot m ON e."CO_MUN" = m."CO_MUN"
    ORDER BY balance ASC
    LIMIT 20;
"""

df_top_deficit = pd.read_sql(query_top_deficit, engine)
df_top_deficit['exp_fob_bn'] = df_top_deficit['exp_fob'] / 1e9
df_top_deficit['imp_fob_bn'] = df_top_deficit['imp_fob'] / 1e9
df_top_deficit['balance_bn'] = df_top_deficit['balance'] / 1e9
top_muns = df_top_deficit['co_mun'].tolist()

## Fetch municipality names from uf_mun
query_mun_names = f"""
    SELECT
        codigo_municipio AS co_mun,
        nome_municipio   AS mun_name,
        sigla_uf         AS sg_uf
    FROM uf_mun
    WHERE codigo_municipio IN ({','.join(str(m) for m in top_muns)});
"""
df_mun_names = pd.read_sql(query_mun_names, engine)
df_top_deficit = df_top_deficit.merge(df_mun_names, on='co_mun', how='left')

#print(f'Top 20 deficit municipalities, {MAX_YEAR}')
#print(f'Combined deficit: ${df_top_deficit["balance_bn"].sum():.2f}bn')

## Styled table: top 20 deficit municipalities

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

FLAGGED_MUNS = {
    3304557: 'Petrópolis ⚠️',
    4208203: 'Itajaí ⚠️',
    3201308: 'Cariacica ⚠️',
}

df_deficit_display = df_top_deficit[[
    'co_mun', 'mun_name', 'sg_uf', 'exp_fob_bn', 'imp_fob_bn', 'balance_bn'
]].rename(columns={
    'co_mun'    : 'CO_MUN',
    'mun_name'  : 'Municipality',
    'sg_uf'     : 'UF',
    'exp_fob_bn': 'Exports (USD bn)',
    'imp_fob_bn': 'Imports (USD bn)',
    'balance_bn': 'Balance (USD bn)',
}).copy()

df_deficit_display.insert(0, 'Rank', range(1, len(df_deficit_display) + 1))

df_deficit_display['CO_MUN'] = df_deficit_display['CO_MUN'].apply(
    lambda x: f'{int(x)}'
)
df_deficit_display['Exports (USD bn)'] = df_deficit_display['Exports (USD bn)'].apply(
    lambda x: f'${x:.2f}bn'
)
df_deficit_display['Imports (USD bn)'] = df_deficit_display['Imports (USD bn)'].apply(
    lambda x: f'${x:.2f}bn'
)
df_deficit_display['Balance (USD bn)'] = df_deficit_display['Balance (USD bn)'].apply(
    lambda x: f'-${abs(x):.2f}bn' if x < 0 else f'+${x:.2f}bn'
)

df_deficit_display['Flag'] = df_top_deficit['co_mun'].apply(
    lambda x: '⚠️ Open' if x in FLAGGED_MUNS else ''
)

def colour_balance(val):
    try:
        v = abs(float(val.replace('-$', '').replace('+$', '').replace('bn', '')))
    except (ValueError, AttributeError):
        return ''
    if v >= 5.0:
        return f'color: {RED}; font-weight: 700;'
    elif v >= 1.0:
        return f'color: {RED}; font-weight: 600;'
    return f'color: {AMBER}; font-weight: 500;'

def colour_flag(val):
    if '⚠️' in str(val):
        return f'color: {AMBER}; font-weight: 700;'
    return ''

def highlight_flagged_rows(row):
    co_mun = int(df_top_deficit.loc[row.name, 'co_mun'])
    if co_mun in FLAGGED_MUNS:
        return [f'background-color: #fff8e6;'] * len(row)
    return [''] * len(row)

combined_deficit = df_top_deficit['balance_bn'].sum()

styler = (
    df_deficit_display.style
    .set_caption(
        f'Top 20 Deficit Municipalities, {MAX_YEAR} '
        f'| Combined deficit: ${combined_deficit:.2f}bn '
        f'| ⚠️ = open investigation from Steps 2–3'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_flagged_rows, axis=1)
    .map(colour_balance,           subset=['Balance (USD bn)'])
    .map(colour_flag,              subset=['Flag'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## CO_MUN
            ('font-family', 'monospace'),
            ('font-size', '11px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Municipality
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## UF
            ('font-family', 'monospace'),
            ('font-weight', '700'),
            ('color', DARK_BLUE),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## Exports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Imports
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [   ## Balance
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
            ('border-left', f'2px solid {LIGHT_BLUE}'),
        ]},
        {'selector': 'tbody td:nth-child(8)', 'props': [   ## Flag
            ('text-align', 'center'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

display(styler)

Rank,CO_MUN,Municipality,UF,Exports (USD bn),Imports (USD bn),Balance (USD bn),Flag
1,1302603,MANAUS,AM,$0.79bn,$15.83bn,-$15.03bn,
2,4208203,ITAJAI,SC,$6.12bn,$16.31bn,-$10.19bn,⚠️ Open
3,3303906,PETROPOLIS,RJ,$0.60bn,$9.47bn,-$8.87bn,
4,3201308,CARIACICA,ES,$0.14bn,$7.17bn,-$7.03bn,⚠️ Open
5,3450308,SAO PAULO,SP,$5.36bn,$10.77bn,-$5.41bn,
6,3436505,PAULINIA,SP,$0.84bn,$6.23bn,-$5.38bn,
7,3425904,JUNDIAI,SP,$0.95bn,$4.30bn,-$3.35bn,
8,4209102,JOINVILLE,SC,$1.30bn,$4.46bn,-$3.16bn,
9,2111300,SAO LUIS,MA,$1.92bn,$4.46bn,-$2.54bn,
10,3125101,EXTREMA,MG,$0.08bn,$2.62bn,-$2.54bn,


**Step 2 — Annual Balance Time Series and Deficit Entry Dates:** Tracks when each municipality first entered deficit. The period distribution (pre-2000, 2001–2010, 2011–2015, 2016–2020, 2021–2025) is the key diagnostic. Concentration in the 2011–2015 or 2016–2020 window confirms the parallel with the Step 2c regression weakening (post-2015 slope shift from ~-0.138 to ~-0.089) is not coincidental.

In [91]:
## Step 2: Annual balance time series for top 20 deficit municipalities

mun_list_sql = ','.join(str(m) for m in top_muns)

query_deficit_ts = f"""
    WITH exp_ts AS (
        SELECT "CO_MUN", "CO_ANO", SUM("VL_FOB") AS exp_fob
        FROM exp_mun
        WHERE "CO_MUN" IN ({mun_list_sql})
        GROUP BY "CO_MUN", "CO_ANO"
    ),
    imp_ts AS (
        SELECT "CO_MUN", "CO_ANO", SUM("VL_FOB") AS imp_fob
        FROM imp_mun
        WHERE "CO_MUN" IN ({mun_list_sql})
        GROUP BY "CO_MUN", "CO_ANO"
    )
    SELECT
        COALESCE(e."CO_MUN", m."CO_MUN")                  AS co_mun,
        COALESCE(e."CO_ANO", m."CO_ANO")                  AS co_ano,
        COALESCE(e.exp_fob, 0)                             AS exp_fob,
        COALESCE(m.imp_fob, 0)                             AS imp_fob,
        COALESCE(e.exp_fob, 0) - COALESCE(m.imp_fob, 0)  AS balance
    FROM exp_ts e
    FULL OUTER JOIN imp_ts m
        ON e."CO_MUN" = m."CO_MUN" AND e."CO_ANO" = m."CO_ANO"
    ORDER BY co_mun, co_ano;
"""

df_deficit_ts = pd.read_sql(query_deficit_ts, engine)

## First deficit year per municipality
first_deficit = (
    df_deficit_ts[df_deficit_ts['balance'] < 0]
    .groupby('co_mun')['co_ano']
    .min()
    .reset_index()
    .rename(columns={'co_ano': 'first_deficit_year'})
    .sort_values('first_deficit_year')
)

## Merge municipality names
first_deficit = first_deficit.merge(
    df_mun_names.rename(columns={'CO_MUN': 'co_mun'}),
    on='co_mun', how='left'
)

## Period classification
bins   = [0, 2000, 2010, 2015, 2020, 2025]
labels = ['Pre-2000', '2001–2010', '2011–2015', '2016–2020', '2021–2025']
first_deficit['period'] = pd.cut(
    first_deficit['first_deficit_year'], bins=bins, labels=labels
)

period_counts = first_deficit['period'].value_counts().sort_index().reset_index()
period_counts.columns = ['Period', 'Count']

#print('First deficit year by municipality (top-20 deficit in 2025):')
#print(first_deficit[['co_mun', 'mun_name', 'sg_uf', 'first_deficit_year', 'period']].to_string(index=False))
#print('\nNew deficit entries by period:')
#print(period_counts.to_string(index=False))

## Styled tables: first deficit year + period distribution

DARK_BLUE  = '#1a2e44'
MID_BLUE   = '#2c4a6e'
LIGHT_BLUE = '#e8f0f7'
WHITE      = '#ffffff'
STRIPE     = '#f4f7fa'
GREEN      = '#1a7a4a'
AMBER      = '#b7660a'
RED        = '#c0392b'
GREY_TEXT  = '#555555'

## Regression weakening window from Step 2c: post-2010, accelerating post-2015
REGRESSION_WINDOW = ['2011–2015', '2016–2020', '2021–2025']

FLAGGED_MUNS = {3304557, 4208203, 3201308}

## ── Table 1: first deficit year by municipality ───────────────────────────────
df_fd_display = first_deficit[[
    'co_mun', 'mun_name', 'sg_uf', 'first_deficit_year', 'period'
]].rename(columns={
    'co_mun'            : 'CO_MUN',
    'mun_name'          : 'Municipality',
    'sg_uf'             : 'UF',
    'first_deficit_year': 'First Deficit Year',
    'period'            : 'Period',
}).copy()

df_fd_display.insert(0, 'Rank', range(1, len(df_fd_display) + 1))
df_fd_display['CO_MUN'] = df_fd_display['CO_MUN'].apply(lambda x: str(int(x)))
df_fd_display['Flag'] = first_deficit['co_mun'].apply(
    lambda x: '⚠️' if x in FLAGGED_MUNS else ''
)

def colour_year(val):
    try:
        v = int(val)
    except (ValueError, TypeError):
        return ''
    if v >= 2016:
        return f'color: {RED}; font-weight: 700;'
    elif v >= 2011:
        return f'color: {AMBER}; font-weight: 600;'
    elif v >= 2001:
        return f'color: {GREY_TEXT};'
    return f'color: {GREEN}; font-weight: 500;'  ## pre-2000: long-established

def colour_period(val):
    if val in ('2016–2020', '2021–2025'):
        return f'color: {RED}; font-weight: 700;'
    elif val == '2011–2015':
        return f'color: {AMBER}; font-weight: 600;'
    elif val == '2001–2010':
        return f'color: {GREY_TEXT};'
    return f'color: {GREEN}; font-weight: 500;'

def colour_flag(val):
    if '⚠️' in str(val):
        return f'color: {AMBER}; font-weight: 700;'
    return ''

def highlight_window_rows(row):
    period = str(first_deficit.loc[row.name, 'period'])
    if period in ('2016–2020', '2021–2025'):
        return [f'background-color: #fdf0ee;'] * len(row)
    elif period == '2011–2015':
        return [f'background-color: #fff8e6;'] * len(row)
    return [''] * len(row)

display(
    df_fd_display.style
    .set_caption(
        f'First Deficit Year — Top 20 Deficit Municipalities ({MAX_YEAR}) '
        f'| Red = post-2015 (regression weakening window) '
        f'| Amber = 2011–2015 '
        f'| ⚠️ = open investigation'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .apply(highlight_window_rows, axis=1)
    .map(colour_year,   subset=['First Deficit Year'])
    .map(colour_period, subset=['Period'])
    .map(colour_flag,   subset=['Flag'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Rank
            ('text-align', 'center'),
            ('font-weight', '600'),
            ('color', GREY_TEXT),
            ('font-size', '12px'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## CO_MUN
            ('font-family', 'monospace'),
            ('font-size', '11px'),
            ('color', GREY_TEXT),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Municipality
            ('font-weight', '500'),
            ('color', DARK_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## UF
            ('font-family', 'monospace'),
            ('font-weight', '700'),
            ('color', DARK_BLUE),
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(5)', 'props': [   ## First Deficit Year
            ('text-align', 'center'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(6)', 'props': [   ## Period
            ('text-align', 'center'),
        ]},
        {'selector': 'tbody td:nth-child(7)', 'props': [   ## Flag
            ('text-align', 'center'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
            ('margin-bottom', '28px'),
        ]},
    ])
    .hide(axis='index')
)

## ── Table 2: period distribution summary ─────────────────────────────────────
regression_total = period_counts[
    period_counts['Period'].isin(REGRESSION_WINDOW)
]['Count'].sum()
total_muns = period_counts['Count'].sum()

period_counts['Share (%)'] = (period_counts['Count'] / total_muns * 100).apply(
    lambda x: f'{x:.1f}%'
)
period_counts['In Regression Window'] = period_counts['Period'].apply(
    lambda x: 'Yes' if x in REGRESSION_WINDOW else 'No'
)

def colour_period_row(val):
    if val in ('2016–2020', '2021–2025'):
        return f'color: {RED}; font-weight: 700;'
    elif val == '2011–2015':
        return f'color: {AMBER}; font-weight: 600;'
    elif val == '2001–2010':
        return f'color: {GREY_TEXT};'
    return f'color: {GREEN}; font-weight: 500;'

def colour_in_window(val):
    if val == 'Yes':
        return f'color: {RED}; font-weight: 700;'
    return f'color: {GREEN}; font-weight: 500;'

def colour_count(val):
    try:
        v = int(val)
    except (ValueError, TypeError):
        return ''
    if v >= 5:
        return f'color: {RED}; font-weight: 700;'
    elif v >= 3:
        return f'color: {AMBER}; font-weight: 600;'
    return f'color: {GREY_TEXT};'

display(
    period_counts.style
    .set_caption(
        f'Deficit Entry Period Distribution — Top 20 Deficit Municipalities '
        f'| {regression_total} of {total_muns} entered deficit in the '
        f'post-2010 regression weakening window '
        f'({100*regression_total/total_muns:.1f}%)'
    )
    .set_properties(**{
        'font-family'  : 'Helvetica Neue, Arial, sans-serif',
        'font-size'    : '13px',
        'text-align'   : 'left',
        'padding'      : '8px 14px',
        'border-bottom': f'1px solid {LIGHT_BLUE}',
    })
    .map(colour_period_row, subset=['Period'])
    .map(colour_count,      subset=['Count'])
    .map(colour_in_window,  subset=['In Regression Window'])
    .set_table_styles([
        {'selector': 'caption', 'props': [
            ('caption-side', 'top'),
            ('font-size', '14px'),
            ('font-weight', '700'),
            ('font-family', 'Helvetica Neue, Arial, sans-serif'),
            ('color', DARK_BLUE),
            ('padding', '0 0 10px 0'),
            ('text-align', 'left'),
        ]},
        {'selector': 'thead th', 'props': [
            ('background-color', DARK_BLUE),
            ('color', WHITE),
            ('font-size', '12px'),
            ('font-weight', '600'),
            ('text-transform', 'uppercase'),
            ('letter-spacing', '0.04em'),
            ('padding', '10px 14px'),
            ('border', 'none'),
        ]},
        {'selector': 'thead tr:last-child th', 'props': [
            ('border-bottom', f'2px solid {MID_BLUE}'),
        ]},
        {'selector': 'tbody tr:nth-child(odd)', 'props': [
            ('background-color', WHITE),
        ]},
        {'selector': 'tbody tr:nth-child(even)', 'props': [
            ('background-color', STRIPE),
        ]},
        {'selector': 'tbody tr:hover', 'props': [
            ('background-color', LIGHT_BLUE),
        ]},
        {'selector': 'tbody td:nth-child(1)', 'props': [   ## Period
            ('font-weight', '500'),
        ]},
        {'selector': 'tbody td:nth-child(2)', 'props': [   ## Count
            ('text-align', 'center'),
            ('font-variant-numeric', 'tabular-nums'),
            ('font-weight', '700'),
        ]},
        {'selector': 'tbody td:nth-child(3)', 'props': [   ## Share
            ('text-align', 'right'),
            ('font-variant-numeric', 'tabular-nums'),
        ]},
        {'selector': 'tbody td:nth-child(4)', 'props': [   ## In Window
            ('text-align', 'center'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
            ('border', f'1px solid {LIGHT_BLUE}'),
            ('border-radius', '6px'),
            ('overflow', 'hidden'),
        ]},
    ])
    .hide(axis='index')
)

Rank,CO_MUN,Municipality,UF,First Deficit Year,Period,Flag
1,1100205,PORTO VELHO,RO,1997,Pre-2000,
2,4125506,SAO JOSE DOS PINHAIS,PR,1997,Pre-2000,
3,4106902,CURITIBA,PR,1997,Pre-2000,
4,3450308,SAO PAULO,SP,1997,Pre-2000,
5,3436505,PAULINIA,SP,1997,Pre-2000,
6,3425904,JUNDIAI,SP,1997,Pre-2000,
7,5301108,ANAPOLIS,GO,1997,Pre-2000,
8,3405708,BARUERI,SP,1997,Pre-2000,
9,3409502,CAMPINAS,SP,1997,Pre-2000,
10,3201308,CARIACICA,ES,1997,Pre-2000,⚠️


Period,Count,Share (%),In Regression Window
Pre-2000,17,85.0%,No
2001–2010,3,15.0%,No
2011–2015,0,0.0%,Yes
2016–2020,0,0.0%,Yes
2021–2025,0,0.0%,Yes


#### Overview

The top 20 deficit municipalities by 2025 balance represent a combined deficit of $83.04bn. The two tables together resolve the deficit onset question and deliver a single cross-cutting finding.

**Manaus leads at -\\$15.03bn, followed by Itajaí at -\\$10.19bn**. As established in prior sections, both are structurally explained: Manaus by the Zona Franca de Manaus import registration mechanism, and Itajaí by port logistics hub registration with a distributed HHI of 0.0614. Petrópolis (-\\$8.87bn, rank 3) and Cariacica (-\\$7.03bn, rank 4) are similarly resolved through Sections 4.1.2 and 4.1.3 respectively. The four open investigations account for \\$41.12bn of the \\$83.04bn combined deficit, or 49.5% of the total.

**São Paulo state places seven municipalities in the top 20**, accounting for a combined deficit of approximately \\$18.47bn across São Paulo city (-\\$5.41bn), Paulínia (-\\$5.38bn), Jundiaí (-\\$3.35bn), Campinas (-\\$2.19bn), Barueri (-\\$2.00bn), and São José dos Pinhais (-\\$2.18bn, Paraná). Santa Catarina places three municipalities: Itajaí, Joinville (-\\$3.16bn), and Araquari (-\\$1.38bn).

**Araquari (-\\$1.38bn, rank 20) is notable given its \\$0.06bn in exports against $1.44bn in imports**. At this export-to-import ratio, Araquari registers as a near-pure import municipality.

**The deficit onset distribution is the primary structural finding**. Of the top 20 deficit municipalities in 2025, 17 entered deficit before 2000 and three entered deficit between 2001 and 2010. Zero municipalities entered deficit in any post-2010 window. The hypothesis that the post-2015 regression slope weakening identified in Step 2c is linked to new deficit municipalities emerging in that period is not supported by the data. The deficit structure of the top 20 is entirely pre-2010 in origin.

**The period distribution table confirms this directly**. The 2011–2015, 2016–2020, and 2021–2025 windows each record 0 municipalities and 0.0% share. The regression weakening documented in Step 2c therefore reflects changes in the magnitude of existing deficits rather than the emergence of new deficit municipalities. The post-2015 deficit growth is concentrated in municipalities that have been deficit registrants since at least 2010, with Itajaí (-\\$10.19bn in 2025 vs -\\$0.27bn at onset in 2009) as the primary driver of the widening.

#### Business Relevance

The pre-2000 concentration of deficit onset dates establishes that Brazil's municipality-level import deficit structure is not a recent phenomenon driven by post-2015 trade liberalisation or exchange rate movements. The deficits are structural and long-standing. For commercial analysis, the actionable implication is that the top 20 deficit municipalities represent established import registration clusters rather than emerging ones, and that new entrants to this ranking in recent years are growing in deficit magnitude rather than being newly formed deficit locations. The São Paulo state cluster of seven municipalities, all pre-2000 deficit registrants, confirms that the São Paulo industrial and commercial corridor has been the dominant import demand registration base for the full duration of the dataset.

---
---

## 4.8 — Open Flag Resolution Summary


| # | Flag | Section | Status | Conclusion |
|---|---|---|---|---|
| 1 | Itajaí \\$16.31bn imports / \\$-10.19bn deficit (SC) | 4.1.2 | Resolved | Logistics hub confirmed via HHI 0.0614; deficit widening is registration-driven, not product-specific. |
| 2 | Cariacica \\$7.17bn imports (ES) | 4.1.1 | Resolved | Vehicle distribution hub confirmed; finished vehicles 96.5%, EV-heavy mix at 52.3% of passenger imports. |
| 3 | Petrópolis \\$9.47bn imports / no established profile (RJ) | 4.1.3 | Resolved | Petroleum and energy registration address; 66.7% energy sector imports, no local industrial demand signal. |
| 4 | SH27 crude vs refined petroleum split | 4.2.1 | Resolved | Brazil exports \\$44.54bn crude, imports \\$9.49bn diesel; net refined product deficit \\$4.90bn. |
| 5 | Ferro-niobium (NCM 72029300) national value + destination markets | 4.2.2 | Resolved | \\$2.66bn national total; China 49.8%, Netherlands 17.3%; most diversified destination mix of any major commodity. |
| 6 | Cotton destination markets — Mato Grosso | 4.3.1 | Resolved | Routes to textile economies; Bangladesh 17.2%, Turkey 16.1%, Vietnam 15.2%, Pakistan 14.4%, China 13.9%. |
| 7 | Tobacco destination markets — Rio Grande do Sul | 4.3.2 | Resolved | Belgium 23.0%, China 22.4%, Indonesia 9.2%; no structural dependency on any single corridor. |
| 8 | Goiás pharmaceutical imports (SH30) — NCM decomposition | 4.3.3 | Resolved | Distribution hub for finished biologics, not API manufacturing; NCM 30021590 alone accounts for 63.4%. |
| 9 | Piauí and Rio de Janeiro dominant NCM codes (product HHI) | 4.4.1 | Resolved | PI: soybeans 82.7% explain 99.3% of HHI. RJ: crude oil registration 79.2% explains 99.4% of HHI. |
| 10 | Rio Grande do Norte / Panama top partner | 4.4.2 | Resolved | Panama entry is a single fuel oil flow (\\$488.0m, NCM 27101922); true demand base is North America and Western Europe. |
| 11 | Vehicle import composition — finished vs components (national) | 4.5.1 | Resolved | Finished vehicles 56.2%, parts 37.1%; diesel trucks lead at \\$3.35bn, EVs exceed ICE passenger vehicles nationally. |
| 12 | Manufactured goods destination markets | 4.5.2 | Resolved | Vehicles: Latin America 72.6%. Machinery/electrical: US-led. Aircraft: US 62.4%. China 0.0% across all sectors. |
| 13 | Paraná trade balance reversal mechanism | 4.6 | Resolved | Export growth driven by meat (+101%) and oilseeds (+37%); vehicle imports contracted 48%; fertiliser imports grew 80%. |
| 14 | Anchieta and Aracruz (ES) export destination markets | 4.7.1 | Partially Resolved | Anchieta: natural stone to US/China. Aracruz: pulp confirmed; machinery and petroleum entries require NCM-level follow-up. |
| 15 | Marabá 9.9% China share — product mix confirmation | 4.7.2 | Resolved | Copper ore dominant at 89.6%, not iron ore; low China share explained by European smelter routing. |
| 16 | Post-2015 deficit convergence — municipality time series | 4.7.3 | Resolved | All top-20 deficit municipalities entered deficit pre-2010; post-2015 widening reflects magnitude growth, not new entrants. |

---
---


## 4.9 — Key Findings


---
